# 读取优化模型

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import Model, GRB

# =============================================================================
# 函数: 合成氨系统优化
# =============================================================================
def optimize_ammonia_system(
    wind_output,
    pv_output,
    target_ammonia_production,
    wind_cost,
    pv_cost,
    battery_energy_cost,
    battery_storage_cost,
    battery_charge_efficiency,
    battery_discharge_efficiency,
    battery_charge_penalty,
    battery_discharge_penalty,
    electrolyzer_cost_AE,
    electrolyzer_cost_PEM,
    electrolyzer_eff_AE,
    electrolyzer_eff_PEM,
    ammonia_synthesis_cost,
    hydrogen_storage_cost,
    hydrogen_storage_elec,
    hydrogen_charge_penalty,
    hydrogen_storage_efficiency,
    ammonia_storage_cost_param,
    discount_rate,
    battery_lifetime,
    wind_lifetime,
    pv_lifetime,
    electrolyzer_lifetime_AE,
    electrolyzer_lifetime_PEM,
    ammonia_synthesis_lifetime,
    hydrogen_storage_lifetime,
    ammonia_storage_lifetime,
    Flex_up,
    Flex_mid,
    Flex_down,
    surplus_penalty,
    aggregation_mode
):
    # —————— 年化成本与运维 ——————
    wind_om                = wind_cost * 0.03
    pv_om                  = pv_cost * 0.03
    battery_energy_om      = battery_energy_cost * 0.03
    battery_storage_om     = battery_storage_cost * 0.03
    electrolyzer_om_AE     = electrolyzer_cost_AE * 0.03
    electrolyzer_om_PEM    = electrolyzer_cost_PEM * 0.03
    ammonia_synthesis_om   = ammonia_synthesis_cost * 0.03
    hydrogen_storage_om    = hydrogen_storage_cost * 0.03
    ammonia_storage_om     = ammonia_storage_cost_param * 0.03

    wind_annual_cost           = wind_cost * discount_rate / (1 - (1+discount_rate)**-wind_lifetime)     + wind_om
    pv_annual_cost             = pv_cost   * discount_rate / (1 - (1+discount_rate)**-pv_lifetime)       + pv_om
    battery_energy_annual_cost = battery_energy_cost  * discount_rate / (1 - (1+discount_rate)**-battery_lifetime) + battery_energy_om
    battery_storage_annual_cost= battery_storage_cost * discount_rate / (1 - (1+discount_rate)**-battery_lifetime) + battery_storage_om
    electrolyzer_annual_cost_AE= electrolyzer_cost_AE  * discount_rate / (1 - (1+discount_rate)**-electrolyzer_lifetime_AE) + electrolyzer_om_AE
    electrolyzer_annual_cost_PEM=electrolyzer_cost_PEM * discount_rate / (1 - (1+discount_rate)**-electrolyzer_lifetime_PEM) + electrolyzer_om_PEM
    ammonia_synthesis_annual_cost=ammonia_synthesis_cost * discount_rate / (1 - (1+discount_rate)**-ammonia_synthesis_lifetime) + ammonia_synthesis_om
    hydrogen_storage_annual_cost= hydrogen_storage_cost * discount_rate / (1 - (1+discount_rate)**-hydrogen_storage_lifetime) + hydrogen_storage_om
    ammonia_storage_annual_cost = ammonia_storage_cost_param * discount_rate / (1 - (1+discount_rate)**-ammonia_storage_lifetime) + ammonia_storage_om

    # 逐小时需求
    hourly_ammonia_demand = target_ammonia_production / 8760
    ammonia_electricity_ratio = 0.303
    ammonia_hydrogen_ratio   = 0.18

    # —————— 建模 ——————
    time_steps = 8760
    model = Model("AmmoniaOptimization_SplitBattery_AEPem")
    model.setParam('OutputFlag', 0)
    model.setParam('TimeLimit', 1000)
    model.setParam('Method', 2); model.setParam('Crossover', 0)
    model.setParam('BarConvTol', 1e-6); model.setParam('Threads', 4)
    model.setParam('Presolve', 1); model.setParam('BarHomogeneous', 1)

    # 决策变量
    wind_capacity      = model.addVar(lb=0, name="wind_capacity")
    pv_capacity        = model.addVar(lb=0, name="pv_capacity")
    battery_energy_capacity= model.addVar(lb=0, name="battery_energy_capacity")
    battery_storage_capacity= model.addVar(lb=0, name="battery_storage_capacity")
    electrolyzer_capacity_AE= model.addVar(lb=0, name="electrolyzer_capacity_AE")
    electrolyzer_capacity_PEM=model.addVar(lb=0, name="electrolyzer_capacity_PEM")
    ammonia_synthesis_capacity=model.addVar(lb=0, name="ammonia_synthesis_capacity")
    hydrogen_storage_capacity =model.addVar(lb=0, name="hydrogen_storage_capacity")
    ammonia_storage_capacity  =model.addVar(lb=0, name="ammonia_storage_capacity")

    # 时步变量
    energy_balance           = model.addVars(time_steps, lb=0, name="energy_balance")
    battery_charge          = model.addVars(time_steps, lb=0, name="battery_charge")
    battery_discharge       = model.addVars(time_steps, lb=0, name="battery_discharge")
    electrolyzer_power_AE   = model.addVars(time_steps, lb=0, name="electrolyzer_power_AE")
    electrolyzer_power_PEM  = model.addVars(time_steps, lb=0, name="electrolyzer_power_PEM")
    ammonia_synthesis_power = model.addVars(time_steps, lb=0, name="ammonia_synthesis_power")
    surplus                 = model.addVars(time_steps, lb=0, name="surplus")
    hydrogen_storage_balance= model.addVars(time_steps, lb=0, name="hydrogen_storage_balance")
    hydrogen_charge         = model.addVars(time_steps, lb=0, name="hydrogen_charge")
    hydrogen_discharge      = model.addVars(time_steps, lb=0, name="hydrogen_discharge")
    ammonia_storage_balance = model.addVars(time_steps, lb=0, name="ammonia_storage_balance")
    ammonia_storage_charge  = model.addVars(time_steps, lb=0, name="ammonia_storage_charge")
    ammonia_storage_discharge=model.addVars(time_steps, lb=0, name="ammonia_storage_discharge")

    # 目标函数
    model.setObjective(
        wind_annual_cost * wind_capacity +
        pv_annual_cost * pv_capacity +
        battery_energy_annual_cost * battery_energy_capacity +
        battery_storage_annual_cost * battery_storage_capacity +
        electrolyzer_annual_cost_AE * electrolyzer_capacity_AE +
        electrolyzer_annual_cost_PEM * electrolyzer_capacity_PEM +
        ammonia_synthesis_annual_cost * ammonia_synthesis_capacity +
        hydrogen_storage_annual_cost * hydrogen_storage_capacity +
        ammonia_storage_annual_cost * ammonia_storage_capacity +
        surplus_penalty * gp.quicksum(surplus[t] for t in range(time_steps)) +
        battery_charge_penalty * gp.quicksum(battery_charge[t] for t in range(time_steps)) +
        battery_discharge_penalty * gp.quicksum(battery_discharge[t] for t in range(time_steps)) +
        hydrogen_charge_penalty * gp.quicksum(hydrogen_charge[t] for t in range(time_steps)),
        GRB.MINIMIZE
    )

# =============================================================================
# 5. 约束条件
# =============================================================================
# 5.1 逐时灵活性
    for t in range(1, time_steps):
        model.addConstr(electrolyzer_power_AE[t]   - electrolyzer_power_AE[t-1]   <= Flex_down * electrolyzer_capacity_AE,
                        name=f"electrolyzer_AE_ramp_up_{t}")
        model.addConstr(electrolyzer_power_AE[t]   - electrolyzer_power_AE[t-1]   >= -Flex_down * electrolyzer_capacity_AE,
                        name=f"electrolyzer_AE_ramp_down_{t}")
        model.addConstr(ammonia_synthesis_power[t] - ammonia_synthesis_power[t-1] <= Flex_mid  * ammonia_synthesis_capacity,
                        name=f"ammonia_ramp_up_{t}")
        model.addConstr(ammonia_synthesis_power[t] - ammonia_synthesis_power[t-1] >= -Flex_down * ammonia_synthesis_capacity,
                        name=f"ammonia_ramp_down_{t}")

    # 5.2 合成氨供需平衡约束（含氨储存充放，支持不同时间聚合尺度）
    start_index = 168  # 避免初始不稳定阶段
    if aggregation_mode == "hourly":
        for t in range(start_index, time_steps):
            model.addConstr(
                ammonia_synthesis_power[t] + ammonia_storage_discharge[t] ==
                hourly_ammonia_demand + ammonia_storage_charge[t],
                name=f"ammonia_balance_{t}"
            )
    elif aggregation_mode == "3hourly":
        interval = 3
        n_intervals = (time_steps - start_index) // interval
        for i in range(n_intervals):
            t_start = start_index + i * interval
            t_end = t_start + interval
            model.addConstr(
                gp.quicksum(ammonia_synthesis_power[t] for t in range(t_start, t_end)) +
                gp.quicksum(ammonia_storage_discharge[t] for t in range(t_start, t_end))
                ==
                hourly_ammonia_demand * interval +
                gp.quicksum(ammonia_storage_charge[t] for t in range(t_start, t_end)),
                name=f"ammonia_balance_3hourly_{i}"
            )
    elif aggregation_mode == "daily":
        interval = 24
        n_intervals = (time_steps - start_index) // interval
        for i in range(n_intervals):
            t_start = start_index + i * interval
            t_end = t_start + interval
            model.addConstr(
                gp.quicksum(ammonia_synthesis_power[t] for t in range(t_start, t_end)) +
                gp.quicksum(ammonia_storage_discharge[t] for t in range(t_start, t_end))
                >=
                0.8 * hourly_ammonia_demand * interval +
                gp.quicksum(ammonia_storage_charge[t] for t in range(t_start, t_end)),
                name=f"ammonia_balance_daily_{i}"
            )
    elif aggregation_mode == "weekly":
        interval = 168  # 一周168小时
        n_intervals = (time_steps - start_index) // interval
        for i in range(n_intervals):
            t_start = start_index + i * interval
            t_end = t_start + interval
            model.addConstr(
                gp.quicksum(ammonia_synthesis_power[t] for t in range(t_start, t_end)) +
                gp.quicksum(ammonia_storage_discharge[t] for t in range(t_start, t_end))
                ==
                hourly_ammonia_demand * interval +
                gp.quicksum(ammonia_storage_charge[t] for t in range(t_start, t_end)),
                name=f"ammonia_balance_weekly_{i}"
            )
    elif aggregation_mode == "monthly":
        # 近似每月730小时（可根据实际情况调整）
        interval = 730
        n_intervals = (time_steps - start_index) // interval
        for i in range(n_intervals):
            t_start = start_index + i * interval
            t_end = t_start + interval
            model.addConstr(
                gp.quicksum(ammonia_synthesis_power[t] for t in range(t_start, t_end)) +
                gp.quicksum(ammonia_storage_discharge[t] for t in range(t_start, t_end))
                ==
                hourly_ammonia_demand * interval +
                gp.quicksum(ammonia_storage_charge[t] for t in range(t_start, t_end)),
                name=f"ammonia_balance_monthly_{i}"
            )
    elif aggregation_mode == "annual":
        model.addConstr(
            gp.quicksum(ammonia_synthesis_power[t] for t in range(start_index, time_steps)) +
            gp.quicksum(ammonia_storage_discharge[t] for t in range(start_index, time_steps))
            ==
            hourly_ammonia_demand * (time_steps - start_index) +
            gp.quicksum(ammonia_storage_charge[t] for t in range(start_index, time_steps)),
            name="ammonia_balance_annual"
        )
    else:
        raise ValueError("Unsupported aggregation_mode value!")

    # 附加：年度合成氨生产总量约束
    model.addConstr(
        gp.quicksum(ammonia_synthesis_power[t] for t in range(time_steps)) >= hourly_ammonia_demand * 8760,
        name="annual_ammonia_production_constraint"
    )

    # 5.3 电力与存储平衡
    conversion_factor_AE  = electrolyzer_eff_AE  * 3.6 / 120
    conversion_factor_PEM = electrolyzer_eff_PEM * 3.6 / 120

    for t in range(time_steps):
        wind_power = wind_output[t] * wind_capacity
        pv_power   = pv_output[t]   * pv_capacity

        # 电池储能约束
        model.addConstr(energy_balance[t] <= battery_storage_capacity,
                        name=f"energy_balance_limit_{t}")
        model.addConstr(battery_charge[t] + battery_discharge[t] <= battery_energy_capacity,
                        name=f"battery_charge_discharge_limit_{t}")

        # 动态电池平衡
        if t == 0:
            model.addConstr(
                energy_balance[t] == battery_charge[t] * battery_charge_efficiency
                                    - battery_discharge[t] / battery_discharge_efficiency,
                name=f"battery_balance_{t}" )
        else:
            model.addConstr(
                energy_balance[t] == energy_balance[t-1] +
                                    battery_charge[t] * battery_charge_efficiency
                                  - battery_discharge[t] / battery_discharge_efficiency,
                name=f"battery_balance_{t}" )

        # 氢气存储约束
        model.addConstr(hydrogen_storage_balance[t] <= hydrogen_storage_capacity,
                        name=f"hydrogen_storage_capacity_limit_{t}")
        model.addConstr(hydrogen_charge[t] + hydrogen_discharge[t] <= hydrogen_storage_capacity,
                        name=f"hydrogen_charge_limit_{t}")

        # 氢气存储动态平衡
        if t == 0:
            model.addConstr(
                hydrogen_storage_balance[t] == hydrogen_charge[t] * hydrogen_storage_efficiency - hydrogen_discharge[t],
                name=f"hydrogen_balance_{t}"
            )
        else:
            model.addConstr(
                hydrogen_storage_balance[t] == hydrogen_storage_balance[t-1] + hydrogen_charge[t] * hydrogen_storage_efficiency - hydrogen_discharge[t],
                name=f"hydrogen_balance_{t}"
            )

        # 氢气供需平衡：电解槽产氢（转换系数 3.6/120）加上氢气释放，
        # 需满足合成氨生产所需的氢气（按氢消耗比）加上充氢量

        # 氨储存平衡
        model.addConstr(ammonia_storage_balance[t] == (ammonia_storage_balance[t-1] if t>0 else ammonia_storage_charge[0])
                                  + ammonia_storage_charge[t] - ammonia_storage_discharge[t],
                        name=f"ammonia_storage_balance_{t}")
        model.addConstr(ammonia_storage_balance[t] <= ammonia_storage_capacity,
                        name=f"ammonia_storage_capacity_limit_{t}")

        # 氨储存：确保合成氨生产加上储存放出不少于储入量
        model.addConstr(
            ammonia_synthesis_power[t] + ammonia_storage_discharge[t] >= ammonia_storage_charge[t],
            name=f"ammonia_min_balance_{t}"
        )

        # 电力平衡
        model.addConstr(
            wind_power + pv_power + battery_discharge[t] ==
            electrolyzer_power_AE[t] + electrolyzer_power_PEM[t] +
            battery_charge[t] + hydrogen_charge[t] * hydrogen_storage_elec +
            ammonia_synthesis_power[t] * ammonia_electricity_ratio + surplus[t],
            name=f"power_balance_{t}" )

        # 氢气供需平衡
        model.addConstr(
            electrolyzer_power_AE[t] * conversion_factor_AE +
            electrolyzer_power_PEM[t] * conversion_factor_PEM +
            hydrogen_discharge[t] >=
            ammonia_synthesis_power[t] * ammonia_hydrogen_ratio + hydrogen_charge[t],
            name=f"hydrogen_PD_balance_{t}" )

    # 5.4 容量与最小运行
    model.addConstrs((electrolyzer_capacity_AE >= electrolyzer_power_AE[t] for t in range(time_steps)), name="electrolyzer_AE_capacity")
    model.addConstrs((electrolyzer_capacity_PEM >= electrolyzer_power_PEM[t] for t in range(time_steps)), name="electrolyzer_PEM_capacity")
    model.addConstrs((electrolyzer_power_AE[t] >= 0.1 * electrolyzer_capacity_AE for t in range(start_index, time_steps)), name="electrolyzer_AE_min")
    model.addConstrs((ammonia_synthesis_capacity >= ammonia_synthesis_power[t] for t in range(time_steps)), name="ammonia_capacity")
    model.addConstrs((ammonia_synthesis_power[t] >= 0.2 * ammonia_synthesis_capacity for t in range(start_index, time_steps)), name="ammonia_min")

    # AE电解需配备电池能量：
    model.addConstr(battery_energy_capacity >= 0.5 * electrolyzer_capacity_AE, name="min_battery_for_AE")


    model.optimize()
    return model



In [ ]:
import numpy as np
import gurobipy as gp
from gurobipy import Model, GRB


def optimize_methanol_system(
    wind_output,
    pv_output,
    target_methanol_production,
    wind_cost,
    pv_cost,
    battery_energy_cost,
    battery_storage_cost,
    battery_charge_efficiency,
    battery_discharge_efficiency,
    battery_charge_penalty,
    battery_discharge_penalty,
    electrolyzer_cost_AE,
    electrolyzer_cost_PEM,
    electrolyzer_eff_AE,
    electrolyzer_eff_PEM,
    methanol_synthesis_cost,
    hydrogen_storage_cost,
    hydrogen_storage_elec,
    hydrogen_charge_penalty,
    hydrogen_storage_efficiency,
    methanol_storage_cost_param,
    methanol_charge_penalty,
    discount_rate,
    battery_lifetime,
    wind_lifetime,
    pv_lifetime,
    electrolyzer_lifetime_AE,
    electrolyzer_lifetime_PEM,
    methanol_synthesis_lifetime,
    hydrogen_storage_lifetime,
    methanol_storage_lifetime,
    heat_pump_cost,
    heat_pump_cop,
    heat_pump_lifetime,
    thermal_storage_cost,
    thermal_storage_efficiency,
    thermal_storage_lifetime,
    dac_cost,
    dac_elec_per_kg,
    dac_heat_per_kg,
    dac_lifetime,
    sorbent_cost,
    co2_compressor_cost,
    co2_compression_energy,
    co2_compressor_lifetime,
    co2_storage_cost,
    co2_storage_lifetime,
    Flex_up,
    Flex_mid,
    Flex_down,
    surplus_penalty,
    aggregation_mode
):
    # 年化运维成本
    wind_om = wind_cost * 0.03
    pv_om = pv_cost * 0.03
    battery_energy_om = battery_energy_cost * 0.03
    battery_storage_om = battery_storage_cost * 0.03
    electrolyzer_om_AE = electrolyzer_cost_AE * 0.03
    electrolyzer_om_PEM = electrolyzer_cost_PEM * 0.03
    methanol_synthesis_om = methanol_synthesis_cost * 0.03
    hydrogen_storage_om = hydrogen_storage_cost * 0.03
    methanol_storage_om = methanol_storage_cost_param * 0.03
    heat_pump_om = heat_pump_cost * 0.03
    thermal_storage_om = thermal_storage_cost * 0.03
    dac_om = dac_cost * 0.03
    co2_compressor_om = co2_compressor_cost * 0.03
    co2_storage_om = co2_storage_cost * 0.03

    # 年化成本
    wind_annual_cost = wind_cost * discount_rate / (1 - (1+discount_rate)**-wind_lifetime) + wind_om
    pv_annual_cost = pv_cost * discount_rate / (1 - (1+discount_rate)**-pv_lifetime) + pv_om
    battery_energy_annual_cost = battery_energy_cost * discount_rate / (1 - (1+discount_rate)**-battery_lifetime) + battery_energy_om
    battery_storage_annual_cost = battery_storage_cost * discount_rate / (1 - (1+discount_rate)**-battery_lifetime) + battery_storage_om
    electrolyzer_annual_cost_AE = electrolyzer_cost_AE * discount_rate / (1 - (1+discount_rate)**-electrolyzer_lifetime_AE) + electrolyzer_om_AE
    electrolyzer_annual_cost_PEM = electrolyzer_cost_PEM * discount_rate / (1 - (1+discount_rate)**-electrolyzer_lifetime_PEM) + electrolyzer_om_PEM
    methanol_synthesis_annual_cost = methanol_synthesis_cost * discount_rate / (1 - (1+discount_rate)**-methanol_synthesis_lifetime) + methanol_synthesis_om
    hydrogen_storage_annual_cost = hydrogen_storage_cost * discount_rate / (1 - (1+discount_rate)**-hydrogen_storage_lifetime) + hydrogen_storage_om
    methanol_storage_annual_cost = methanol_storage_cost_param * discount_rate / (1 - (1+discount_rate)**-methanol_storage_lifetime) + methanol_storage_om
    heat_pump_annual_cost = heat_pump_cost * discount_rate / (1 - (1+discount_rate)**-heat_pump_lifetime) + heat_pump_om
    thermal_storage_annual_cost = thermal_storage_cost * discount_rate / (1 - (1+discount_rate)**-thermal_storage_lifetime) + thermal_storage_om
    dac_annual_cost = dac_cost * discount_rate / (1 - (1+discount_rate)**-dac_lifetime) + dac_om
    co2_compressor_annual_cost = co2_compressor_cost * discount_rate / (1 - (1+discount_rate)**-co2_compressor_lifetime) + co2_compressor_om
    co2_storage_annual_cost = co2_storage_cost * discount_rate / (1 - (1+discount_rate)**-co2_storage_lifetime) + co2_storage_om

    hourly_methanol_demand = target_methanol_production / 8760
    methanol_electricity_ratio = 0.7
    methanol_hydrogen_ratio = 0.19
    methanol_co2_ratio = 1.4

    time_steps = 8760
    model = Model("MethanolDACOptimization")
    model.setParam('OutputFlag', 0)
    model.setParam('TimeLimit', 1000)
    model.setParam('Method', 2)
    model.setParam('Crossover', 0)
    model.setParam('BarConvTol', 1e-6)
    model.setParam('Threads', 4)
    model.setParam('Presolve', 1)
    model.setParam('BarHomogeneous', 1)

    # 决策变量
    wind_capacity = model.addVar(lb=0, name="wind_capacity")
    pv_capacity = model.addVar(lb=0, name="pv_capacity")
    battery_energy_capacity = model.addVar(lb=0, name="battery_energy_capacity")
    battery_storage_capacity = model.addVar(lb=0, name="battery_storage_capacity")
    electrolyzer_capacity_AE = model.addVar(lb=0, name="electrolyzer_capacity_AE")
    electrolyzer_capacity_PEM = model.addVar(lb=0, name="electrolyzer_capacity_PEM")
    methanol_synthesis_capacity = model.addVar(lb=0, name="methanol_synthesis_capacity")
    heat_pump_capacity = model.addVar(lb=0, name="heat_pump_capacity")
    thermal_storage_capacity = model.addVar(lb=0, name="thermal_storage_capacity")
    methanol_storage_capacity = model.addVar(lb=0, name="methanol_storage_capacity")
    hydrogen_storage_capacity = model.addVar(lb=0, name="hydrogen_storage_capacity")
    dac_capacity = model.addVar(lb=0, name="dac_capacity")
    co2_compressor_capacity = model.addVar(lb=0, name="co2_compressor_capacity")
    co2_storage_capacity = model.addVar(lb=0, name="co2_storage_capacity")

    # 时序变量
    energy_balance = model.addVars(time_steps, lb=0, name="energy_balance")
    battery_charge = model.addVars(time_steps, lb=0, name="battery_charge")
    battery_discharge = model.addVars(time_steps, lb=0, name="battery_discharge")
    electrolyzer_power_AE = model.addVars(time_steps, lb=0, name="electrolyzer_power_AE")
    electrolyzer_power_PEM = model.addVars(time_steps, lb=0, name="electrolyzer_power_PEM")
    methanol_synthesis_power = model.addVars(time_steps, lb=0, name="methanol_synthesis_power")
    surplus = model.addVars(time_steps, lb=0, name="surplus")
    hydrogen_storage_balance = model.addVars(time_steps, lb=0, name="hydrogen_storage_balance")
    hydrogen_charge = model.addVars(time_steps, lb=0, name="hydrogen_charge")
    hydrogen_discharge = model.addVars(time_steps, lb=0, name="hydrogen_discharge")
    methanol_storage_balance = model.addVars(time_steps, lb=0, name="methanol_storage_balance")
    methanol_storage_charge = model.addVars(time_steps, lb=0, name="methanol_storage_charge")
    methanol_storage_discharge = model.addVars(time_steps, lb=0, name="methanol_storage_discharge")
    dac_CO2 = model.addVars(time_steps, lb=0, name="dac_CO2")
    dac_electricity = model.addVars(time_steps, lb=0, name="dac_electricity")
    dac_heat_input = model.addVars(time_steps, lb=0, name="dac_heat_input")
    heat_pump_output = model.addVars(time_steps, lb=0, name="heat_pump_output")
    thermal_charge = model.addVars(time_steps, lb=0, name="thermal_charge")
    thermal_discharge = model.addVars(time_steps, lb=0, name="thermal_discharge")
    thermal_storage_level = model.addVars(time_steps, lb=0, name="thermal_storage_level")
    co2_comp_elec = model.addVars(time_steps, lb=0, name="co2_comp_elec")
    co2_storage_level = model.addVars(time_steps, lb=0, name="co2_storage_level")
    co2_charge = model.addVars(time_steps, lb=0, name="co2_charge")
    co2_discharge = model.addVars(time_steps, lb=0, name="co2_discharge")
    direct_co2 = model.addVars(time_steps, lb=0, name="direct_co2")
    stored_co2 = model.addVars(time_steps, lb=0, name="stored_co2")

    # 计算转换系数
    conversion_factor_AE = electrolyzer_eff_AE * 3.6 / 120
    conversion_factor_PEM = electrolyzer_eff_PEM * 3.6 / 120

    # 目标函数
    model.setObjective(
        wind_annual_cost * wind_capacity +
        pv_annual_cost * pv_capacity +
        battery_energy_annual_cost * battery_energy_capacity +
        battery_storage_annual_cost * battery_storage_capacity +
        electrolyzer_annual_cost_AE * electrolyzer_capacity_AE +
        electrolyzer_annual_cost_PEM * electrolyzer_capacity_PEM +
        methanol_synthesis_annual_cost * methanol_synthesis_capacity +
        heat_pump_annual_cost * heat_pump_capacity +
        thermal_storage_annual_cost * thermal_storage_capacity +
        hydrogen_storage_annual_cost * hydrogen_storage_capacity +
        methanol_storage_annual_cost * methanol_storage_capacity +
        dac_annual_cost * dac_capacity +
        co2_compressor_annual_cost * co2_compressor_capacity +
        co2_storage_annual_cost * co2_storage_capacity +
        sorbent_cost * gp.quicksum(dac_CO2[t] for t in range(time_steps)) +
        surplus_penalty * gp.quicksum(surplus[t] for t in range(time_steps)) +
        battery_charge_penalty * gp.quicksum(battery_charge[t] for t in range(time_steps)) +
        battery_discharge_penalty * gp.quicksum(battery_discharge[t] for t in range(time_steps)) +
        hydrogen_charge_penalty * gp.quicksum(hydrogen_charge[t] for t in range(time_steps)) +
        methanol_charge_penalty * gp.quicksum(methanol_storage_charge[t] for t in range(time_steps)),
        GRB.MINIMIZE
    )

    # 添加约束
    for t in range(1, time_steps):
        model.addConstr(electrolyzer_power_AE[t] - electrolyzer_power_AE[t-1] <= Flex_down * electrolyzer_capacity_AE)
        model.addConstr(electrolyzer_power_AE[t] - electrolyzer_power_AE[t-1] >= -Flex_down * electrolyzer_capacity_AE)
        model.addConstr(methanol_synthesis_power[t] - methanol_synthesis_power[t-1] <= Flex_up * methanol_synthesis_capacity)
        model.addConstr(methanol_synthesis_power[t] - methanol_synthesis_power[t-1] >= -Flex_down * methanol_synthesis_capacity)
        model.addConstr(dac_CO2[t] - dac_CO2[t-1] <= Flex_up * dac_capacity)
        model.addConstr(dac_CO2[t] - dac_CO2[t-1] >= -Flex_down * dac_capacity)

    # 5.2 甲醇供需平衡约束（含储存充放，支持多种聚合尺度）
    start_index = 168  # 避开初始不稳定阶段
    if aggregation_mode == "hourly":
        for t in range(start_index, time_steps):
            model.addConstr(
                methanol_synthesis_power[t] + methanol_storage_discharge[t] >=
                hourly_methanol_demand + methanol_storage_charge[t],
                name=f"methanol_balance_hourly_{t}"
            )
    elif aggregation_mode == "3hourly":
        interval = 3
        n_intervals = (time_steps - start_index) // interval
        for i in range(n_intervals):
            t_start = start_index + i * interval
            t_end = t_start + interval
            model.addConstr(
                gp.quicksum(methanol_synthesis_power[t] for t in range(t_start, t_end)) +
                gp.quicksum(methanol_storage_discharge[t] for t in range(t_start, t_end))
                ==
                hourly_methanol_demand * interval +
                gp.quicksum(methanol_storage_charge[t] for t in range(t_start, t_end)),
                name=f"methanol_balance_3hourly_{i}"
            )
    elif aggregation_mode == "daily":
        interval = 24
        n_intervals = (time_steps - start_index) // interval
        for i in range(n_intervals):
            t_start = start_index + i * interval
            t_end = t_start + interval
            model.addConstr(
                gp.quicksum(methanol_synthesis_power[t] for t in range(t_start, t_end)) +
                gp.quicksum(methanol_storage_discharge[t] for t in range(t_start, t_end))
                >=
                0.8 * hourly_methanol_demand * interval +
                gp.quicksum(methanol_storage_charge[t] for t in range(t_start, t_end)),
                name=f"methanol_balance_daily_{i}"
            )
    elif aggregation_mode == "weekly":
        interval = 168
        n_intervals = (time_steps - start_index) // interval
        for i in range(n_intervals):
            t_start = start_index + i * interval
            t_end = t_start + interval
            model.addConstr(
                gp.quicksum(methanol_synthesis_power[t] for t in range(t_start, t_end)) +
                gp.quicksum(methanol_storage_discharge[t] for t in range(t_start, t_end))
                ==
                hourly_methanol_demand * interval +
                gp.quicksum(methanol_storage_charge[t] for t in range(t_start, t_end)),
                name=f"methanol_balance_weekly_{i}"
            )
    elif aggregation_mode == "monthly":
        interval = 730
        n_intervals = (time_steps - start_index) // interval
        for i in range(n_intervals):
            t_start = start_index + i * interval
            t_end = t_start + interval
            model.addConstr(
                gp.quicksum(methanol_synthesis_power[t] for t in range(t_start, t_end)) +
                gp.quicksum(methanol_storage_discharge[t] for t in range(t_start, t_end))
                ==
                hourly_methanol_demand * interval +
                gp.quicksum(methanol_storage_charge[t] for t in range(t_start, t_end)),
                name=f"methanol_balance_monthly_{i}"
            )
    elif aggregation_mode == "annual":
        model.addConstr(
            gp.quicksum(methanol_synthesis_power[t] for t in range(start_index, time_steps)) +
            gp.quicksum(methanol_storage_discharge[t] for t in range(start_index, time_steps))
            ==
            hourly_methanol_demand * (time_steps - start_index) +
            gp.quicksum(methanol_storage_charge[t] for t in range(start_index, time_steps)),
            name="methanol_balance_annual"
        )
    else:
        raise ValueError("Unsupported aggregation_mode value!")

    # 附加：年生产总量约束
    model.addConstr(
        gp.quicksum(methanol_synthesis_power[t] for t in range(time_steps)) >= hourly_methanol_demand * 8760,
        name="annual_methanol_production"
    )

    # 5.3 甲醇储存动态与容量约束
    model.addConstr(methanol_storage_balance[0] ==
                    methanol_storage_charge[0] - methanol_storage_discharge[0],
                    name="methanol_storage_balance_0")
    for t in range(1, time_steps):
        model.addConstr(
            methanol_storage_balance[t] == methanol_storage_balance[t-1] + methanol_storage_charge[t] - methanol_storage_discharge[t],
            name=f"methanol_storage_balance_{t}"
        )
        model.addConstr(methanol_storage_balance[t] <= methanol_storage_capacity,
                        name=f"methanol_storage_capacity_limit_{t}")

    # 5.4 能量平衡与储能动态约束
    conversion_factor_AE  = electrolyzer_eff_AE  * 3.6 / 120
    conversion_factor_PEM = electrolyzer_eff_PEM * 3.6 / 120

    for t in range(time_steps):
        # 计算当期风电与光伏发电
        wind_power = wind_output[t] * wind_capacity
        pv_power = pv_output[t] * pv_capacity
        
        # 甲醇储存：要求合成功率与储存放出至少满足充入量
        model.addConstr(methanol_synthesis_power[t] + methanol_storage_discharge[t] >=
                        methanol_storage_charge[t],
                        name=f"methanol_min_balance_{t}")
                        
        # 电池储能及充放电限额
        model.addConstr(energy_balance[t] <= battery_storage_capacity,
                        name=f"energy_balance_limit_{t}")
        model.addConstr(battery_charge[t] + battery_discharge[t] <= battery_energy_capacity,
                        name=f"battery_charge_discharge_limit_{t}")
        # 氢气存储上限
        model.addConstr(hydrogen_storage_balance[t] <= hydrogen_storage_capacity,
                        name=f"hydrogen_storage_capacity_limit_{t}")
        model.addConstr(hydrogen_charge[t] + hydrogen_discharge[t] <= hydrogen_storage_capacity,
                        name=f"hydrogen_charge_limit_{t}")
        
        # 电力平衡（含电解槽、甲醇合成、储能、热泵、DAC与CO2压缩电耗等）
        model.addConstr(
            wind_power + pv_power + battery_discharge[t] ==
            (electrolyzer_power_AE[t] + electrolyzer_power_PEM[t]) +
            methanol_synthesis_power[t] * methanol_electricity_ratio +
            battery_charge[t] + hydrogen_charge[t] * hydrogen_storage_elec +
            (heat_pump_output[t] / heat_pump_cop) + dac_electricity[t] + surplus[t] + co2_comp_elec[t],
            name=f"power_balance_{t}"
        )
        # 电池动态平衡
        if t == 0:
            model.addConstr(
                energy_balance[t] == battery_charge[t] * battery_charge_efficiency
                                    - battery_discharge[t] / battery_discharge_efficiency,
                name=f"battery_balance_{t}" )
        else:
            model.addConstr(
                energy_balance[t] == energy_balance[t-1]+
                                    battery_charge[t] * battery_charge_efficiency
                                - battery_discharge[t] / battery_discharge_efficiency,
                name=f"battery_balance_{t}" )
        # 氢气存储动态平衡
        if t == 0:
            model.addConstr(
                hydrogen_storage_balance[t] == hydrogen_charge[t] * hydrogen_storage_efficiency - hydrogen_discharge[t],
                name=f"hydrogen_balance_{t}"
            )
        else:
            model.addConstr(
                hydrogen_storage_balance[t] == hydrogen_storage_balance[t-1] + hydrogen_charge[t] * hydrogen_storage_efficiency - hydrogen_discharge[t],
                name=f"hydrogen_balance_{t}"
            )
        # 氢气供需平衡：电解槽产氢需满足甲醇合成及储氢要求
        model.addConstr(
            electrolyzer_power_AE[t] * conversion_factor_AE +
            electrolyzer_power_PEM[t] * conversion_factor_PEM + hydrogen_discharge[t] >=
            methanol_synthesis_power[t] * methanol_hydrogen_ratio + hydrogen_charge[t],
            name=f"hydrogen_PD_balance_{t}"
        )
        # DAC部分：热、电耗与捕集CO2量关系
        model.addConstr(dac_heat_input[t] == dac_CO2[t] * dac_heat_per_kg,
                        name=f"dac_heat_consumption_{t}")
        model.addConstr(dac_electricity[t] == dac_CO2[t] * dac_elec_per_kg,
                        name=f"dac_electricity_consumption_{t}")
        model.addConstr(dac_CO2[t] <= dac_capacity,
                        name=f"dac_capacity_limit_{t}")
        
        # CO2分流：直接供给与进入储存
        model.addConstr(direct_co2[t] + stored_co2[t] == dac_CO2[t],
                        name=f"co2_split_{t}")
        model.addConstr(stored_co2[t] <= co2_compressor_capacity,
                        name=f"co2_compressor_capacity_limit_{t}")
        model.addConstr(co2_comp_elec[t] == stored_co2[t] * co2_compression_energy,
                        name=f"co2_compression_energy_{t}")
        model.addConstr(co2_charge[t] == stored_co2[t],
                        name=f"co2_charge_relation_{t}")
        
        # 热泵与热储热平衡
        model.addConstr(
            heat_pump_output[t] - thermal_charge[t] + thermal_discharge[t] >= dac_heat_input[t],
            name=f"heat_supply_balance_{t}"
        )
        # CO2储存动态及容量约束
        if t == 0:
            model.addConstr(
                co2_storage_level[t] == co2_charge[t] - co2_discharge[t],
                name="co2_storage_balance_0"
            )
        else:
            model.addConstr(
                co2_storage_level[t] == co2_storage_level[t-1] + co2_charge[t] - co2_discharge[t],
                name=f"co2_storage_balance_{t}"
            )
        model.addConstr(co2_storage_level[t] <= co2_storage_capacity,
                        name=f"co2_storage_level_limit_{t}")
        
        # 储热系统动态及容量约束
        if t == 0:
            model.addConstr(
                thermal_storage_level[t] == thermal_charge[t] * thermal_storage_efficiency - thermal_discharge[t] / thermal_storage_efficiency,
                name="thermal_storage_balance_0"
            )
        else:
            model.addConstr(
                thermal_storage_level[t] == thermal_storage_level[t-1] + thermal_charge[t] * thermal_storage_efficiency - thermal_discharge[t] / thermal_storage_efficiency,
                name=f"thermal_storage_balance_{t}"
            )
        model.addConstr(thermal_storage_level[t] <= thermal_storage_capacity,
                        name=f"thermal_storage_level_limit_{t}")

    # 5.5 装机容量与最小运行约束
    model.addConstrs((heat_pump_capacity >= heat_pump_output[t] for t in range(time_steps)),
                    name="heat_pump_capacity_constraint")
    model.addConstrs((heat_pump_output[t] >= 0.25 * heat_pump_capacity for t in range(168, time_steps)),
                    name="heat_pump_min_output")
    model.addConstrs((electrolyzer_capacity_AE >= electrolyzer_power_AE[t] for t in range(time_steps)),
                    name="electrolyzer_AE_capacity_constraint")
    model.addConstrs((electrolyzer_capacity_PEM >= electrolyzer_power_PEM[t] for t in range(time_steps)),
                    name="electrolyzer_PEM_capacity_constraint")
    model.addConstrs((electrolyzer_power_AE[t] >= 0.1 * electrolyzer_capacity_AE for t in range(168, time_steps)),
                    name="electrolyzer_AE_min_power")
    model.addConstrs((dac_CO2[t] >= 0.25 * dac_capacity for t in range(168, time_steps)),
                    name="heat_pump_min_output")
    model.addConstrs((methanol_synthesis_power[t] >= 0.25 * methanol_synthesis_capacity for t in range(168, time_steps)),
                    name="methanol_synthesis_min_output")
    model.addConstrs((methanol_synthesis_power[t] <= methanol_synthesis_capacity for t in range(time_steps)),
                    name="methanol_synthesis_max_output")

    # 5.6 每小时CO2供需平衡约束：直接供给加储存放出需满足当期甲醇CO2需求

    model.addConstrs((direct_co2[t] + co2_discharge[t] >= methanol_synthesis_power[t] * methanol_co2_ratio for t in range(time_steps)),
            name="co2_hourly_balance")

    # 5.7 年度CO2供给约束
    model.addConstr(gp.quicksum(direct_co2[t] + co2_discharge[t] for t in range(time_steps)) >= target_methanol_production * methanol_co2_ratio,
                    name="annual_co2_supply_constraint")

    # 5.8 新增：AE型电解槽必须配备至少10%（此处0.5倍）储能电池
    model.addConstr(battery_energy_capacity >= 0.5 * electrolyzer_capacity_AE,
                    name="min_battery_for_AE")

    model.optimize()
    return model


In [ ]:
import numpy as np
import gurobipy as gp
from gurobipy import Model, GRB


def optimize_ftl_system(
    wind_output,
    pv_output,
    target_ftl_production,
    # 成本 / 参数输入（允许外部灵活指定）
    wind_cost,
    pv_cost,
    battery_energy_cost,
    battery_storage_cost,
    battery_charge_efficiency,
    battery_discharge_efficiency,
    battery_charge_penalty,
    battery_discharge_penalty,
    electrolyzer_cost_AE,
    electrolyzer_cost_PEM,
    electrolyzer_eff_AE,
    electrolyzer_eff_PEM,
    electrolyzer_lifetime_AE,
    electrolyzer_lifetime_PEM,
    ftl_synthesis_cost,
    ftl_synthesis_lifetime,
    ftl_hydrogen_ratio,
    ftl_co2_ratio,
    ftl_electricity_ratio,
    hydrogen_storage_cost,
    hydrogen_storage_elec,
    hydrogen_charge_penalty,
    hydrogen_storage_efficiency,
    hydrogen_storage_lifetime,
    ftl_storage_cost_param,
    ftl_storage_lifetime,
    ftl_charge_penalty,
    heat_pump_cost,
    heat_pump_cop,
    heat_pump_lifetime,
    thermal_storage_cost,
    thermal_storage_efficiency,
    thermal_storage_lifetime,
    dac_cost,
    dac_elec_per_kg,
    dac_heat_per_kg,
    dac_lifetime,
    sorbent_cost,
    co2_compressor_cost,
    co2_compression_energy,
    co2_compressor_lifetime,
    co2_storage_cost,
    co2_storage_lifetime,
    discount_rate,
    battery_lifetime,
    wind_lifetime,
    pv_lifetime,
    Flex_up,
    Flex_mid,
    Flex_down,
    surplus_penalty,
    aggregation_mode
):
    """FT Liquid + DAC 优化模型（复用用户提供脚本逻辑，封装为函数）

    参数说明：传入逐小时风/光容量因子数组（长度需=8760），以及各类成本、效率、寿命与灵活性参数。
    返回：包含容量配置、利用率、平准化成本等指标的结果字典；若不可行返回空字典。
    """
    time_steps = len(wind_output)
    assert len(pv_output) == time_steps, "pv_output length must match wind_output"

    # 年化 O&M（使用统一 3% 或指定 6% 的比率）
    wind_om = wind_cost * 0.03
    pv_om = pv_cost * 0.03
    battery_energy_om = battery_energy_cost * 0.03
    battery_storage_om = battery_storage_cost * 0.03
    electrolyzer_om_AE = electrolyzer_cost_AE * 0.03
    electrolyzer_om_PEM = electrolyzer_cost_PEM * 0.03
    ftl_synthesis_om = ftl_synthesis_cost * 0.06  # 按用户脚本示例
    hydrogen_storage_om = hydrogen_storage_cost * 0.03
    ftl_storage_om = ftl_storage_cost_param * 0.03
    heat_pump_om = heat_pump_cost * 0.03
    thermal_storage_om = thermal_storage_cost * 0.03
    dac_om = dac_cost * 0.03
    co2_compressor_om = co2_compressor_cost * 0.03
    co2_storage_om = co2_storage_cost * 0.03

    hourly_ftl_demand = target_ftl_production / 8760.0

    # 年化资本 + O&M
    def crf(dr, life):
        return dr / (1 - (1 + dr) ** (-life))

    wind_annual_cost = wind_cost * crf(discount_rate, wind_lifetime) + wind_om
    pv_annual_cost = pv_cost * crf(discount_rate, pv_lifetime) + pv_om
    battery_energy_annual_cost = battery_energy_cost * crf(discount_rate, battery_lifetime) + battery_energy_om
    battery_storage_annual_cost = battery_storage_cost * crf(discount_rate, battery_lifetime) + battery_storage_om
    electrolyzer_annual_cost_AE = electrolyzer_cost_AE * crf(discount_rate, electrolyzer_lifetime_AE) + electrolyzer_om_AE
    electrolyzer_annual_cost_PEM = electrolyzer_cost_PEM * crf(discount_rate, electrolyzer_lifetime_PEM) + electrolyzer_om_PEM
    ftl_synthesis_annual_cost = ftl_synthesis_cost * crf(discount_rate, ftl_synthesis_lifetime) + ftl_synthesis_om
    hydrogen_storage_annual_cost = hydrogen_storage_cost * crf(discount_rate, hydrogen_storage_lifetime) + hydrogen_storage_om
    dac_annual_cost = dac_cost * crf(discount_rate, dac_lifetime) + dac_om
    co2_compressor_annual_cost = co2_compressor_cost * crf(discount_rate, co2_compressor_lifetime) + co2_compressor_om
    co2_storage_annual_cost = co2_storage_cost * crf(discount_rate, co2_storage_lifetime) + co2_storage_om
    heat_pump_annual_cost = heat_pump_cost * crf(discount_rate, heat_pump_lifetime) + heat_pump_om
    thermal_storage_annual_cost = thermal_storage_cost * crf(discount_rate, thermal_storage_lifetime) + thermal_storage_om
    ftl_storage_annual_cost = ftl_storage_cost_param * crf(discount_rate, ftl_storage_lifetime) + ftl_storage_om

    model = Model("HydrogenFTLOptimization")
    model.setParam('OutputFlag', 0)
    model.setParam('TimeLimit', 1000)
    model.setParam('Method', 2)
    model.setParam('Crossover', 0)
    model.setParam('BarConvTol', 1e-6)
    model.setParam('Threads', 1)
    model.setParam('Presolve', 1)
    model.setParam('BarHomogeneous', 1)

    # 容量变量
    wind_capacity = model.addVar(lb=0, name="wind_capacity")
    pv_capacity = model.addVar(lb=0, name="pv_capacity")
    battery_energy_capacity = model.addVar(lb=0, name="battery_energy_capacity")
    battery_storage_capacity = model.addVar(lb=0, name="battery_storage_capacity")
    electrolyzer_capacity_AE = model.addVar(lb=0, name="electrolyzer_capacity_AE")
    electrolyzer_capacity_PEM = model.addVar(lb=0, name="electrolyzer_capacity_PEM")
    ftl_synthesis_capacity = model.addVar(lb=0, name="FTL_capacity")
    heat_pump_capacity = model.addVar(lb=0, name="heat_pump_capacity")
    thermal_storage_capacity = model.addVar(lb=0, name="thermal_storage_capacity")
    ftl_storage_capacity = model.addVar(lb=0, name="FTL_storage_capacity")
    hydrogen_storage_capacity = model.addVar(lb=0, name="hydrogen_storage_capacity")
    dac_capacity = model.addVar(lb=0, name="dac_capacity")
    co2_compressor_capacity = model.addVar(lb=0, name="co2_compressor_capacity")
    co2_storage_capacity = model.addVar(lb=0, name="co2_storage_capacity")

    # 时序变量
    energy_balance = model.addVars(time_steps, lb=0, name="energy_balance")
    battery_charge = model.addVars(time_steps, lb=0, name="battery_charge")
    battery_discharge = model.addVars(time_steps, lb=0, name="battery_discharge")
    electrolyzer_power_AE = model.addVars(time_steps, lb=0, name="electrolyzer_power_AE")
    electrolyzer_power_PEM = model.addVars(time_steps, lb=0, name="electrolyzer_power_PEM")
    ftl_synthesis_power = model.addVars(time_steps, lb=0, name="FTL_synthesis_power")
    ftl_storage_balance = model.addVars(time_steps, lb=0, name="FTL_storage_balance")
    ftl_storage_charge = model.addVars(time_steps, lb=0, name="FTL_storage_charge")
    ftl_storage_discharge = model.addVars(time_steps, lb=0, name="FTL_storage_discharge")
    hydrogen_storage_balance = model.addVars(time_steps, lb=0, name="hydrogen_storage_balance")
    hydrogen_charge = model.addVars(time_steps, lb=0, name="hydrogen_charge")
    hydrogen_discharge = model.addVars(time_steps, lb=0, name="hydrogen_discharge")
    dac_CO2 = model.addVars(time_steps, lb=0, name="dac_CO2")
    dac_electricity = model.addVars(time_steps, lb=0, name="dac_electricity")
    dac_heat_input = model.addVars(time_steps, lb=0, name="dac_heat_input")
    heat_pump_output = model.addVars(time_steps, lb=0, name="heat_pump_output")
    thermal_charge = model.addVars(time_steps, lb=0, name="thermal_charge")
    thermal_discharge = model.addVars(time_steps, lb=0, name="thermal_discharge")
    thermal_storage_level = model.addVars(time_steps, lb=0, name="thermal_storage_level")
    co2_comp_elec = model.addVars(time_steps, lb=0, name="co2_comp_elec")
    co2_storage_level = model.addVars(time_steps, lb=0, name="co2_storage_level")
    co2_charge = model.addVars(time_steps, lb=0, name="co2_charge")
    co2_discharge = model.addVars(time_steps, lb=0, name="co2_discharge")
    direct_co2 = model.addVars(time_steps, lb=0, name="direct_co2")
    stored_co2 = model.addVars(time_steps, lb=0, name="stored_co2")
    surplus = model.addVars(time_steps, lb=0, name="surplus")

    # 目标函数
    model.setObjective(
        wind_annual_cost * wind_capacity +
        pv_annual_cost * pv_capacity +
        battery_energy_annual_cost * battery_energy_capacity +
        battery_storage_annual_cost * battery_storage_capacity +
        electrolyzer_annual_cost_AE * electrolyzer_capacity_AE +
        electrolyzer_annual_cost_PEM * electrolyzer_capacity_PEM +
        ftl_synthesis_annual_cost * ftl_synthesis_capacity +
        heat_pump_annual_cost * heat_pump_capacity +
        thermal_storage_annual_cost * thermal_storage_capacity +
        hydrogen_storage_annual_cost * hydrogen_storage_capacity +
        ftl_storage_annual_cost * ftl_storage_capacity +
        dac_annual_cost * dac_capacity +
        co2_compressor_annual_cost * co2_compressor_capacity +
        co2_storage_annual_cost * co2_storage_capacity +
        surplus_penalty * gp.quicksum(surplus[t] for t in range(time_steps)) +
        battery_charge_penalty * gp.quicksum(battery_charge[t] for t in range(time_steps)) +
        battery_discharge_penalty * gp.quicksum(battery_discharge[t] for t in range(time_steps)) +
        sorbent_cost * gp.quicksum(dac_CO2[t] for t in range(time_steps)) +
        hydrogen_charge_penalty * gp.quicksum(hydrogen_charge[t] for t in range(time_steps)) +
        ftl_charge_penalty * gp.quicksum(ftl_storage_charge[t] for t in range(time_steps)),
        GRB.MINIMIZE
    )

    # 爬坡约束
    for t in range(1, time_steps):
        model.addConstr(electrolyzer_power_AE[t] - electrolyzer_power_AE[t-1] <= Flex_down * electrolyzer_capacity_AE)
        model.addConstr(electrolyzer_power_AE[t] - electrolyzer_power_AE[t-1] >= -Flex_down * electrolyzer_capacity_AE)
        model.addConstr(ftl_synthesis_power[t] - ftl_synthesis_power[t-1] <= Flex_up * ftl_synthesis_capacity)
        model.addConstr(ftl_synthesis_power[t] - ftl_synthesis_power[t-1] >= -Flex_down * ftl_synthesis_capacity)
        model.addConstr(dac_CO2[t] - dac_CO2[t-1] <= Flex_up * dac_capacity)
        model.addConstr(dac_CO2[t] - dac_CO2[t-1] >= -Flex_down * dac_capacity)

    # FTL 供需聚合约束
    start_index = 168
    if aggregation_mode == "hourly":
        for t in range(start_index, time_steps):
            model.addConstr(ftl_synthesis_power[t] + ftl_storage_discharge[t] >= hourly_ftl_demand + ftl_storage_charge[t])
    elif aggregation_mode == "3hourly":
        interval = 3
        n = (time_steps - start_index) // interval
        for i in range(n):
            s = start_index + i * interval
            e = s + interval
            model.addConstr(
                gp.quicksum(ftl_synthesis_power[t] for t in range(s, e)) +
                gp.quicksum(ftl_storage_discharge[t] for t in range(s, e)) ==
                hourly_ftl_demand * interval +
                gp.quicksum(ftl_storage_charge[t] for t in range(s, e))
            )
    elif aggregation_mode == "daily":
        interval = 24
        n = (time_steps - start_index) // interval
        for i in range(n):
            s = start_index + i * interval
            e = s + interval
            model.addConstr(
                gp.quicksum(ftl_synthesis_power[t] for t in range(s, e)) +
                gp.quicksum(ftl_storage_discharge[t] for t in range(s, e)) >=
                0.8 * hourly_ftl_demand * interval +
                gp.quicksum(ftl_storage_charge[t] for t in range(s, e))
            )
    elif aggregation_mode == "weekly":
        interval = 168
        n = (time_steps - start_index) // interval
        for i in range(n):
            s = start_index + i * interval
            e = s + interval
            model.addConstr(
                gp.quicksum(ftl_synthesis_power[t] for t in range(s, e)) +
                gp.quicksum(ftl_storage_discharge[t] for t in range(s, e)) ==
                hourly_ftl_demand * interval +
                gp.quicksum(ftl_storage_charge[t] for t in range(s, e))
            )
    elif aggregation_mode == "monthly":
        interval = 730
        n = (time_steps - start_index) // interval
        for i in range(n):
            s = start_index + i * interval
            e = s + interval
            model.addConstr(
                gp.quicksum(ftl_synthesis_power[t] for t in range(s, e)) +
                gp.quicksum(ftl_storage_discharge[t] for t in range(s, e)) ==
                hourly_ftl_demand * interval +
                gp.quicksum(ftl_storage_charge[t] for t in range(s, e))
            )
    elif aggregation_mode == "annual":
        model.addConstr(
            gp.quicksum(ftl_synthesis_power[t] for t in range(start_index, time_steps)) +
            gp.quicksum(ftl_storage_discharge[t] for t in range(start_index, time_steps)) ==
            hourly_ftl_demand * (time_steps - start_index) +
            gp.quicksum(ftl_storage_charge[t] for t in range(start_index, time_steps))
        )
    else:
        raise ValueError("Unsupported aggregation_mode")

    # 年生产量约束
    model.addConstr(gp.quicksum(ftl_synthesis_power[t] for t in range(time_steps)) >= target_ftl_production)

    # 储存动态
    model.addConstr(ftl_storage_balance[0] == ftl_storage_charge[0] - ftl_storage_discharge[0])
    for t in range(1, time_steps):
        model.addConstr(ftl_storage_balance[t] == ftl_storage_balance[t-1] + ftl_storage_charge[t] - ftl_storage_discharge[t])
        model.addConstr(ftl_storage_balance[t] <= ftl_storage_capacity)

    conversion_factor_AE = electrolyzer_eff_AE * 3.6 / 120
    conversion_factor_PEM = electrolyzer_eff_PEM * 3.6 / 120

    for t in range(time_steps):
        wind_power = wind_output[t] * wind_capacity
        pv_power = pv_output[t] * pv_capacity

        model.addConstr(ftl_synthesis_power[t] + ftl_storage_discharge[t] >= ftl_storage_charge[t])
        model.addConstr(energy_balance[t] <= battery_storage_capacity)
        model.addConstr(battery_charge[t] + battery_discharge[t] <= battery_energy_capacity)
        model.addConstr(hydrogen_storage_balance[t] <= hydrogen_storage_capacity)
        model.addConstr(hydrogen_charge[t] + hydrogen_discharge[t] <= hydrogen_storage_capacity)

        model.addConstr(
            wind_power + pv_power + battery_discharge[t] ==
            (electrolyzer_power_AE[t] + electrolyzer_power_PEM[t]) +
            ftl_synthesis_power[t] * ftl_electricity_ratio +
            battery_charge[t] + hydrogen_charge[t] * hydrogen_storage_elec +
            (heat_pump_output[t] / heat_pump_cop) + dac_electricity[t] + surplus[t] + co2_comp_elec[t],
            name=f"power_balance_{t}"
        )
        if t == 0:
            model.addConstr(energy_balance[t] == battery_charge[t] * battery_charge_efficiency - battery_discharge[t] / battery_discharge_efficiency)
            model.addConstr(hydrogen_storage_balance[t] == hydrogen_charge[t] * hydrogen_storage_efficiency - hydrogen_discharge[t])
            model.addConstr(thermal_storage_level[t] == thermal_charge[t] * thermal_storage_efficiency - thermal_discharge[t] / thermal_storage_efficiency)
            model.addConstr(co2_storage_level[t] == co2_charge[t] - co2_discharge[t])
        else:
            model.addConstr(energy_balance[t] == energy_balance[t-1] + battery_charge[t] * battery_charge_efficiency - battery_discharge[t] / battery_discharge_efficiency)
            model.addConstr(hydrogen_storage_balance[t] == hydrogen_storage_balance[t-1] + hydrogen_charge[t] * hydrogen_storage_efficiency - hydrogen_discharge[t])
            model.addConstr(thermal_storage_level[t] == thermal_storage_level[t-1] + thermal_charge[t] * thermal_storage_efficiency - thermal_discharge[t] / thermal_storage_efficiency)
            model.addConstr(co2_storage_level[t] == co2_storage_level[t-1] + co2_charge[t] - co2_discharge[t])

        model.addConstr(
            electrolyzer_power_AE[t] * conversion_factor_AE +
            electrolyzer_power_PEM[t] * conversion_factor_PEM + hydrogen_discharge[t] >=
            ftl_synthesis_power[t] * ftl_hydrogen_ratio + hydrogen_charge[t],
            name=f"hydrogen_PD_balance_{t}"
        )

        model.addConstr(dac_heat_input[t] == dac_CO2[t] * dac_heat_per_kg)
        model.addConstr(dac_electricity[t] == dac_CO2[t] * dac_elec_per_kg)
        model.addConstr(dac_CO2[t] <= dac_capacity)

        model.addConstr(direct_co2[t] + stored_co2[t] == dac_CO2[t])
        model.addConstr(stored_co2[t] <= co2_compressor_capacity)
        model.addConstr(co2_comp_elec[t] == stored_co2[t] * co2_compression_energy)
        model.addConstr(co2_charge[t] == stored_co2[t])

        model.addConstr(heat_pump_output[t] - thermal_charge[t] + thermal_discharge[t] >= dac_heat_input[t])
        model.addConstr(thermal_storage_level[t] <= thermal_storage_capacity)
        model.addConstr(co2_storage_level[t] <= co2_storage_capacity)

    # 设备容量/最小出力
    model.addConstrs((heat_pump_capacity >= heat_pump_output[t] for t in range(time_steps)))
    model.addConstrs((heat_pump_output[t] >= 0.25 * heat_pump_capacity for t in range(168, time_steps)))
    model.addConstrs((electrolyzer_capacity_AE >= electrolyzer_power_AE[t] for t in range(time_steps)))
    model.addConstrs((electrolyzer_capacity_PEM >= electrolyzer_power_PEM[t] for t in range(time_steps)))
    model.addConstrs((electrolyzer_power_AE[t] >= 0.1 * electrolyzer_capacity_AE for t in range(168, time_steps)))
    model.addConstrs((dac_CO2[t] >= 0.25 * dac_capacity for t in range(168, time_steps)))
    model.addConstrs((ftl_synthesis_power[t] <= ftl_synthesis_capacity for t in range(time_steps)))
    model.addConstrs(
        (ftl_synthesis_power[t] >= 0.25 * ftl_synthesis_capacity for t in range(168, time_steps)),
        name="ftl_min_output"
    )
    model.addConstrs((direct_co2[t] + co2_discharge[t] >= ftl_synthesis_power[t] * ftl_co2_ratio for t in range(time_steps)))
    model.addConstr(gp.quicksum(direct_co2[t] + co2_discharge[t] for t in range(time_steps)) >= target_ftl_production * ftl_co2_ratio)
    model.addConstr(battery_energy_capacity >= 0.5 * electrolyzer_capacity_AE)

    model.optimize()
    return model


In [ ]:
# -*- coding: utf-8 -*-
from typing import Optional, Dict, Tuple, List
import os
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB

# ------------------------------------------------------------
# 依赖：以下函数 / 变量均“已在你的工程中定义”，这里直接调用：
# - create_model_for_product(...)
# - pick_cost_maps(year_tag) -> (wind_map, pv_map)
# - select_params_by_year(year_tag) -> dict
# - normalize_scenario(name) -> str
# - classify_scenario_for_cost(name) -> 'RE'/'WIND'/...
# - scenario_output_folder(name) -> str
# - pick_year_tag_from_filename(fn) -> '2030'/'2040'/'2050'/'2035'
# - out_root, cf_dir, YEAR_FILTER
# - _product_target_from_params(prod_key, params)
# - _extract_capacities(model, prod_key)
# ------------------------------------------------------------

def pick_year_tag_from_filename(fn: str) -> str:
    """返回 '2030' / '2040' / '2050' / '2035'(默认)"""
    s = fn.lower()
    if "2050" in s: return "2050"
    if "2040" in s: return "2040"
    if "2030" in s: return "2030"
    return "2035"

def _pick_one_cf_file_for_scenario(scen_raw: str) -> Optional[str]:
    """
    选择该 scen 的一个 CF 文件来用于敏感性分析。
    优先 2023（无后缀），若缺则 2022，再缺则 2021；都没有则返回 None。
    """
    base = f"median_points_cf_timeseries_{scen_raw}"
    cands = [
        os.path.join(cf_dir, f"{base}.csv"),           # 2023（默认）
        os.path.join(cf_dir, f"{base}_2022.csv"),
        os.path.join(cf_dir, f"{base}_2021.csv"),
    ]
    for p in cands:
        if os.path.exists(p):
            return p
    return None


def _build_one_model_from_group_for_cost_case(
    group: pd.DataFrame,
    prod_key: str,
    wind_cost: float,
    pv_cost: float,
    params: dict
) -> Tuple[gp.Model, int]:
    """
    与你原有 _build_one_model_from_group 一致，只是函数名更语义化一点
    """
    group = group.sort_values('hour')
    wind_output = group['CF_wind_avg'].to_numpy()
    pv_output   = group['CF_pv_avg'].to_numpy()
    T = len(wind_output)
    model = create_model_for_product(prod_key, wind_output, pv_output, wind_cost, pv_cost, params)
    return model, T


def _apply_cost_case_on_maps_and_params(
    base_wind_cost: float,
    base_pv_cost: float,
    base_params: dict,
    case_overrides: Dict
) -> Tuple[float, float, dict]:
    """
    将“敏感性情景”的改动应用到（风/光 CAPEX）与（其他参数）上。
    规则：
      - 支持乘数键：'wind_capex_mult', 'pv_capex_mult'
      - 支持绝对值覆盖：'wind_capex_abs', 'pv_capex_abs'
      - 对于任意 params 键（例如 'battery_energy_cost'、'electrolyzer_cost_AE' 等），
        直接把 case_overrides 中的同名键写入 params 覆盖（数值型）。
    """
    wind_cost = float(base_wind_cost)
    pv_cost   = float(base_pv_cost)
    params    = dict(base_params)  # 浅拷贝

    # --- 先乘数，再绝对值覆盖（若给了绝对值，则最终以绝对值为准）---
    if 'wind_capex_mult' in case_overrides:
        try:
            wind_cost *= float(case_overrides['wind_capex_mult'])
        except Exception:
            pass
    if 'pv_capex_mult' in case_overrides:
        try:
            pv_cost *= float(case_overrides['pv_capex_mult'])
        except Exception:
            pass

    if 'wind_capex_abs' in case_overrides:
        try:
            wind_cost = float(case_overrides['wind_capex_abs'])
        except Exception:
            pass
    if 'pv_capex_abs' in case_overrides:
        try:
            pv_cost = float(case_overrides['pv_capex_abs'])
        except Exception:
            pass

    # --- 其他参数：若键名恰好是 params 的键，则覆盖 ---
    for k, v in case_overrides.items():
        if k in ('wind_capex_mult', 'pv_capex_mult', 'wind_capex_abs', 'pv_capex_abs'):
            continue
        if k in params:
            try:
                params[k] = float(v)
            except Exception:
                # 也允许 bool / str 等，但通常为数值
                params[k] = v

    return wind_cost, pv_cost, params


def compare_cost_sensitivities_for_scenario(
    scen_raw: str,
    sensitivity_cases: Dict[str, Dict],
    include_base_case: bool = True,
    only_one_node: Optional[Dict[str, str]] = None,
    stop_after_first: bool = False
):
    """
    对给定 scen_raw，固定一套 CF（优先 2023，缺则 2022/2021），在“相同节点与相同 CF”下，
    对一组成本/参数敏感性情景进行求解对比（记录 LCO/各装机/储能等），不出图，仅输出 CSV。

    参数：
      - sensitivity_cases: 形如
          {
            "Base_x1": {},  # 空字典表示与基准相同（若 include_base_case=False，也可不写）
            "Wind-20%": {"wind_capex_mult": 0.8},
            "PV+20%":   {"pv_capex_mult": 1.2},
            "LowElec":  {"electrolyzer_cost_AE": 300, "electrolyzer_cost_PEM": 350},
            "Battx1.5": {"battery_energy_cost": 150, "battery_storage_cost": 150},
          }
      - include_base_case: 若 True，则在跑 cases 之前自动加入一个 "Base" 基准。
      - only_one_node: 可选，筛选一个节点（如 {"product":"ammonia","continent":"Europe","cluster":0}）
      - stop_after_first: True 则只跑筛中的第一个节点（用于快速测试）
    输出：
      <out_root>/Year_<year_tag>/<scenario_folder>/sensitivity_compare/<prod>_<continent>_cluster<k>/cost_sensitivity_summary.csv
    """
    cf_path = _pick_one_cf_file_for_scenario(scen_raw)
    if not cf_path or (not os.path.exists(cf_path)):
        print(f"· {scen_raw}: 未找到可用的 CF 文件（2023/2022/2021 均缺），跳过。", flush=True)
        return

    # 基于场景名确定规划年口径（决定 CAPEX 映射 & 参数集）
    year_tag = YEAR_FILTER  # 与建模脚本一致：按该口径选择成本映射与参数（2040→2035）
    wind_map, pv_map = pick_cost_maps(year_tag)
    params_base = select_params_by_year(year_tag)

    scen_norm   = normalize_scenario(scen_raw)
    scen_type   = classify_scenario_for_cost(scen_raw)
    scen_folder = scenario_output_folder(scen_raw)

    print(f"\n=== 成本敏感性 | {scen_norm} | 口径={year_tag} | 类型={scen_type} | CF文件={os.path.basename(cf_path)} ===", flush=True)

    # 读取 CF：与“气候对比”不同，这里只用 1 年的 CF；但仍按节点分组逐个建模
    cf_df = pd.read_csv(cf_path)
    if 'scenario' not in cf_df.columns: cf_df['scenario'] = scen_norm
    if 'product'  not in cf_df.columns: cf_df['product']  = 'methanol'

    group_cols = [c for c in ['scenario','product','continent','cluster','latitude','longitude','region'] if c in cf_df.columns]
    if not group_cols:
        print(f"· {scen_raw}: 缺少必要分组键，跳过。", flush=True); return

    # 所有节点（或筛后的某个节点）
    groups_list = list(cf_df.groupby(group_cols, sort=False))
    if only_one_node:
        groups_list = [
            (keys, grp) for (keys, grp) in groups_list
            if all(k in dict(zip(group_cols, keys if isinstance(keys, tuple) else (keys,))) for k in only_one_node)
            and all(str(dict(zip(group_cols, keys if isinstance(keys, tuple) else (keys,)))[k]) == str(v)
                    for k, v in only_one_node.items())
        ]
    if not groups_list:
        print("· 未匹配到任何节点，跳过。", flush=True); return

    # 输出目录
    scen_out_root = os.path.join(out_root, f"Year_{year_tag}", scen_folder, "sensitivity_compare")
    os.makedirs(scen_out_root, exist_ok=True)

    # 如需附加一个标准 Base
    cases = dict(sensitivity_cases or {})
    if include_base_case and ("Base" not in cases):
        cases = {"Base": {}} | cases  # Python 3.9+: dict 合并

    # 逐节点进行敏感性对比
    for gi, (keys, group) in enumerate(groups_list, 1):
        key_map = dict(zip(group_cols, keys if isinstance(keys, tuple) else (keys,)))
        prod_key = (key_map.get('product') or 'methanol').lower()
        cont     = key_map.get('continent', 'Unknown')
        cl       = key_map.get('cluster', 'X')
        region_code = int(key_map.get('region')) if key_map.get('region') is not None else None

        print(f"\n▶ 节点 {gi}/{len(groups_list)} | {scen_norm} | {prod_key.upper()} | {cont} | 簇={cl}", flush=True)

        # 基准成本（按 RegionWind/RegionalWind 的 PV=225 规则）
        base_wind_cost = wind_map.get(region_code, 800)
        if scen_type == 'WIND':
            base_pv_cost = 225
        else:
            base_pv_cost = pv_map.get(region_code, 350)

        # 记录每个案例的结果
        recs: List[Dict] = []

        for case_name, case_overrides in cases.items():
            wind_cost_i, pv_cost_i, params_i = _apply_cost_case_on_maps_and_params(
                base_wind_cost, base_pv_cost, params_base, case_overrides
            )

            try:
                model, T = _build_one_model_from_group_for_cost_case(
                    group, prod_key, wind_cost_i, pv_cost_i, params_i
                )
                model.optimize()
                if model.status != GRB.OPTIMAL:
                    print(f"  · {case_name}: 非最优（{model.status}），跳过。", flush=True)
                    continue
            except Exception as e:
                print(f"  · {case_name}: 构建/求解失败：{e}", flush=True)
                continue

            annual_cost = float(model.ObjVal)
            tgt = _product_target_from_params(prod_key, params_i)
            lco = (annual_cost / tgt) if tgt and tgt > 0 else np.nan

            caps = _extract_capacities(model, prod_key)
            rec = {"case": case_name, "Annual_cost_$": annual_cost, f"LCO_{prod_key}_$/t": lco,
                   "wind_capex_used": wind_cost_i, "pv_capex_used": pv_cost_i}
            rec.update(caps)
            # 如需把参数覆盖也一起记录到 CSV 中，可选追加：
            for k, v in case_overrides.items():
                if k in ("wind_capex_mult","pv_capex_mult","wind_capex_abs","pv_capex_abs"):
                    rec[k] = v
                elif k in params_base:
                    rec[f"param_{k}"] = v

            recs.append(rec)

        if not recs:
            print("  · 所有案例均失败/缺失，跳过该节点。", flush=True)
            continue

        df_res = pd.DataFrame(recs).set_index("case")
        node_dir = os.path.join(scen_out_root, f"{prod_key}_{cont}_cluster{cl}")
        os.makedirs(node_dir, exist_ok=True)

        out_csv = os.path.join(node_dir, "cost_sensitivity_summary.csv")
        df_res.to_csv(out_csv)
        print(f"    ✓ 已保存：{out_csv}", flush=True)

        if stop_after_first:
            print("· 单节点测试完成（成本敏感性），提前结束。", flush=True)
            break


# ========================= 示例调用（按需放到你的主流程中）========================
# 你可以在主脚本处理完每个 scen_raw 后，追加如下调用（或集中跑）：
#
# sensitivity_cases = {
#     # 空对象可省略（若 include_base_case=True 会自动添加 Base）
#     # "Base": {},
#     "Wind-20%": {"wind_capex_mult": 0.8},
#     "PV+20%":   {"pv_capex_mult": 1.2},
#     "LowElec":  {"electrolyzer_cost_AE": 300, "electrolyzer_cost_PEM": 350},
#     "Battx1.5": {"battery_energy_cost": 150, "battery_storage_cost": 150},
#     "HP_COP+10%": {"heat_pump_cop": 3.52},  # 示例：提升热泵COP
# }
#
# compare_cost_sensitivities_for_scenario(
#     scen_raw="RegionRE_LowAE",                # 举例：和你的 CF 命名一致的 scen 片段
#     sensitivity_cases=sensitivity_cases,
#     include_base_case=True,
#     only_one_node=None,                       # 或 {'product':'ammonia','continent':'Europe','cluster':0}
#     stop_after_first=False
# )
#
# 也可以在循环中对多个 scen_raw 批量调用。
# ==============================================================================



# 进行优化

In [ ]:
# -*- coding: utf-8 -*-
"""
批量生成 8760 典型周供需图 + 年度热力图（中文版） + 气候年对比（2021/2022/2023）
- 自动适配文件名中的 2030/2040/2050：
  * 风/光成本分别读取 RegionalCost.csv 中的 2030ATB / 2035ATB(用于2040) / 2050ATB 列；
  * 参数集自动切换为 params_2030 / params_2050（2040 暂用 params_base，或自行扩展 params_2040）。
- Region/Regional 归并（RegionRE/RegionalRE；RegionWind/RegionalWind）
- RegionWind/RegionalWind：PV 成本统一 225；其余用区域映射（按年列）
- Nature Climate Change 风格（近似），数学下标（H₂/CO₂）
- SOC 无模型直接输出时：按充/放电自行计算；若能读取容量则按容量归一化，否则按自身范围归一化到[0,1]
- 年度热力图：Battery 改为 SOC，新增 H₂ Storage SOC 与 Methanol/FTL/Ammonia Storage SOC
- 新增：气候年对比（读取 _2021/_2022/默认(2023) 三个逐时 CF 对同一节点建模，输出 LCO 与装机对比）
"""

import os
import re
import glob
from typing import Optional, Dict, Iterable, List, Tuple

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import colors, cm
import gurobipy as gp
from gurobipy import GRB

from datetime import datetime
try:
    from timezonefinder import TimezoneFinder  # pip install timezonefinder
except Exception:
    TimezoneFinder = None
try:
    from zoneinfo import ZoneInfo  # Python 3.9+
except Exception:
    ZoneInfo = None


# =====================================================
# 0) 路径与全局设置
# =====================================================
cf_dir   = r"D:\OneDrive\POTSED\03 Cluster"
out_root = r"D:\OneDrive\POTSED\8760Supply&Demand\Global_cn"
os.makedirs(out_root, exist_ok=True)

# 区域资本成本（风/光）
regional_cost_path = r"D:\OneDrive\POTSED\DataOutput\RegionalCost.csv"

# ========== 仅测试开关（可选） ==========
ONLY_ONE_SCENARIO: Optional[str] = None          # 正则筛选：例如 r"RegionRE"
ONLY_ONE_NODE: Optional[Dict[str, str]] = None   # 例如 {'product':'ammonia','continent':'Europe','cluster':0}
STOP_AFTER_FIRST_NODE: bool = True

# 仅跑指定“情景年份标签”（支持 '2030' / '2040' / '2050' / '2035'）
YEAR_FILTER = "2035"

# ========= y 轴留白（方法3）可调项 =========
Y_MARGIN_FRAC_POWER: float = 0.08
Y_MARGIN_FRAC_H2: float    = 0.08

# =====================================================
# 画风：NCC + 中文字体
# =====================================================
def apply_cn_style():
    # 优先中文字体，自动回退
    font_candidates = [
        "Microsoft YaHei", "SimHei", "Noto Sans CJK SC",
        "Source Han Sans SC", "Alibaba PuHuiTi 2.0",
        "WenQuanYi Zen Hei", "Arial Unicode MS", "DejaVu Sans"
    ]
    plt.rcParams.update({
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "figure.facecolor": "white",
        "savefig.facecolor": "white",
        "font.family": "sans-serif",
        "font.sans-serif": font_candidates,
        "axes.unicode_minus": False,  # 解决负号显示
        "pdf.fonttype": 42, "ps.fonttype": 42,
        "text.usetex": False, "mathtext.fontset": "stix", "mathtext.default": "rm",
        "axes.facecolor": "white", "axes.edgecolor": "black", "axes.linewidth": 0.8,
        "axes.labelsize": 9, "axes.titlesize": 10, "axes.titlepad": 6,
        "xtick.labelsize": 8, "ytick.labelsize": 8,
        "xtick.direction": "in", "ytick.direction": "in",
        "xtick.major.size": 3, "ytick.major.size": 3,
        "xtick.major.width": 0.8, "ytick.major.width": 0.8,
        "grid.linestyle": (0, (3, 3)), "grid.color": "0.85", "grid.linewidth": 0.6,
        "legend.fontsize": 8, "legend.frameon": True, "legend.framealpha": 0.9, "legend.edgecolor": "0.8",
        "lines.linewidth": 1.2,
    })
apply_cn_style()

# 典型周（0 为第一周）
season_weeks = {
    '冬季': 1,    # 1月首周
    '春季': 14,   # 4月首周
    '夏季': 27,   # 7月首周
    '秋季': 40    # 10月首周
}

# 颜色
wind_color       = '#4F81BD'
pv_color         = '#C0504D'
battery_color    = '#9BBB59'
ae_color         = '#8064A2'
pem_color        = '#F79646'
synth_color      = '#4BACC6'
bat_charge_color = '#F05542'
hydrogen_color   = '#8F5CC1'
heatpump_color   = '#4C9F70'
dac_color        = '#D99694'
surplus_color    = '#B0B0B0'
co2_color        = '#696969'

# 数学下标
H2  = r'$H_2$'
CO2 = r'$CO_2$'

# ====================工具：时序调整========================
def infer_utc_offset_hours(lat: float, lon: float, year: int = 2040) -> int:
    """
    返回该点在给定年份中段（7月1日中午）的本地UTC时差（单位：小时，四舍五入到整数）。
    优先使用真实时区（timezonefinder + zoneinfo），失败则经度近似 lon/15。
    """
    # 真实时区（若可用）
    if TimezoneFinder is not None and ZoneInfo is not None:
        try:
            tf = TimezoneFinder()
            tzname = tf.timezone_at(lat=lat, lng=lon)
            if tzname:
                dt = datetime(year, 7, 1, 12, 0, 0)
                tz = ZoneInfo(tzname)
                offset_sec = tz.utcoffset(dt).total_seconds()
                return int(round(offset_sec / 3600.0))
        except Exception:
            pass

    # 近似：经度 / 15 取整，并限制到 [-12, +14]
    approx = int(round(lon / 15.0))
    return max(-12, min(14, approx))


def roll_series_dict(d: dict, shift_hours: int, keys: list[str]) -> None:
    """
    原地把 d 中指定 keys 的一维 numpy 序列循环平移 shift_hours。
    shift_hours>0 表示向右滚动（即“本地时间 = UTC + shift”）。
    """
    if not shift_hours:
        return
    for k in keys:
        if k in d and d[k] is not None:
            d[k] = np.roll(np.asarray(d[k], dtype=float), shift_hours)


# =================== 工具：y 轴留白 ===================
def set_y_margin(ax: plt.Axes, positive_stack: np.ndarray, negative_stack: np.ndarray, pad_frac: float):
    pos_max = float(np.nanmax(positive_stack)) if positive_stack.size else 0.0
    neg_max = float(np.nanmax(negative_stack)) if negative_stack.size else 0.0
    top, bottom = pos_max, -neg_max
    if top == bottom:
        span = abs(top) + 1.0
        ax.set_ylim(bottom - pad_frac * span, top + pad_frac * span)
    else:
        span = top - bottom
        pad = pad_frac * span
        ax.set_ylim(bottom - pad, top + pad)

# =================== 工具：批量抓取变量 ===================
VAR_TEMPLATES: Dict[str, List[str]] = {
    "electrolyzer_AE":     ["electrolyzer_power_AE[{t}]"],
    "electrolyzer_PEM":    ["electrolyzer_power_PEM[{t}]"],
    "synthesis_power":     ["ammonia_synthesis_power[{t}]", "methanol_synthesis_power[{t}]", "FTL_synthesis_power[{t}]"],
    "dac_elec":            ["dac_electricity[{t}]"],
    "heat_pump_output":    ["heat_pump_output[{t}]"],
    "battery_charge":      ["battery_charge[{t}]"],
    "battery_discharge":   ["battery_discharge[{t}]"],
    "hydrogen_charge":     ["hydrogen_charge[{t}]"],
    "hydrogen_discharge":  ["hydrogen_discharge[{t}]"],
    "product_charge":      ["methanol_storage_charge[{t}]", "FTL_storage_charge[{t}]",
                            "ammonia_storage_charge[{t}]", "product_storage_charge[{t}]"],
    "product_discharge":   ["methanol_storage_discharge[{t}]", "FTL_storage_discharge[{t}]",
                            "ammonia_storage_discharge[{t}]", "product_storage_discharge[{t}]"],
    "surplus":             ["surplus[{t}]"],
    "co2_comp":            ["co2_comp_elec[{t}]"],
    "battery_balance":     ["energy_balance[{t}]"],
    "hydrogen_storage_balance": ["hydrogen_storage_balance[{t}]"],
    "product_storage_balance":  ["methanol_storage_balance[{t}]","FTL_storage_balance[{t}]","ammonia_storage_balance[{t}]"]
}

VAR_CAPACITY: Dict[str, List[str]] = {
    "battery":  ["battery_storage_capacity", "battery_storage_capacity_kWh"],
    "hydrogen": ["hydrogen_storage_capacity", "hydrogen_storage_capacity_kgH2"],
    "product":  ["methanol_storage_capacity", "FTL_storage_capacity", "ammonia_storage_capacity"],
}

def get_series_by_templates(model: gp.Model, templates: Iterable[str], T: int) -> Optional[np.ndarray]:
    vals = np.zeros(T, dtype=float); found_any = False
    for t in range(T):
        v = None
        for temp in templates:
            var = model.getVarByName(temp.format(t=t))
            if var is not None:
                v = var.x; found_any = True; break
        vals[t] = float(v) if v is not None else 0.0
    return vals if found_any else None

def ensure_series(arr: Optional[np.ndarray], T: int, fill: float = 0.0) -> np.ndarray:
    if arr is None:
        return np.full(T, fill, dtype=float)
    a = np.asarray(arr, dtype=float).ravel()
    if a.size < T:
        out = np.full(T, fill, dtype=float); out[:a.size] = a; return out
    return a[:T]

def get_first_var_value(model: gp.Model, names: List[str]) -> Optional[float]:
    for nm in names:
        var = model.getVarByName(nm)
        if var is not None:
            try:
                return float(var.x)
            except Exception:
                pass
    return None

# ========== 年度热力图工具 ==========
def reshape_to_day(series: np.ndarray, hours_per_day: int = 24) -> np.ndarray:
    T = int(series.size)
    days = int(np.ceil(T / hours_per_day))
    mat = np.full((days, hours_per_day), np.nan, dtype=float)
    mat.flat[:T] = series
    return mat

def month_tick_positions(days: int) -> Tuple[List[int], List[str]]:
    month_days = [31,28,31,30,31,30,31,31,30,31,30,31]
    edges = np.cumsum([0]+month_days)
    centers = [(edges[i] + edges[i+1])//2 for i in range(12)]
    scale = days / 365.0
    pos = [int(c * scale) for c in centers if int(c*scale) < days]
    labels = ["1月","2月","3月","4月","5月","6月","7月","8月","9月","10月","11月","12月"][:len(pos)]
    return pos, labels

def normalize01(mat: np.ndarray) -> np.ndarray:
    m = np.nanmin(mat); M = np.nanmax(mat)
    if not np.isfinite(m) or not np.isfinite(M) or M <= m:
        return np.nan_to_num(mat, nan=0.0)
    return (mat - m) / (M - m)

def _truncate_cmap(base_cmap, lo=0.35, hi=0.65, n=256):
    new_colors = base_cmap(np.linspace(max(0.0, lo), min(1.0, hi), n))
    return colors.LinearSegmentedColormap.from_list(
        f"{getattr(base_cmap,'name','cmap')}_slice_{lo:.2f}_{hi:.2f}", new_colors
    )

def plot_annual_heatmap_panel(
    items,
    title_prefix,
    out_path,
    cmap: str = "YlGnBu",
    cmap_slice: tuple | None = (0., 0.95),
    norm_range: tuple | None = None,
    # 字体
    fs_title: int = 16,
    fs_rowlab: int = 28,
    fs_axislabel: int = 28,
    fs_tick: int = 28,
    fs_cbar: int =28,
    # 布局控制
    hspace: float  = 0.10,      # 行间距
    rowlab_x: float = 1.01,     # 行标题初始位置（轴坐标）
    # 先给个较宽的右边距，后面会自动再调
    left: float = 0.08, right: float = 0.90, top: float = 0.965, bottom: float = 0.12,
    # colorbar 位置（figure 坐标，绝对控制）
    cbar_left: float = 0.08, cbar_bottom: float = 0.00, cbar_width: float = 0.87, cbar_height: float = 0.02,
):
    import numpy as np
    from matplotlib import colors

    # 过滤无效项
    valid = []
    for lab, ser in items:
        if ser is None:
            continue
        arr = np.asarray(ser, dtype=float)
        if np.allclose(arr, 0.0, atol=1e-12):
            continue
        valid.append((lab, arr))
    if not valid:
        return False

    days = int(np.ceil(max(len(s) for _, s in valid) / 24.0))
    month_days = [31,28,31,30,31,30,31,31,30,31,30,31]
    edges = np.cumsum([0]+month_days)
    centers = [(edges[i] + edges[i+1])//2 for i in range(12)]
    scale = days / 365.0
    xticks = [int(c * scale) for c in centers if int(c*scale) < days]
    xlabels = ["1月","2月","3月","4月","5月","6月","7月","8月","9月","10月","11月","12月"][:len(xticks)]

    n = len(valid)
    fig = plt.figure(figsize=(15, 1.75 * n + 0.9))
    fig.text(0.5, 0.992, f"{title_prefix} — 年度归一化运行剖面",
             ha="center", va="top", fontsize=fs_title)

    gs = fig.add_gridspec(nrows=n, ncols=1, hspace=hspace)
    fig.subplots_adjust(left=left, right=right, top=top, bottom=bottom)

    base = plt.get_cmap(cmap)
    cmap = colors.LinearSegmentedColormap.from_list(
        f"{getattr(base,'name','cmap')}_slice", base(np.linspace(float(cmap_slice[0]), float(cmap_slice[1]), 256))
    ) if cmap_slice else base
    norm = colors.Normalize(vmin=0.0, vmax=1.0) if norm_range is None \
           else colors.Normalize(vmin=float(norm_range[0]), vmax=float(norm_range[1]), clip=True)

    axes = []
    right_texts = []
    last_img = None
    for i, (lab, ser) in enumerate(valid):
        ax = fig.add_subplot(gs[i, 0], sharex=axes[0] if axes else None)
        axes.append(ax)

        T = len(ser)
        mat = np.full((int(np.ceil(T/24.0)), 24), np.nan, float)
        mat.flat[:T] = ser
        mmin, mmax = np.nanmin(mat), np.nanmax(mat)
        mat_n = (mat - mmin) / (mmax - mmin) if np.isfinite(mmin) and np.isfinite(mmax) and mmax>mmin else np.nan_to_num(mat)

        last_img = ax.imshow(mat_n.T, origin='lower', aspect='auto', cmap=cmap, norm=norm)

        ax.set_yticks([0, 8, 16])
        ax.tick_params(axis='both', labelsize=fs_tick)
        ax.set_ylabel("小时", rotation=90, labelpad=10, fontsize=fs_axislabel-6)

        # —— 右侧竖排行标题（不裁剪）——
        t = ax.text(
            1.02, 0.5, lab,
            transform=ax.transAxes,
            rotation=90,
            rotation_mode='anchor',
            va='center',
            ha='center',
            multialignment='center',
            fontsize=fs_rowlab,
            linespacing=1.0,
            clip_on=False
        )
        right_texts.append(t)

        if i < n - 1:
            ax.tick_params(labelbottom=False)

    axes[-1].set_xticks(xticks)
    axes[-1].set_xticklabels(xlabels, fontsize=fs_tick)
    axes[-1].set_xlabel("月份", labelpad=6, fontsize=fs_axislabel)

    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    max_w_px = 0
    for t in right_texts:
        bb = t.get_window_extent(renderer=renderer)
        max_w_px = max(max_w_px, bb.width)
    fig_w_in = fig.get_size_inches()[0]
    fig_dpi  = fig.dpi
    pad_frac = 0.02
    margin_needed = (max_w_px / (fig_w_in * fig_dpi)) + pad_frac
    new_right = max(0.60, 1.0 - max(0.07, margin_needed))
    fig.subplots_adjust(right=new_right)

    fig.savefig(out_path, dpi=600)
    plt.close(fig)
    return True


# =====================================================
# 1) 读取区域成本映射（仅一次）
# =====================================================
print("▶ 读取区域资本成本表...", flush=True)
if os.path.exists(regional_cost_path):
    regional_cost_df = pd.read_csv(regional_cost_path)
    # 预制三列：2030/2035/2050（2040 复用 2035）
    def _make_map(tech: str, col: str):
        sub = regional_cost_df[
            (regional_cost_df["Type"] == tech) &
            (regional_cost_df["Cost Type 1"] == "Total installed cost")
        ]
        if col not in sub.columns:
            print(f"  ⚠ 成本列 {col} 不在表内，稍后将自动回退。")
            return {}
        return sub.set_index("region_code")[col].to_dict()

    wind_map_2030 = _make_map("Onshore Wind", "2030ATB")
    wind_map_2035 = _make_map("Onshore Wind", "2035ATB")
    wind_map_2050 = _make_map("Onshore Wind", "2050ATB")

    pv_map_2030   = _make_map("Solar PV", "2030ATB")
    pv_map_2035   = _make_map("Solar PV", "2035ATB")
    pv_map_2050   = _make_map("Solar PV", "2050ATB")

    print(f"   ✓ 风区数：2030={len(wind_map_2030)}, 2035={len(wind_map_2035)}, 2050={len(wind_map_2050)}")
    print(f"   ✓ 光区数：2030={len(pv_map_2030)}, 2035={len(pv_map_2035)}, 2050={len(pv_map_2050)}")
else:
    wind_map_2030 = wind_map_2035 = wind_map_2050 = {}
    pv_map_2030   = pv_map_2035   = pv_map_2050   = {}
    print(f"   ⚠ 未找到区域成本文件：{regional_cost_path}，将使用默认值。", flush=True)

# =====================================================
# 2) 扫描所有情景的 CF 时序文件
# =====================================================

# ========= 情景名工具（提前定义，供后续使用）=========
def normalize_scenario(name: str) -> str:
    return re.sub(r'(?i)Regional', 'Region', name)

def classify_scenario_for_cost(name: str) -> str:
    s = name.lower()
    if re.search(r'region(al)?re', s):   return 'RE'
    if re.search(r'region(al)?wind', s): return 'WIND'
    return 'OTHER'

def scenario_output_folder(name: str) -> str:
    typ = classify_scenario_for_cost(name)
    if typ == 'RE':   return 'RegionRE'
    if typ == 'WIND': return 'RegionWind'
    return normalize_scenario(name)

def pick_year_tag_from_filename(fn: str) -> str:
    """返回 '2030' / '2040' / '2050' / '2035'(默认)"""
    s = fn.lower()
    if "2050" in s: return "2050"
    if "2040" in s: return "2040"
    if "2030" in s: return "2030"
    return "2035"

def pick_climate_year_from_path(path_or_name: str) -> int:
    """
    从时序文件名判断气候年：
    ..._2021.csv → 2021
    ..._2022.csv → 2022
    ... .csv(无后缀) → 2023
    """
    name = os.path.basename(path_or_name).lower()
    if name.endswith("_2021.csv"): return 2021
    if name.endswith("_2022.csv"): return 2022
    # 默认：无后缀的当作 2023
    return 2023

def pick_cost_maps(year_tag: str):
    # 2040 复用 2035 映射
    if year_tag == "2050":
        return wind_map_2050, pv_map_2050
    if year_tag == "2030":
        return wind_map_2030, pv_map_2030
    if year_tag == "2040":
        return wind_map_2035, pv_map_2035
    return wind_map_2035, pv_map_2035

cf_files = sorted(glob.glob(os.path.join(cf_dir, "median_points_cf_timeseries_*.csv")))
print(f"▶ 在 {cf_dir} 找到 {len(cf_files)} 个情景文件。", flush=True)
# —— 仅保留指定年份（根据文件名中的年份标签判断）——
cf_files = [fp for fp in cf_files if pick_year_tag_from_filename(os.path.basename(fp)) == YEAR_FILTER]
print(f"▶ 仅保留 {YEAR_FILTER} 年情景：{len(cf_files)} 个。", flush=True)
if not cf_files:
    raise FileNotFoundError(f"未找到 {YEAR_FILTER} 年的 median_points_cf_timeseries_*.csv")

# =====================================================
# 3) 参数集：基础 + 2030/2050 占位（可按需覆盖）
# =====================================================

# 是否把序列转换到当地时区再绘图
ADJUST_TO_LOCAL_TZ = True

params_base = {
    "target_methanol_production": 10e3,
    "battery_energy_cost": 100,
    "battery_storage_cost": 100,
    "battery_charge_efficiency": 0.985,
    "battery_discharge_efficiency": 0.975,
    "battery_charge_penalty": 1e-5,
    "battery_discharge_penalty": 0,
    "electrolyzer_cost_AE": 250,
    "electrolyzer_cost_PEM": 500,
    "electrolyzer_eff_AE": 0.65,
    "electrolyzer_eff_PEM": 0.70,
    "methanol_synthesis_cost": 5700,
    "hydrogen_storage_cost": 500,
    "hydrogen_storage_elec": 0,
    "hydrogen_charge_penalty": 1e-5,
    "hydrogen_storage_efficiency": 1.0,
    "methanol_storage_cost_param": 0.056,
    "methanol_charge_penalty": 1e-5,
    "discount_rate": 0.07,
    "battery_lifetime": 20,
    "wind_lifetime": 30,
    "pv_lifetime": 40,
    "electrolyzer_lifetime_AE": 25,
    "electrolyzer_lifetime_PEM": 25,
    "methanol_synthesis_lifetime": 30,
    "hydrogen_storage_lifetime": 30,
    "methanol_storage_lifetime": 30,
    "heat_pump_cost": 950,
    "heat_pump_cop": 3.2,
    "heat_pump_lifetime": 20,
    "thermal_storage_cost": 30,
    "thermal_storage_efficiency": 0.98,
    "thermal_storage_lifetime": 30,
    "dac_cost": 4000,
    "dac_elec_per_kg": 0.25,
    "dac_heat_per_kg": 1.0,
    "dac_lifetime": 30,
    "sorbent_cost":0.012,
    "co2_compressor_cost": 365,
    "co2_compression_energy": 0.1,
    "co2_compressor_lifetime": 20,
    "co2_storage_cost": 24,
    "co2_storage_lifetime": 30,
    "Flex_up": 0.02, "Flex_mid": 0.1, "Flex_down": 0.2,
    "surplus_penalty": 1e-5,
    "aggregation_mode": "daily",
    # 供 FTL/氨 的默认占位
    "target_ftl_production": 10e3,
    "ftl_synthesis_cost": 15400,
    "ftl_synthesis_lifetime": 25,
    "ftl_hydrogen_ratio": 0.49,
    "ftl_co2_ratio": 3.6,
    "ftl_electricity_ratio": 0.3,
    "ftl_storage_cost_param": 0.056,
    "ftl_storage_lifetime": 30,
    "ftl_charge_penalty": 1e-5,
    "target_ammonia_production": 10e3,
    "ammonia_synthesis_cost": 6500,
    "ammonia_synthesis_lifetime": 30,
    "ammonia_storage_cost_param": 0.59,
    "ammonia_storage_lifetime": 30,
}

params_2030 = dict(params_base)
params_2030.update({
    "battery_energy_cost": 200,
    "battery_storage_cost": 200,
    "electrolyzer_cost_AE": 430,
    "electrolyzer_cost_PEM": 650,
    "electrolyzer_eff_AE": 0.65,
    "electrolyzer_eff_PEM": 0.70,
    "ftl_synthesis_cost": 22000,
    "ftl_hydrogen_ratio": 0.51,
    "ftl_co2_ratio": 3.9,
    "ftl_electricity_ratio": 0.3,
    "methanol_synthesis_cost": 6500,
    "ammonia_synthesis_cost": 8000,
    "hydrogen_storage_cost": 900,
    "dac_cost": 7800,
    "dac_elec_per_kg": 0.31,
    "dac_heat_per_kg": 1.3,
    "sorbent_cost":0.02,
    "heat_pump_cost": 1000,
})

params_2050 = dict(params_base)
params_2050.update({
    "battery_energy_cost": 50,
    "battery_storage_cost": 50,
    "electrolyzer_cost_AE": 250,
    "electrolyzer_cost_PEM": 350,
    "electrolyzer_eff_AE": 0.65,
    "electrolyzer_eff_PEM": 0.70,
    "ftl_synthesis_cost": 9400,
    "ftl_hydrogen_ratio": 0.48,
    "ftl_co2_ratio": 3.3,
    "ftl_electricity_ratio": 0.3,
    "methanol_synthesis_cost": 5200,
    "ammonia_synthesis_cost": 5000,
    "hydrogen_storage_cost": 300,
    "dac_cost": 3000,
    "dac_elec_per_kg": 0.22,
    "dac_heat_per_kg": 1.0,
    "sorbent_cost":0.009,
    "heat_pump_cost": 900,
})

def select_params_by_year(year_tag: str) -> dict:
    if str(year_tag) == "2050":
        return params_2050
    if str(year_tag) == "2030":
        return params_2030
    return params_base


# ========== 模型路由（与你原版一致；需存在 optimize_* 函数） ==========
def create_model_for_product(prod_key: str,
                             wind_output: np.ndarray,
                             pv_output: np.ndarray,
                             wind_cost: float,
                             pv_cost: float,
                             p: dict):
    prod_key = (prod_key or "methanol").lower()
    # 这里假定你在同文件或可导入处定义了 optimize_methanol_system / optimize_ftl_system / optimize_ammonia_system
    if prod_key == "methanol":
        return optimize_methanol_system(
            wind_output, pv_output,
            p["target_methanol_production"],
            wind_cost, pv_cost,
            p["battery_energy_cost"], p["battery_storage_cost"],
            p["battery_charge_efficiency"], p["battery_discharge_efficiency"],
            p["battery_charge_penalty"], p["battery_discharge_penalty"],
            p["electrolyzer_cost_AE"], p["electrolyzer_cost_PEM"],
            p["electrolyzer_eff_AE"], p["electrolyzer_eff_PEM"],
            p["methanol_synthesis_cost"], p["hydrogen_storage_cost"],
            p["hydrogen_storage_elec"], p["hydrogen_charge_penalty"],
            p["hydrogen_storage_efficiency"], p["methanol_storage_cost_param"],
            p["methanol_charge_penalty"], p["discount_rate"],
            p["battery_lifetime"], p["wind_lifetime"], p["pv_lifetime"],
            p["electrolyzer_lifetime_AE"], p["electrolyzer_lifetime_PEM"],
            p["methanol_synthesis_lifetime"], p["hydrogen_storage_lifetime"],
            p["methanol_storage_lifetime"], p["heat_pump_cost"],
            p["heat_pump_cop"], p["heat_pump_lifetime"],
            p["thermal_storage_cost"], p["thermal_storage_efficiency"],
            p["thermal_storage_lifetime"], p["dac_cost"],
            p["dac_elec_per_kg"], p["dac_heat_per_kg"], p["dac_lifetime"],p["sorbent_cost"],
            p["co2_compressor_cost"], p["co2_compression_energy"],
            p["co2_compressor_lifetime"], p["co2_storage_cost"],
            p["co2_storage_lifetime"], p["Flex_up"], p["Flex_mid"],
            p["Flex_down"], p["surplus_penalty"], p["aggregation_mode"]
        )
    if prod_key == "ftl":
        return optimize_ftl_system(
            wind_output, pv_output, p["target_ftl_production"],
            wind_cost, pv_cost,
            p["battery_energy_cost"], p["battery_storage_cost"],
            p["battery_charge_efficiency"], p["battery_discharge_efficiency"],
            p["battery_charge_penalty"], p["battery_discharge_penalty"],
            p["electrolyzer_cost_AE"], p["electrolyzer_cost_PEM"],
            p["electrolyzer_eff_AE"], p["electrolyzer_eff_PEM"],
            p["electrolyzer_lifetime_AE"], p["electrolyzer_lifetime_PEM"],
            p["ftl_synthesis_cost"], p["ftl_synthesis_lifetime"],
            p["ftl_hydrogen_ratio"], p["ftl_co2_ratio"], p["ftl_electricity_ratio"],
            p["hydrogen_storage_cost"], p["hydrogen_storage_elec"],
            p["hydrogen_charge_penalty"], p["hydrogen_storage_efficiency"],
            p["hydrogen_storage_lifetime"], p["ftl_storage_cost_param"],
            p["ftl_storage_lifetime"], p["ftl_charge_penalty"],
            p["heat_pump_cost"], p["heat_pump_cop"], p["heat_pump_lifetime"],
            p["thermal_storage_cost"], p["thermal_storage_efficiency"], p["thermal_storage_lifetime"],
            p["dac_cost"], p["dac_elec_per_kg"], p["dac_heat_per_kg"], p["dac_lifetime"],
            p["sorbent_cost"], 
            p["co2_compressor_cost"], p["co2_compression_energy"], p["co2_compressor_lifetime"],
            p["co2_storage_cost"], p["co2_storage_lifetime"],
            p["discount_rate"], p["battery_lifetime"], p["wind_lifetime"], p["pv_lifetime"],
            p["Flex_up"], p["Flex_mid"], p["Flex_down"], p["surplus_penalty"], p["aggregation_mode"]
        )
    if prod_key == "ammonia":
        return optimize_ammonia_system(
            wind_output, pv_output, p["target_ammonia_production"],
            wind_cost, pv_cost,
            p["battery_energy_cost"], p["battery_storage_cost"],
            p["battery_charge_efficiency"], p["battery_discharge_efficiency"],
            p["battery_charge_penalty"], p["battery_discharge_penalty"],
            p["electrolyzer_cost_AE"], p["electrolyzer_cost_PEM"],
            p["electrolyzer_eff_AE"], p["electrolyzer_eff_PEM"],
            p["ammonia_synthesis_cost"],
            p["hydrogen_storage_cost"], p["hydrogen_storage_elec"],
            p["hydrogen_charge_penalty"], p["hydrogen_storage_efficiency"],
            p["ammonia_storage_cost_param"], p["discount_rate"],
            p["battery_lifetime"], p["wind_lifetime"], p["pv_lifetime"],
            p["electrolyzer_lifetime_AE"], p["electrolyzer_lifetime_PEM"],
            p["ammonia_synthesis_lifetime"], p["hydrogen_storage_lifetime"],
            p["ammonia_storage_lifetime"],
            p["Flex_up"], p["Flex_mid"], p["Flex_down"],
            p["surplus_penalty"], p["aggregation_mode"]
        )
    raise ValueError(f"Unsupported product: {prod_key}")


# ================================
# 新增：气候年对比（2021/2022/2023）
# ================================
def _climate_year_filepaths(scen_raw: str) -> dict:
    """返回该 scen 的三年文件路径字典：{2021: path or None, 2022: ..., 2023: ...}"""
    fp = {}
    base = f"median_points_cf_timeseries_{scen_raw}"
    candidates = {
        2021: os.path.join(cf_dir, f"{base}_2021.csv"),
        2022: os.path.join(cf_dir, f"{base}_2022.csv"),
        2023: os.path.join(cf_dir, f"{base}.csv"),
    }
    for y, p in candidates.items():
        fp[y] = p if os.path.exists(p) else None
    return fp

def _extract_capacities(model: gp.Model, prod_key: str) -> dict:
    """从模型取关键容量；不在则默认为 0。"""
    safe = lambda nm: (model.getVarByName(nm).x if model.getVarByName(nm) is not None else 0.0)
    out = {
        "Cap_wind_kW": safe("wind_capacity"),
        "Cap_pv_kW": safe("pv_capacity"),
        "Cap_batt_E_kWh": safe("battery_storage_capacity") or safe("battery_storage_capacity_kWh"),
        "Cap_batt_P_kW": safe("battery_energy_capacity") or safe("battery_energy_capacity_kW"),
        "Cap_AE_kW": safe("electrolyzer_capacity_AE") or safe("electrolyzer_capacity_AE_kW"),
        "Cap_PEM_kW": safe("electrolyzer_capacity_PEM") or safe("electrolyzer_capacity_PEM_kW"),
        "Cap_H2_store": safe("hydrogen_storage_capacity") or safe("hydrogen_storage_capacity_kgH2"),
        "Cap_DAC_kgCO2h": safe("dac_capacity") or safe("dac_capacity_kgCO2h"),
        "Cap_CO2_store": safe("co2_storage_capacity") or safe("co2_storage_capacity_kgCO2"),
        "Cap_CO2_comp": safe("co2_compressor_capacity") or safe("co2_compressor_capacity_kgCO2h"),
        "Cap_thermal_store": safe("thermal_storage_capacity") or safe("thermal_storage_capacity_kWh"),
        "Cap_heat_pump_kW": safe("heat_pump_capacity") or safe("heat_pump_capacity_kW"),
    }

    if prod_key == "ammonia":
        out["Cap_syn"] = safe("ammonia_synthesis_capacity")
        out["Cap_stor"] = safe("ammonia_storage_capacity")
    elif prod_key == "methanol":
        out["Cap_syn"] = safe("methanol_synthesis_capacity")
        out["Cap_stor"] = safe("methanol_storage_capacity")
    else:
        out["Cap_syn"] = safe("FTL_capacity")
        out["Cap_stor"] = safe("FTL_storage_capacity")
    return out

def _product_target_from_params(prod_key: str, p: dict) -> float:
    if prod_key == "ammonia":
        return float(p.get("target_ammonia_production", 0.0))
    if prod_key == "methanol":
        return float(p.get("target_methanol_production", 0.0))
    if prod_key == "ftl":
        return float(p.get("target_ftl_production", 0.0))
    return 0.0

def _build_one_model_from_group(group: pd.DataFrame,
                                prod_key: str,
                                wind_cost: float,
                                pv_cost: float,
                                p: dict) -> tuple[gp.Model, int]:
    """按你现有接口创建并返回模型与时序长度 T；异常外抛由上层处理"""
    group = group.sort_values('hour')
    wind_output = group['CF_wind_avg'].to_numpy()
    pv_output   = group['CF_pv_avg'].to_numpy()
    T = len(wind_output)
    model = create_model_for_product(prod_key, wind_output, pv_output, wind_cost, pv_cost, p)
    return model, T

def compare_climate_years_for_scenario(scen_raw: str,
                                       only_one_node: Optional[Dict[str, str]] = None,
                                       stop_after_first: bool = False):
    """
    对给定 scen_raw，读取 _2021/_2022/默认(2023) 三年，并对“共同的分组键”逐年求解，输出 LCO 和容量对比。
    输出：
      - 每个节点一份 CSV：<YearTag>/<ScenarioFolder>/climate_compare/<prod>_<continent>_cluster<k>_compare.csv
      - 可选两张图：LCO 折线 + 容量占比条形图
    """
    fps = _climate_year_filepaths(scen_raw)
    year_dfs = {}
    for y, pth in fps.items():
        if pth is None:
            print(f"   · {scen_raw}: 缺少 {y} 年文件，已跳过该年。", flush=True)
            continue
        df_y = pd.read_csv(pth)
        if 'scenario' not in df_y.columns:
            df_y['scenario'] = normalize_scenario(scen_raw)
        if 'product' not in df_y.columns:
            df_y['product'] = 'methanol'
        year_dfs[y] = df_y

    if not year_dfs:
        print(f"   · {scen_raw}: 三年均缺失，跳过气候年对比。", flush=True)
        return

    group_cols = [c for c in ['scenario','product','continent','cluster','latitude','longitude','region'] if all(c in v.columns for v in year_dfs.values())]
    common_keys = None
    for y, df_y in year_dfs.items():
        keys_y = set(tuple(vals) for vals in df_y[group_cols].itertuples(index=False, name=None))
        common_keys = keys_y if common_keys is None else (common_keys & keys_y)
    if not common_keys:
        print(f"   · {scen_raw}: 三年未找到公共节点，跳过。", flush=True)
        return

    year_tag  = pick_year_tag_from_filename(scen_raw)  # 2030/2040/2050/2035（影响 capex 选择）
    wind_map, pv_map = pick_cost_maps(year_tag)
    params = select_params_by_year(year_tag)
    scen_norm   = normalize_scenario(scen_raw)
    scen_type   = classify_scenario_for_cost(scen_raw)
    scen_folder = scenario_output_folder(scen_raw)

    scen_out_root = os.path.join(out_root, f"Year_{year_tag}", scen_folder, "climate_compare")
    os.makedirs(scen_out_root, exist_ok=True)

    for tup in common_keys:
        key_map = dict(zip(group_cols, tup))
        if only_one_node and not (all(k in key_map for k in only_one_node) and all(str(key_map[k])==str(v) for k,v in only_one_node.items())):
            continue

        prod_key = (key_map.get('product') or 'methanol').lower()
        cont = key_map.get('continent', 'Unknown')
        cl   = key_map.get('cluster', 'X')
        region_code = int(key_map.get('region')) if key_map.get('region') is not None else None

        print(f"\n   ▶ 气候对比 | {scen_norm} | {prod_key.upper()} | {cont} | 簇={cl}", flush=True)

        # 年度结果容器
        recs = []

        for Y in [2021, 2022, 2023]:
            if Y not in year_dfs:
                continue
            df_y = year_dfs[Y]
            # 取该节点对应行
            cond = np.ones(len(df_y), dtype=bool)
            for col, val in key_map.items():
                cond &= (df_y[col] == val)
            group = df_y.loc[cond]
            if group.empty:
                print(f"     · {Y}: 无该节点记录，跳过。", flush=True)
                continue

            wind_cost_i = wind_map.get(region_code, 800)
            if scen_type == 'WIND':
                pv_cost_i = 225
            else:
                pv_cost_i = pv_map.get(region_code, 350)

            try:
                model, T = _build_one_model_from_group(group, prod_key, wind_cost_i, pv_cost_i, params)
                model.optimize()
                if model.status != GRB.OPTIMAL:
                    print(f"     · {Y}: 非最优（{model.status}），跳过。", flush=True)
                    continue
            except Exception as e:
                print(f"     · {Y}: 构建/求解失败：{e}", flush=True)
                continue

            annual_cost = float(model.ObjVal)
            tgt = _product_target_from_params(prod_key, params)
            lco = (annual_cost / tgt) if tgt and tgt > 0 else np.nan

            caps = _extract_capacities(model, prod_key)
            rec = {"year": Y, "Annual_cost_$": annual_cost, f"LCO_{prod_key}_$/t": lco}
            rec.update(caps)
            recs.append(rec)

        if not recs:
            print("     · 三年均失败/缺失，跳过该节点。", flush=True)
            continue

        df_res = pd.DataFrame(recs).set_index("year").sort_index()
        if 2023 in df_res.index:
            delta = df_res - df_res.loc[2023]
        else:
            delta = pd.DataFrame()

        node_dir = os.path.join(scen_out_root, f"{prod_key}_{cont}_cluster{cl}")
        os.makedirs(node_dir, exist_ok=True)
        out_csv = os.path.join(node_dir, "climate_compare_summary.csv")
        df_res.to_csv(out_csv)
        if not delta.empty:
            delta.to_csv(os.path.join(node_dir, "climate_compare_delta_vs_2023.csv"))
        print(f"       ✓ 已保存：{out_csv}", flush=True)

        # 简易图：LCO 折线 + 容量占比（按年归一化）
        try:
            lco_col = f"LCO_{prod_key}_$/t"
            if lco_col in df_res.columns and df_res[lco_col].notna().any():
                plt.figure(figsize=(6.4, 3.6))
                df_res[lco_col].plot(marker="o")
                plt.title(f"{scen_norm} | {prod_key.upper()} | {cont} 簇{cl} — LCO 对比(气候年)")
                plt.ylabel("$/t"); plt.xlabel("气候年份")
                plt.grid(True, alpha=0.35)
                plt.tight_layout()
                plt.savefig(os.path.join(node_dir, "climate_compare_LCO.png"), dpi=300)
                plt.close()

            cap_cols = [c for c in df_res.columns if c.startswith("Cap_")]
            if cap_cols:
                share = df_res[cap_cols].div(df_res[cap_cols].sum(axis=1), axis=0)
                ax = share.plot(kind="bar", figsize=(8.5, 4.2))
                plt.title(f"{scen_norm} | {prod_key.UPPER()} | {cont} 簇{cl} — 装机占比(气候年)")
                plt.ylabel("Share"); plt.xlabel("气候年份")
                plt.grid(True, axis="y", alpha=0.35)
                plt.legend(ncols=2, fontsize=8, frameon=True)
                plt.tight_layout()
                plt.savefig(os.path.join(node_dir, "climate_compare_capacity_share.png"), dpi=300)
                plt.close()
        except Exception:
            pass

        if stop_after_first:
            print("   · 单节点测试完成（climate compare），提前结束。", flush=True)
            break


# ========== 主流程（原始：逐节点输出典型周 & 年度热力图） ==========
total_groups = solved_groups = skipped_groups = 0
early_stop = False
pat = re.compile(r"median_points_cf_timeseries_(?P<scenario>.+)\.csv$", re.IGNORECASE)

for fi, cf_path in enumerate(cf_files, 1):
    m = pat.search(os.path.basename(cf_path))
    if not m:
        print(f"⚠ 无法解析情景名：{cf_path}，已跳过。", flush=True); continue

    scen_raw = m.group('scenario')
    if ONLY_ONE_SCENARIO and (re.search(ONLY_ONE_SCENARIO, scen_raw, flags=re.IGNORECASE) is None):
        print(f"· 跳过非目标情景：{scen_raw}", flush=True); continue

    # 规划年与气候年
    year_tag  = pick_year_tag_from_filename(scen_raw)   # '2030'/'2040'/'2050'/'2035'
    climate_year = pick_climate_year_from_path(cf_path) # 2021/2022/2023

    # 先规范场景、分类，再得到输出文件夹名（这三行要放在用到 scen_folder 之前）
    scen_norm   = normalize_scenario(scen_raw)
    scen_type   = classify_scenario_for_cost(scen_raw)
    scen_folder = scenario_output_folder(scen_raw)

    # —— 正确的目录层级：规划年 / 场景 / 气候年 ——（只建一次，后面不要再覆盖 scen_out）
    scen_out_root = os.path.join(out_root, f"Year_{year_tag}", scen_folder)
    os.makedirs(scen_out_root, exist_ok=True)
    scen_out = os.path.join(scen_out_root, f"Climate_{climate_year}")
    os.makedirs(scen_out, exist_ok=True)

    # 成本与参数
    wind_map, pv_map = pick_cost_maps(year_tag)
    params = select_params_by_year(year_tag)

    print(f"\n================ 处理情景 [{fi}/{len(cf_files)}]: {scen_raw} → 标题[{scen_norm}] | 输出[{scen_folder}] | 年份: {year_tag} | 气候年: {climate_year} | 类型: {scen_type} ================", flush=True)


    cf_df = pd.read_csv(cf_path)
    print(f"   - 读取行数：{len(cf_df)}", flush=True)

    if 'scenario' not in cf_df.columns: cf_df['scenario'] = scen_norm
    if 'product'  not in cf_df.columns: cf_df['product']  = 'methanol'

    group_cols = [c for c in ['scenario','product','continent','cluster','latitude','longitude','region'] if c in cf_df.columns]
    print(f"   - 分组键：{group_cols}", flush=True)

    groups_list = list(cf_df.groupby(group_cols, sort=False))
    print(f"   - 分组数量：{len(groups_list)}", flush=True)

    for gi, (keys, group) in enumerate(groups_list, 1):
        key_map = dict(zip(group_cols, keys if isinstance(keys, tuple) else (keys,)))
        if ONLY_ONE_NODE and (not (
            all(k in key_map for k in ONLY_ONE_NODE) and all(str(key_map[k])==str(v) for k,v in ONLY_ONE_NODE.items())
        )):
            continue

        total_groups += 1
        cont = key_map.get('continent', 'Unknown')
        cl   = key_map.get('cluster', 'X')
        prod_key = (key_map.get('product') or 'methanol').lower()
        prod = prod_key.upper()
        lat  = key_map.get('latitude'); lon  = key_map.get('longitude')
        region_code = int(key_map.get('region')) if key_map.get('region') is not None else None

        try:
            lat_f = float(lat) if lat is not None else float('nan')
            lon_f = float(lon) if lon is not None else float('nan')
        except Exception:
            lat_f, lon_f = float('nan'), float('nan')

        print(f"\n   ▶ 组 {gi}/{len(groups_list)} | {prod} | {cont} | 簇={cl} | "
              f"lat={lat_f:.2f} lon={lon_f:.2f} | region_code={region_code}", flush=True)

        group = group.sort_values('hour')
        wind_output = group['CF_wind_avg'].to_numpy()
        pv_output   = group['CF_pv_avg'].to_numpy()
        T = len(wind_output)
        print(f"     · 时序长度={T}", flush=True)

        wind_cost_i = wind_map.get(region_code, 800)
        if scen_type == 'WIND':
            pv_cost_i = 225; pv_cost_note = "（统一225）"
        else:
            pv_cost_i = pv_map.get(region_code, 350); pv_cost_note = f"（{year_tag if year_tag!='2035' else '2035'}ATB 区域映射）"
        print(f"     · 成本：风={wind_cost_i:.1f} | 光={pv_cost_i:.1f}{pv_cost_note}", flush=True)
        print(f"     · 参数集：params_{year_tag if year_tag in ['2030','2050'] else 'base'}", flush=True)

        # 构建与求解
        try:
            print("     · 构建模型...", flush=True)
            model = create_model_for_product(prod_key, wind_output, pv_output, wind_cost_i, pv_cost_i, params)
        except NameError as e:
            print(f"     ✗ 构建失败（缺少优化函数）：{e}", flush=True); skipped_groups += 1; continue
        except NotImplementedError as e:
            print(f"     ✗ 构建失败（未实现）：{e}", flush=True); skipped_groups += 1; continue
        except Exception as e:
            print(f"     ✗ 构建失败（异常）：{e}", flush=True); skipped_groups += 1; continue

        print("     · 开始求解...", flush=True)
        model.optimize()
        if model.status != GRB.OPTIMAL:
            print(f"     ✗ 非最优解，状态码={model.status}，跳过。", flush=True)
            skipped_groups += 1; continue

        solved_groups += 1
        wind_capacity = model.getVarByName('wind_capacity').x if model.getVarByName('wind_capacity') else 0.0
        pv_capacity   = model.getVarByName('pv_capacity').x if model.getVarByName('pv_capacity') else 0.0
        print(f"     ✓ 求解成功：风机容量={wind_capacity:.2f} kW, 光伏容量={pv_capacity:.2f} kW", flush=True)

        # ========= 全年序列 =========
        series_wind = wind_output * wind_capacity
        series_pv   = pv_output   * pv_capacity

        series_ae    = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["electrolyzer_AE"], T), T)
        series_pem   = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["electrolyzer_PEM"], T), T)
        series_syn   = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["synthesis_power"], T), T)
        series_dac   = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["dac_elec"], T), T)
        series_hpout = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["heat_pump_output"], T), T)

        series_b_ch   = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["battery_charge"], T), T)
        series_b_dis  = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["battery_discharge"], T), T)
        series_h2_ch  = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["hydrogen_charge"], T), T)
        series_h2_dis = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["hydrogen_discharge"], T), T)
        series_prod_ch  = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["product_charge"], T), T)
        series_prod_dis = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["product_discharge"], T), T)

        series_b_balance   = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["battery_balance"], T), T)
        series_h2_balance  = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["hydrogen_storage_balance"], T), T)
        series_p_balance   = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["product_storage_balance"], T), T)

        bat_cap = get_first_var_value(model, VAR_CAPACITY["battery"])
        h2_cap  = get_first_var_value(model, VAR_CAPACITY["hydrogen"])
        prod_cap= get_first_var_value(model, VAR_CAPACITY["product"])

        series_b_soc   = series_b_balance / (bat_cap  if bat_cap  else max(np.nanmax(series_b_balance),1.0))
        series_h2_soc  = series_h2_balance/ (h2_cap   if h2_cap   else max(np.nanmax(series_h2_balance),1.0))
        series_prod_soc= series_p_balance / (prod_cap if prod_cap else max(np.nanmax(series_p_balance),1.0))

        syn_elec_label = '甲醇合成' if prod_key == 'methanol' else ('FTL合成' if prod_key == 'ftl' else '合成')

        # ========= （新增）将序列调整到“当地时区”再绘图 =========
        if ADJUST_TO_LOCAL_TZ and np.isfinite(lat_f) and np.isfinite(lon_f):
            year_for_tz = int(year_tag) if str(year_tag).isdigit() else 2040
            offset_hours = infer_utc_offset_hours(lat_f, lon_f, year_for_tz)
            print(f"     · 本地时区偏移：UTC{offset_hours:+d}（基于坐标与{year_for_tz}年）", flush=True)

            series_map = {
                "series_wind": series_wind,
                "series_pv":   series_pv,
                "series_ae":   series_ae,
                "series_pem":  series_pem,
                "series_syn":  series_syn,
                "series_dac":  series_dac,
                "series_hpout": series_hpout,
                "series_b_ch": series_b_ch,
                "series_b_dis": series_b_dis,
                "series_h2_ch": series_h2_ch,
                "series_h2_dis": series_h2_dis,
                "series_prod_ch": series_prod_ch,
                "series_prod_dis": series_prod_dis,
                "series_b_balance": series_b_balance,
                "series_h2_balance": series_h2_balance,
                "series_p_balance": series_p_balance,
                "series_b_soc": series_b_soc,
                "series_h2_soc": series_h2_soc,
                "series_prod_soc": series_prod_soc,
            }
            roll_series_dict(series_map, offset_hours, list(series_map.keys()))
            (series_wind, series_pv, series_ae, series_pem, series_syn, series_dac, series_hpout,
             series_b_ch, series_b_dis, series_h2_ch, series_h2_dis, series_prod_ch, series_prod_dis,
             series_b_balance, series_h2_balance, series_p_balance,
             series_b_soc, series_h2_soc, series_prod_soc) = (
                series_map["series_wind"], series_map["series_pv"], series_map["series_ae"],
                series_map["series_pem"], series_map["series_syn"], series_map["series_dac"], series_map["series_hpout"],
                series_map["series_b_ch"], series_map["series_b_dis"], series_map["series_h2_ch"], series_map["series_h2_dis"],
                series_map["series_prod_ch"], series_map["series_prod_dis"],
                series_map["series_b_balance"], series_map["series_h2_balance"], series_map["series_p_balance"],
                series_map["series_b_soc"], series_map["series_h2_soc"], series_map["series_prod_soc"]
            )
        else:
            print("     · 未进行时区调整（坐标缺失或关闭 ADJUST_TO_LOCAL_TZ）。", flush=True)

        # ========= 年度热力图（Battery/H2/Product 用 SOC）=========
        title_prefix = f"{scen_norm} | {prod} {cont} 簇 {cl} ({lat_f:.2f},{lon_f:.2f})"
        if   prod_key == 'methanol': prod_soc_label = '甲醇储能'
        elif prod_key == 'ftl':      prod_soc_label = 'FTL储能'
        elif prod_key == 'ammonia':  prod_soc_label = '氨储能'
        else:                        prod_soc_label = '产品储能'

        annual_items = [
            ("风电",          series_wind),
            ("光伏",          series_pv),
            ("电解槽",        series_ae + series_pem),
            (syn_elec_label,  series_syn),
            ("DAC",           series_dac),
            ("电池",          series_b_soc),
            (f"{H2} 储能",     series_h2_soc),
            (prod_soc_label,  series_prod_soc),
        ]
        out_heat = os.path.join(scen_out, f"{prod_key}_{cont}_cluster{cl}_annual_heatmap_CN_CY{climate_year}.png")
        ok = plot_annual_heatmap_panel(annual_items, title_prefix, out_heat, cmap="YlGnBu_r")
        if ok:
            print(f"       ✓ 保存年度热力图：{out_heat}", flush=True)
        else:
            print("       · 年度热力图：无有效序列，已跳过。", flush=True)

        # 影子价格Y轴范围（示例固定）
        def get_ylim(year_tag, price_type, prod_key):
            if price_type == 'power':
                return (-0.05, 0.40)
            return (-0.5, 4.0)

        # ========= 典型周供需堆叠图（电力 + 电力影子价格双Y轴）=========
        syn_elec_ratio = (
            params.get('methanol_electricity_ratio', 0.1) if prod_key == 'methanol' 
            else params.get('ammonia_electricity_ratio', 0.303) if prod_key == 'ammonia'
            else params.get('ftl_electricity_ratio', 0.1)
        )
        heat_pump_cop = params["heat_pump_cop"]

        for season, week_idx in season_weeks.items():
            week_start = 168 * week_idx
            week_end   = min(week_start + 168, T)
            if week_start >= T:
                print(f"     · {season}: 超出时序范围，跳过。", flush=True)
                continue
            x = np.arange(week_end - week_start)
            print(f"     · 绘图 {season}: 小时 {week_start}–{week_end-1}", flush=True)

            supply_wind = series_wind[week_start:week_end]
            supply_pv   = series_pv[week_start:week_end]
            supply_batt = series_b_dis[week_start:week_end]

            demand_ae   = series_ae[week_start:week_end]
            demand_pem  = series_pem[week_start:week_end]
            demand_syn_elec = series_syn[week_start:week_end] * syn_elec_ratio
            demand_bat_ch   = series_b_ch[week_start:week_end]
            demand_h2       = np.zeros_like(demand_ae)
            demand_hp       = (series_hpout[week_start:week_end] / max(heat_pump_cop,1e-6))
            demand_dac      = series_dac[week_start:week_end]
            demand_surplus  = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["surplus"], T), T)[week_start:week_end]
            demand_co2      = ensure_series(get_series_by_templates(model, VAR_TEMPLATES["co2_comp"], T), T)[week_start:week_end]

            fig, ax1 = plt.subplots(figsize=(15, 8))
            ax1.bar(x, supply_wind, label='风电', color=wind_color, edgecolor='black', linewidth=0.3)
            ax1.bar(x, supply_pv, bottom=supply_wind, label='光伏', color=pv_color, edgecolor='black', linewidth=0.3)
            ax1.bar(x, supply_batt, bottom=supply_wind + supply_pv,
                    label='电池放电', color=battery_color, edgecolor='black', linewidth=0.3)

            b = -(demand_ae)
            ax1.bar(x, b, label='AE', color=ae_color, edgecolor='black', linewidth=0.3)
            ax1.bar(x, -demand_pem, bottom=b, label='PEM', color=pem_color, edgecolor='black', linewidth=0.3)
            b2 = -(demand_ae + demand_pem)
            ax1.bar(x, -demand_syn_elec, bottom=b2, label='合成用电', color=synth_color, edgecolor='black', linewidth=0.3)
            b3 = -(demand_ae + demand_pem + demand_syn_elec)
            ax1.bar(x, -demand_bat_ch, bottom=b3, label='电池充电', color=bat_charge_color, edgecolor='black', linewidth=0.3)
            b5 = -(demand_ae + demand_pem + demand_syn_elec + demand_bat_ch + demand_h2)
            ax1.bar(x, -demand_hp, bottom=b5, label='热泵', color=heatpump_color, edgecolor='black', linewidth=0.3)
            b6 = -(demand_ae + demand_pem + demand_syn_elec + demand_bat_ch + demand_h2 + demand_hp)
            ax1.bar(x, -demand_dac, bottom=b6, label='DAC用电', color=dac_color, edgecolor='black', linewidth=0.3)
            b7 = -(demand_ae + demand_pem + demand_syn_elec + demand_bat_ch + demand_h2 + demand_hp + demand_dac)
            ax1.bar(x, -demand_surplus, bottom=b7, label='弃电', color=surplus_color, edgecolor='black', linewidth=0.3)

            ax1.set_xlim(0, len(x))
            ax1.axhline(0, color='k', linewidth=0.8)
            ax1.set_xlabel('周内小时', fontsize=26)
            ax1.set_ylabel('电力（kW）', fontsize=26, color='black')
            ax1.tick_params(axis='both', labelsize=26)
            ax1.tick_params(axis='y', labelcolor='black')

            pos_sum = supply_wind + supply_pv + supply_batt
            neg_sum = (demand_ae + demand_pem + demand_syn_elec + demand_bat_ch +
                    demand_h2 + demand_hp + demand_dac + demand_surplus + demand_co2)
            set_y_margin(ax1, pos_sum, neg_sum, Y_MARGIN_FRAC_POWER)

            ax2 = ax1.twinx()
            try:
                elec_shadow = np.array([model.getConstrByName(f"power_balance_{t}").Pi 
                                    for t in range(week_start, week_end)])
                ax2.plot(x, elec_shadow, color="#000000", linewidth=2.5, 
                        label='电力影子价格', linestyle='--', alpha=0.85)
                ylim = get_ylim(year_tag, 'power', prod_key)
                ax2.set_ylim(ylim)
                ax2.set_ylabel('电力影子价格（$/kWh）', fontsize=26, color="#000000")
                ax2.tick_params(axis='y', labelcolor="#000000", labelsize=26)
                ax2.spines['right'].set_color("#000000")
                ax2.spines['right'].set_linewidth(0.8)
            except Exception as e:
                print(f"       · 电力影子价格提取失败：{e}", flush=True)
                ax2.set_ylabel(''); ax2.set_yticks([])

            ax1.set_title(f'{title_prefix} — {season} 典型周（电力）', fontsize=5)
            plt.tight_layout()
            out_path = os.path.join(scen_out, f'{prod_key}_{cont}_cluster{cl}_{season}_power_CN_CY{climate_year}.png')
            plt.savefig(out_path, dpi=300, bbox_inches='tight')
            plt.close(fig)
            print(f"       ✓ 保存：{out_path}", flush=True)

            # ===== 氢气堆叠图 + 氢气影子价格（双Y轴）=====
            supply_h2_prod = (series_ae * params.get('electrolyzer_eff_AE', 0.65) + series_pem * params.get('electrolyzer_eff_PEM', 0.70)) * 3.6 / 120.0
            supply_h2_dis  = series_h2_dis
            demand_syn_h2 = series_syn * (
                params.get('methanol_hydrogen_ratio', 0.19) if prod_key == 'methanol'
                else params.get('ammonia_hydrogen_ratio', 0.18) if prod_key == 'ammonia'
                else params.get('ftl_hydrogen_ratio', 0.49)
            )
            demand_h2_chg  = series_h2_ch

            sh = slice(week_start, week_end)
            xH = np.arange(week_end - week_start)
            
            fig2, ax3 = plt.subplots(figsize=(15, 6))
            ax3.bar(xH, supply_h2_prod[sh], label=f'{H2} 产量', color='#4F6BED', edgecolor='black', linewidth=0.3)
            ax3.bar(xH, supply_h2_dis[sh], bottom=supply_h2_prod[sh], label=f'{H2} 放电', color='#7DA2F8', edgecolor='black', linewidth=0.3)
            ax3.bar(xH, -demand_syn_h2[sh], label=f'{H2} 合成需求', color='#CC79A7', edgecolor='black', linewidth=0.3)
            ax3.bar(xH, -demand_h2_chg[sh], bottom=-demand_syn_h2[sh], label=f'{H2} 充电', color='#D94A38', edgecolor='black', linewidth=0.3)
            
            ax3.axhline(0, color='k', linewidth=0.8)
            ax3.set_xlim(0, len(xH))
            ax3.set_xticks(np.arange(0, len(xH)+1, 24))
            ax3.set_xlabel('周内小时', fontsize=26)
            ax3.set_ylabel('氢（kg/h）', fontsize=26, color='black')
            ax3.tick_params(axis='both', labelsize=26)
            ax3.tick_params(axis='y', labelcolor='black')

            set_y_margin(ax3, supply_h2_prod[sh] + supply_h2_dis[sh], demand_syn_h2[sh] + demand_h2_chg[sh], Y_MARGIN_FRAC_H2)

            ax4 = ax3.twinx()
            try:
                h2_shadow = np.array([model.getConstrByName(f"hydrogen_PD_balance_{t}").Pi 
                                    for t in range(week_start, week_end)])
                ax4.plot(xH, h2_shadow, color="#000000", linewidth=2.5,
                        label=f"{H2}影子价格", linestyle='-.', alpha=0.85)
                ylim = get_ylim(year_tag, 'hydrogen', prod_key)
                ax4.set_ylim(ylim)
                ax4.set_ylabel('氢影子价格（$/kg）', fontsize=26, color="#000000")
                ax4.tick_params(axis='y', labelcolor="#000000", labelsize=26)
                ax4.spines['right'].set_color("#000000")
                ax4.spines['right'].set_linewidth(0.8)
            except Exception as e:
                print(f"       · 氢气影子价格提取失败：{e}", flush=True)
                ax4.set_ylabel(''); ax4.set_yticks([])

            ax3.set_title(f'{title_prefix} — {season} 典型周（{H2}）', fontsize=5)
            plt.tight_layout()
            out_path = os.path.join(scen_out, f'{prod_key}_{cont}_cluster{cl}_{season}_h2_CN_CY{climate_year}.png')
            plt.savefig(out_path, dpi=300, bbox_inches='tight')
            plt.close(fig2)
            print(f"       ✓ 保存：{out_path}", flush=True)

        if ONLY_ONE_NODE and STOP_AFTER_FIRST_NODE:
            early_stop = True
            print("\n🛑 单节点测试完成，提前结束。", flush=True)
            break

    # ========= 附加：对当前 scen 做“气候年对比” =========
    try:
        compare_climate_years_for_scenario(
            scen_raw,
            only_one_node=ONLY_ONE_NODE,
            stop_after_first=STOP_AFTER_FIRST_NODE
        )
    except Exception as e:
        print(f"⚠ 气候年对比执行失败（{scen_raw}）：{e}", flush=True)

    if early_stop: break

# 总结
print("\n============== 总结 ==============")
print(f"总分组数：{total_groups}")
print(f"最优求解：{solved_groups}")
print(f"跳过分组：{skipped_groups}")
print("✅ 全部完成。")


# 气候数据比较

In [ ]:
# -*- coding: utf-8 -*-
"""
CAPEX/参数敏感性分析（参数直覆版，不含气候年对比）
----------------------------------------------------------------
对同一套时序数据（单表/单年），在多个“参数直覆画像”（profiles）下重复求解：
- 画像以“params直接覆盖”的形式提供：支持点号路径覆盖（如 capex.ammonia_storage_cost_param）
- 可直接覆盖区域输入成本键：wind_cost_input、pv_cost_input（若未提供则仍来自区域成本图）
- 输出：每个节点一个 CSV（索引 capex_profile），包含 LCO*、年度投资贡献、容量、以及**所有覆盖后的变量值**

计算口径与之前一致：
- LCOE_$/MWh（风+光+电池，年金化CAPEX+OPEX（opex_frac）+FOM）
- LCOH_$/kg（LCOE + 电解槽 + 氢储）
- LCOHeat_$/MWh_th（LCOE + 热泵 + 储热）
- LCOC_$/tCO2（LCOE/LCOHeat + DAC + CO2压缩/运输/封存）
- 合成装置/合成产品储存年度投资贡献：Synth_annual_cost_$py / Synth_store_annual_cost_$py
- 容量列统一：Cap_syn / Cap_stor
- 储存CAPEX参数统一键：ammonia_storage_cost_param / methanol_storage_cost_param / ftl_storage_cost_param

依赖（工程中已有）：
- pick_year_tag_from_filename, pick_cost_maps, select_params_by_year,
- normalize_scenario, classify_scenario_for_cost, scenario_output_folder,
- _build_one_model_from_group, _product_target_from_params, _extract_capacities
"""

import os, re, glob, argparse, copy, json
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Any
from gurobipy import GRB

# ============== 可按需修改 ==============
YEAR_FILTER = "2035"             # 仍用于选择参数/成本图（但不再做气候年对比）
ONLY_ONE_SCENARIO = None
# Force all products to use the same hourly wind/PV node table for sensitivity reruns.
FORCED_CF_PATTERN = os.path.join(
    r"D:\OneDrive\POTSED", "00 Figures", "Fig3.9_3.20_*", "Input",
    "median_points_cf_timeseries_RegionalRE_LowAE.csv"
)
FORCED_CF_MATCHES = sorted(glob.glob(FORCED_CF_PATTERN))
FORCED_CF_TIMESERIES = FORCED_CF_MATCHES[0] if FORCED_CF_MATCHES else None
ONLY_ONE_NODE = None             # 例: {'product':'ammonia','continent':'Europe','cluster':0}
STOP_AFTER_FIRST_NODE = False
PRINT_DEBUG_SYNTH = False        # 合成贡献调试打印


# ============== 常用小工具 ==============
def _crf(wacc: float, lifetime: int) -> float:
    r, n = float(wacc), int(lifetime)
    return r * (1 + r) ** n / ((1 + r) ** n - 1) if n > 0 else np.inf

def _get_param(dct, *keys, default=None):
    cur = dct
    try:
        for k in keys:
            cur = cur[k]
        return cur
    except Exception:
        return default

def _opex_frac(params, default=0.03) -> float:
    return float(_get_param(params, 'econ', 'opex_frac', default=default))

def _fom_frac(params, block: str) -> float:
    return float(_get_param(params, 'fom_frac', block, default=0.0) or 0.0)

def _annual_unit(capex_unit: float, lifetime: int, wacc: float, opex_f: float, fom_f: float=0.0) -> float:
    """单位CAPEX → 年单位成本"""
    return float(capex_unit) * (_crf(wacc, lifetime) + float(opex_f) + float(fom_f))

def _pick_series(df: pd.DataFrame, candidates) -> np.ndarray | None:
    for c in candidates:
        if c in df.columns:
            try:
                return pd.to_numeric(df[c], errors="coerce").to_numpy()
            except Exception:
                pass
    return None


# ============== 画像（参数直覆）解析与应用 ==============
def _coerce_value(s: str) -> Any:
    """把字符串转为 float/bool/int/json，否则原样字符串"""
    s_strip = s.strip()
    # 尝试 JSON（支持 true/false/null、列表、对象）
    try:
        return json.loads(s_strip)
    except Exception:
        pass
    # 布尔
    if s_strip.lower() in ("true", "false"):
        return s_strip.lower() == "true"
    # 数字
    try:
        if re.match(r"^[+-]?\d+(\.\d+)?$", s_strip):
            v = float(s_strip)
            return int(v) if v.is_integer() else v
    except Exception:
        pass
    return s  # 作为字符串返回

def _parse_override_profiles(argstr: str) -> List[Dict[str, Any]]:
    """
    解析 --param-profiles
    形如（分号隔多个画像；逗号隔键值）：
      name=Base
      name=HiAE,capex.electrolyzer_cost_AE=850,econ.opex_frac=0.035
      name=NH3storUp,capex.ammonia_storage_cost_param=0.72
      name=FixedPV,wind_cost_input=900,pv_cost_input=250

    返回：每个画像是 dict（含 name 字段；其余为要覆盖的键和值）
    """
    profiles = []
    if not argstr:
        return [{"name": "Base"}]
    for blk in re.split(r";\s*", argstr.strip()):
        if not blk:
            continue
        d: Dict[str, Any] = {}
        for kv in re.split(r",\s*", blk):
            if not kv:
                continue
            if "=" not in kv:
                continue
            k, v = kv.split("=", 1)
            k = k.strip()
            v = _coerce_value(v)
            d[k] = v
        if "name" not in d:
            d["name"] = f"Profile{len(profiles)+1}"
        profiles.append(d)
    if not profiles:
        profiles = [{"name": "Base"}]
    return profiles

def _set_by_path(obj: dict, path: str, value: Any):
    """
    按点号路径设置字典：capex.ammonia_storage_cost_param = value
    中途不存在的层级将自动创建为 dict
    """
    parts = path.split(".")
    cur = obj
    for p in parts[:-1]:
        if p not in cur or not isinstance(cur[p], dict):
            cur[p] = {}
        cur = cur[p]
    cur[parts[-1]] = value

def _apply_overrides_to_params(raw_params: dict,
                               overrides: Dict[str, Any]) -> Tuple[dict, Dict[str, Any]]:
    """
    将 overrides（除了 name 与 wind/pv 输入覆写）应用到 params（深拷贝）。
    返回 (params_applied, applied_dict)，其中 applied_dict 仅保留真正写入 params 的键值（带点号路径）。
    """
    params = copy.deepcopy(raw_params)
    applied: Dict[str, Any] = {}
    for k, v in overrides.items():
        if k == "name" or k in ("wind_cost_input", "pv_cost_input"):
            continue
        try:
            _set_by_path(params, k, v)
            applied[k] = v
        except Exception:
            # 跳过不能设置的路径
            pass
    return params, applied


# ============== LCO* 计算（与旧版一致） ==============
def _levelize_electricity(params, caps: Dict, group_df: pd.DataFrame, model, wind_cost: float, pv_cost: float) -> Dict:
    wacc   = float(params.get('discount_rate', 0.08))
    opex_f = _opex_frac(params, 0.03)
    life_wind = int(params.get('wind_lifetime', 25))
    life_pv   = int(params.get('pv_lifetime', 25))
    life_batt = int(params.get('battery_lifetime', 12))
    fom_wind = _fom_frac(params, 'wind')
    fom_pv   = _fom_frac(params, 'pv')
    fom_be   = _fom_frac(params, 'battery_energy')
    fom_bs   = _fom_frac(params, 'battery_storage')

    wc = (model.getVarByName('wind_capacity') or 0)
    pc = (model.getVarByName('pv_capacity') or 0)
    be = (model.getVarByName('battery_energy_capacity') or 0)
    bs = (model.getVarByName('battery_storage_capacity') or 0)
    wc = float(getattr(wc, "x", 0.0))
    pc = float(getattr(pc, "x", 0.0))
    be = float(getattr(be, "x", 0.0))
    bs = float(getattr(bs, "x", 0.0))

    wind_annual_unit = _annual_unit(wind_cost, life_wind, wacc, opex_f, fom_wind)
    pv_annual_unit   = _annual_unit(pv_cost,   life_pv,   wacc, opex_f, fom_pv)
    be_unit_cost     = _annual_unit(params.get('battery_energy_cost', 120), life_batt, wacc, opex_f, fom_be)
    bs_unit_cost     = _annual_unit(params.get('battery_storage_cost',120), life_batt, wacc, opex_f, fom_bs)

    batt_charge_pen = float(params.get('battery_charge_penalty', 1e-5))
    batt_dis_pen    = float(params.get('battery_discharge_penalty', 1e-5))

    T   = len(group_df)
    cfw_tmp = _pick_series(group_df, ['CF_wind_avg','cf_wind_avg','CF_Wind','cf_wind','wind_cf'])
    cfw = cfw_tmp if cfw_tmp is not None else np.zeros(T)
    cfp_tmp = _pick_series(group_df, ['CF_pv_avg','cf_pv_avg','CF_PV','cf_pv','pv_cf'])
    cfp = cfp_tmp if cfp_tmp is not None else np.zeros(T)

    surplus = [model.getVarByName(f'surplus[{t}]') for t in range(T)]
    bchg    = [model.getVarByName(f'battery_charge[{t}]') for t in range(T)]
    bdis    = [model.getVarByName(f'battery_discharge[{t}]') for t in range(T)]

    gen_kwh    = float(np.sum(cfw) * wc + np.sum(cfp) * pc)
    surplus_kw = float(sum((v.x if v else 0.0) for v in surplus))
    supply_kwh = max(gen_kwh - surplus_kw, 0.0)

    capex_py = wind_annual_unit*wc + pv_annual_unit*pc + be_unit_cost*be + bs_unit_cost*bs
    var_py   = batt_charge_pen * float(sum(v.x if v else 0.0 for v in bchg)) \
             + batt_dis_pen    * float(sum(v.x if v else 0.0 for v in bdis))

    LCOE = (capex_py + var_py) / max(supply_kwh/1000.0, 1e-9)

    return {
        'LCOE_$/MWh': float(LCOE),
        'E_supply_MWh': float(supply_kwh/1000.0),
        'annual_contribs': {
            "Wind_annual_cost_$py":  wind_annual_unit * wc,
            "PV_annual_cost_$py":    pv_annual_unit   * pc,
            "BattE_annual_cost_$py": be_unit_cost     * be,
            "BattS_annual_cost_$py": bs_unit_cost     * bs,
        }
    }

def _levelize_hydrogen(params, caps: Dict, lcoe_per_mwh: float, model) -> Dict:
    wacc   = float(params.get('discount_rate', 0.08))
    opex_f = _opex_frac(params, 0.03)
    life_el_ae  = int(params.get('electrolyzer_lifetime_AE', 15))
    life_el_pem = int(params.get('electrolyzer_lifetime_PEM', 15))
    life_h2s    = int(params.get('hydrogen_storage_lifetime', 30))
    fom_ae  = _fom_frac(params, 'electrolyzer_AE')
    fom_pem = _fom_frac(params, 'electrolyzer_PEM')
    fom_h2s = _fom_frac(params, 'hydrogen_storage')

    cap_AE  = (model.getVarByName('electrolyzer_capacity_AE')  or 0)
    cap_PEM = (model.getVarByName('electrolyzer_capacity_PEM') or 0)
    cap_H2S = (model.getVarByName('hydrogen_storage_capacity') or 0)
    cap_AE  = float(getattr(cap_AE, "x", 0.0))
    cap_PEM = float(getattr(cap_PEM,"x", 0.0))
    cap_H2S = float(getattr(cap_H2S,"x", 0.0))

    AE_cost  = float(params.get('electrolyzer_cost_AE', 700))
    PEM_cost = float(params.get('electrolyzer_cost_PEM', 900))
    H2S_cost = float(params.get('hydrogen_storage_cost', 800))

    AE_unit  = _annual_unit(AE_cost,  life_el_ae,  wacc, opex_f, fom_ae)
    PEM_unit = _annual_unit(PEM_cost, life_el_pem, wacc, opex_f, fom_pem)
    H2S_unit = _annual_unit(H2S_cost, life_h2s,    wacc, opex_f, fom_h2s)

    T = int(caps.get('T', 8760))
    p_AE  = [model.getVarByName(f'electrolyzer_power_AE[{t}]')  for t in range(T)]
    p_PEM = [model.getVarByName(f'electrolyzer_power_PEM[{t}]') for t in range(T)]
    h_chg = [model.getVarByName(f'hydrogen_charge[{t}]') for t in range(T)]

    eff_AE  = float(params.get('electrolyzer_eff_AE', 0.65))
    eff_PEM = float(params.get('electrolyzer_eff_PEM', 0.70))
    h2s_elec_factor = float(params.get('hydrogen_storage_elec', 0.001))

    prod_kwh = float(sum(((p_AE[t].x if p_AE[t] else 0.0) * eff_AE
                        + (p_PEM[t].x if p_PEM[t] else 0.0) * eff_PEM) for t in range(T)))
    elec_kwh = float(sum(((p_AE[t].x if p_AE[t] else 0.0)
                        + (p_PEM[t].x if p_PEM[t] else 0.0)) for t in range(T)))
    h2s_kwh  = float(sum(((h_chg[t].x if h_chg[t] else 0.0) * h2s_elec_factor) for t in range(T)))

    capex_py = AE_unit*cap_AE + PEM_unit*cap_PEM + H2S_unit*cap_H2S
    lcoe_per_kwh = float(lcoe_per_mwh)/1000.0
    energy_py = lcoe_per_kwh * (elec_kwh + h2s_kwh)

    kg_H2 = max(prod_kwh / 33.3, 1e-9)
    LCOH  = (capex_py + energy_py) / kg_H2

    return {
        'LCOH_$/kg': float(LCOH),
        'H2_tpy': float(kg_H2/1000.0),
        'annual_contribs': {
            "AE_annual_cost_$py":  AE_unit  * cap_AE,
            "PEM_annual_cost_$py": PEM_unit * cap_PEM,
            "H2S_annual_cost_$py": H2S_unit * cap_H2S,
        }
    }

def _levelize_heat(params, caps: Dict, lcoe_per_mwh: float, model) -> Dict:
    wacc   = float(params.get('discount_rate', 0.08))
    opex_f = _opex_frac(params, 0.03)
    life_hp = int(params.get('heat_pump_lifetime', 18))
    life_ts = int(params.get('thermal_storage_lifetime', 25))
    fom_hp  = _fom_frac(params, 'heat_pump')
    fom_ts  = _fom_frac(params, 'thermal_storage')

    cap_HP = (model.getVarByName('heat_pump_capacity') or 0)
    cap_TS = (model.getVarByName('thermal_storage_capacity') or 0)
    cap_HP = float(getattr(cap_HP, "x", 0.0))
    cap_TS = float(getattr(cap_TS, "x", 0.0))

    HP_cost = float(params.get('heat_pump_cost', 500))
    TS_cost = float(params.get('thermal_storage_cost', 50))
    HP_unit = _annual_unit(HP_cost, life_hp, wacc, opex_f, fom_hp)
    TS_unit = _annual_unit(TS_cost, life_ts, wacc, opex_f, fom_ts)

    T = int(caps.get('T', 8760))
    hp_out = [model.getVarByName(f'heat_pump_output[{t}]') for t in range(T)]
    cop = float(params.get('heat_pump_cop', 3.0))

    total_heat_kWhth = float(sum((v.x if v else 0.0) for v in hp_out))
    if total_heat_kWhth <= 0:
        return {'LCOHeat_$/MWh_th': 0.0, 'Heat_MWhth_y': 0.0, 'annual_contribs': {}}

    elec_for_hp_kWh = total_heat_kWhth / max(cop, 1e-9)
    lcoe_per_kwh = float(lcoe_per_mwh)/1000.0
    capex_py = HP_unit*cap_HP + TS_unit*cap_TS
    energy_py = lcoe_per_kwh * elec_for_hp_kWh
    lcoheat_kWhth = (capex_py + energy_py) / total_heat_kWhth

    return {
        'LCOHeat_$/MWh_th': float(lcoheat_kWhth * 1000.0),
        'Heat_MWhth_y': float(total_heat_kWhth/1000.0),
        'annual_contribs': {
            "HP_annual_cost_$py": HP_unit * cap_HP,
            "TS_annual_cost_$py": TS_unit * cap_TS,
        }
    }

def _levelize_dac(params, caps: Dict, lcoe_per_mwh: float, lcoheat_per_mwhth: float, model) -> Dict:
    wacc   = float(params.get('discount_rate', 0.08))
    opex_f = _opex_frac(params, 0.03)
    life_dac      = int(params.get('dac_lifetime', 20))
    life_co2_comp = int(params.get('co2_compressor_lifetime', 15))
    life_co2_sto  = int(params.get('co2_storage_lifetime', 30))
    fom_dac = _fom_frac(params, 'dac')
    fom_cc  = _fom_frac(params, 'co2_compressor')
    fom_cs  = _fom_frac(params, 'co2_storage')

    cap_DAC  = (model.getVarByName('dac_capacity') or 0)
    cap_CO2c = (model.getVarByName('co2_compressor_capacity') or 0)
    cap_CO2s = (model.getVarByName('co2_storage_capacity') or 0)
    cap_DAC  = float(getattr(cap_DAC, "x", 0.0))
    cap_CO2c = float(getattr(cap_CO2c,"x", 0.0))
    cap_CO2s = float(getattr(cap_CO2s,"x", 0.0))

    DAC_cost  = float(params.get('dac_cost', 600))
    CO2c_cost = float(params.get('co2_compressor_cost', 50))
    CO2s_cost = float(params.get('co2_storage_cost', 800))

    DAC_unit  = _annual_unit(DAC_cost,  life_dac,      wacc, opex_f, fom_dac)
    CO2c_unit = _annual_unit(CO2c_cost, life_co2_comp, wacc, opex_f, fom_cc)
    CO2s_unit = _annual_unit(CO2s_cost, life_co2_sto,  wacc, opex_f, fom_cs)

    sorbent_cost    = float(params.get('sorbent_cost', 0.0))
    dac_elec_per_kg = float(params.get('dac_elec_per_kg', 0.6))
    co2_comp_elec   = float(params.get('co2_compression_energy', 0.11))
    heat_pump_cop   = float(params.get('heat_pump_cop', 3.0))

    T = int(caps.get('T', 8760))
    dac_CO2        = [model.getVarByName(f'dac_CO2[{t}]') for t in range(T)]
    dac_heat_input = [model.getVarByName(f'dac_heat_input[{t}]') for t in range(T)]
    total_CO2_kg   = float(sum((v.x if v else 0.0) for v in dac_CO2))
    if total_CO2_kg <= 0:
        return {'LCOC_$/tCO2': 0.0, 'CO2_tpy': 0.0, 'annual_contribs': {}}

    total_heat_kWhth = float(sum((v.x if v else 0.0) for v in dac_heat_input))
    elec_kWh = total_CO2_kg * (dac_elec_per_kg + co2_comp_elec)

    lcoe_per_kwh = float(lcoe_per_mwh)/1000.0
    if lcoheat_per_mwhth and lcoheat_per_mwhth > 0:
        heat_price_kWhth = float(lcoheat_per_mwhth)/1000.0
    else:
        heat_price_kWhth = lcoe_per_kwh / max(heat_pump_cop, 1e-9)

    capex_py   = DAC_unit*cap_DAC + CO2c_unit*cap_CO2c + CO2s_unit*cap_CO2s
    sorbent_py = sorbent_cost * total_CO2_kg
    energy_py  = lcoe_per_kwh * elec_kWh + heat_price_kWhth * total_heat_kWhth

    LCOC = (capex_py + sorbent_py + energy_py) / max(total_CO2_kg/1000.0, 1e-9)

    return {
        'LCOC_$/tCO2': float(LCOC),
        'CO2_tpy': float(total_CO2_kg/1000.0),
        'annual_contribs': {
            "DAC_annual_cost_$py":       DAC_unit  * cap_DAC,
            "CO2_comp_annual_cost_$py":  CO2c_unit * cap_CO2c,
            "CO2_store_annual_cost_$py": CO2s_unit * cap_CO2s,
        }
    }

def _synthesis_contribs(params, model, caps: Dict, product_key: str) -> Dict:
    pk = (product_key or '').lower()
    if pk in ("nh3",): pk = "ammonia"
    if pk in ("meoh",): pk = "methanol"

    cap_syn  = float(caps.get("Cap_syn", 0.0) or 0.0)
    cap_stor = float(caps.get("Cap_stor", 0.0) or 0.0)

    if cap_syn == 0.0:
        for nm in ["synthesis_capacity", f"{pk}_synthesis_capacity",
                   "FTL_synthesis_capacity", "MeOH_synthesis_capacity", "NH3_synthesis_capacity"]:
            v = model.getVarByName(nm)
            if v is not None:
                try:
                    cap_syn = float(v.x); break
                except:
                    pass

    if cap_stor == 0.0:
        for nm in ["product_storage_capacity", "synthesis_storage_capacity",
                   f"{pk}_storage_capacity", "FTL_storage_capacity",
                   "MeOH_storage_capacity", "NH3_storage_capacity"]:
            v = model.getVarByName(nm)
            if v is not None:
                try:
                    cap_stor = float(v.x); break
                except:
                    pass

    wacc   = float(params.get("discount_rate", 0.08))
    opex_f = float(params.get("econ", {}).get("opex_frac", 0.03))
    fom_syn  = float(params.get("fom_frac", {}).get(f"{pk}_synthesis", 0.0) or 0.0)
    fom_stor = float(params.get("fom_frac", {}).get(f"{pk}_storage", 0.0) or 0.0)

    synth_life = int((params.get(f"{pk}_synthesis_lifetime")
                      or params.get("synthesis_lifetime") or 20))
    stor_life = int((params.get(f"{pk}_storage_lifetime")
                     or params.get("synthesis_storage_lifetime")
                     or params.get("product_storage_lifetime")
                     or 20))

    def _get_cost(keys, default=0.0):
        for k in keys:
            if k in params:
                try: return float(params[k])
                except: pass
            v = params.get("capex", {}).get(k, None)
            if v is not None:
                try: return float(v)
                except: pass
        return float(default)

    store_cost = _get_cost([f"{pk}_storage_cost_param"], default=0.0)
    synth_cost = _get_cost([f"{pk}_synthesis_cost_param", f"{pk}_synthesis_cost",
                            "synthesis_cost_param", "synthesis_cost"], default=0.0)

    def _annual_unit_local(capex_unit, life, opex, fom):
        return float(capex_unit) * (_crf(wacc, life) + float(opex) + float(fom))

    synth_unit = _annual_unit_local(synth_cost, synth_life, opex_f, fom_syn)
    stor_unit  = _annual_unit_local(store_cost, stor_life, opex_f, fom_stor)

    synth_py = synth_unit * cap_syn
    stor_py  = stor_unit  * cap_stor

    if PRINT_DEBUG_SYNTH:
        print(f"[SYNTH] prod={pk} | Cap_syn={cap_syn}, Cap_stor={cap_stor} | "
              f"synth_cost={synth_cost}, store_cost={store_cost} | "
              f"life_syn={synth_life}, life_stor={stor_life} | py_syn={synth_py}, py_stor={stor_py}")

    return {
        "Synth_annual_cost_$py": float(synth_py),
        "Synth_store_annual_cost_$py": float(stor_py),
    }


# ============== 主流程（仅“参数直覆敏感性”；不再分气候年） ==============
def _scenario_from_forced_cf_file() -> str:
    bn = os.path.basename(FORCED_CF_TIMESERIES)
    m = re.match(r"median_points_cf_timeseries_(?P<scenario>.+)\.csv$", bn, re.IGNORECASE)
    if not m:
        raise ValueError(f"Cannot parse scenario from forced CF file name: {bn}")
    return m.group("scenario")

def _list_scenarios_single_table(cf_dir: str, year_filter: str | None) -> list[str]:
    if FORCED_CF_TIMESERIES:
        if not os.path.exists(FORCED_CF_TIMESERIES):
            raise FileNotFoundError(FORCED_CF_TIMESERIES)
        return [_scenario_from_forced_cf_file()]

    import glob
    pat = re.compile(r"median_points_cf_timeseries_(?P<scenario>.+)\.csv$", re.IGNORECASE)
    files = sorted(glob.glob(os.path.join(cf_dir, "median_points_cf_timeseries_*.csv")))
    kept = []
    for fp in files:
        bn = os.path.basename(fp)
        m = pat.search(bn)
        if not m:
            continue
        scen_raw = m.group("scenario")

        if re.search(r"_(2021|2022|2023)$", scen_raw):
            continue

        if year_filter and str(year_filter).strip():
            try:
                yr = pick_year_tag_from_filename(scen_raw)
            except Exception:
                yr = None
            if str(yr) != str(year_filter):
                continue

        kept.append(scen_raw)

    global ONLY_ONE_SCENARIO
    if ONLY_ONE_SCENARIO:
        kept = [s for s in kept if re.search(ONLY_ONE_SCENARIO, s, flags=re.IGNORECASE)]
    return kept

def _expand_shared_cf_products(df: pd.DataFrame) -> pd.DataFrame:
    """Use the same hourly CF nodes for ammonia, methanol, and SAF/FTL."""
    products = ["ammonia", "methanol", "ftl"]
    if "product" not in df.columns:
        base = df.copy()
    else:
        base = df.copy()
        present = {str(v).lower() for v in base["product"].dropna().unique()}
        if products[0] in present:
            base = base[base["product"].astype(str).str.lower() == products[0]].copy()
    frames = []
    for prod in products:
        tmp = base.copy()
        tmp["product"] = prod
        frames.append(tmp)
    return pd.concat(frames, ignore_index=True)

def _load_single_cf_table(cf_dir: str, scen_raw: str) -> pd.DataFrame | None:
    fp = FORCED_CF_TIMESERIES if FORCED_CF_TIMESERIES else os.path.join(cf_dir, f"median_points_cf_timeseries_{scen_raw}.csv")
    if not os.path.exists(fp):
        return None
    df = pd.read_csv(fp)
    if 'scenario' not in df.columns:
        df['scenario'] = normalize_scenario(scen_raw)
    if FORCED_CF_TIMESERIES:
        df = _expand_shared_cf_products(df)
    elif 'product' not in df.columns:
        df['product'] = 'methanol'
    return df

def _run_param_override_for_scenario(scen_raw: str,
                                     cf_dir: str,
                                     out_root: str,
                                     profiles: List[Dict[str, Any]],
                                     only_one_node: dict | None = None,
                                     stop_after_first: bool = False):
    """对单场景（单表）在多个“参数直覆画像”下运行；每个节点输出一个 CSV，行索引为 capex_profile。"""
    df_all = _load_single_cf_table(cf_dir, scen_raw)
    if df_all is None or df_all.empty:
        print(f"· {scen_raw}: 数据缺失，跳过。"); 
        return

    year_tag  = pick_year_tag_from_filename(scen_raw)
    wind_map, pv_map = pick_cost_maps(year_tag)
    params_base = select_params_by_year(year_tag)
    scen_type   = classify_scenario_for_cost(scen_raw)  # 仅用于 WIND 特例
    scen_folder = scenario_output_folder(scen_raw)

    # 输出目录：param_sensitivity
    scen_out_root = os.path.join(out_root, f"Year_{year_tag}", scen_folder, "param_sensitivity")
    os.makedirs(scen_out_root, exist_ok=True)

    # 节点键
    group_cols = [c for c in ['scenario','product','continent','cluster','latitude','longitude','region']
                  if c in df_all.columns]
    unique_keys = set(tuple(v) for v in df_all[group_cols].drop_duplicates().itertuples(index=False, name=None))

    for tup in unique_keys:
        key_map = dict(zip(group_cols, tup))
        if only_one_node and not all(str(key_map.get(k)) == str(v) for k, v in only_one_node.items()):
            continue

        prod_key = (key_map.get('product') or 'methanol').lower()
        cont     = key_map.get('continent', 'Unknown')
        cl       = key_map.get('cluster', 'X')
        region_code = int(key_map.get('region')) if key_map.get('region') is not None else None

        # 该节点的时序切片
        cond = np.ones(len(df_all), dtype=bool)
        for col, val in key_map.items():
            cond &= (df_all[col] == val)
        group = df_all.loc[cond]
        if group.empty:
            continue

        recs = []
        for overrides in profiles:
            profile_name = str(overrides.get("name", "Base"))

            # 区域风/光成本（先取区域图/特例，再允许画像覆写）
            wind_cost_in = float(wind_map.get(region_code, 800))
            pv_cost_in   = float(225 if scen_type == 'WIND' else pv_map.get(region_code, 350))
            if "wind_cost_input" in overrides:
                try: wind_cost_in = float(overrides["wind_cost_input"])
                except: pass
            if "pv_cost_input" in overrides:
                try: pv_cost_in = float(overrides["pv_cost_input"])
                except: pass

            # 应用参数直覆
            params_applied, applied_params = _apply_overrides_to_params(params_base, overrides)

            # 建模&优化
            try:
                model, T = _build_one_model_from_group(group, prod_key, wind_cost_in, pv_cost_in, params_applied)
                model.optimize()
                if model.status != GRB.OPTIMAL:
                    print(f"   · {scen_raw} | {cont} 簇{cl} | {prod_key} | {profile_name}: 非最优({model.status})，跳过。")
                    continue
            except Exception as e:
                print(f"   · {scen_raw} | {cont} 簇{cl} | {prod_key} | {profile_name}: 失败：{e}")
                continue

            annual_cost = float(model.ObjVal)
            tgt = _product_target_from_params(prod_key, params_applied)
            lco_prod = (annual_cost / tgt) if tgt and tgt > 0 else np.nan
            caps = _extract_capacities(model, prod_key)
            caps['T'] = T  # 时序长度

            # 计算 LCO*
            lcoe_pack    = _levelize_electricity(params_applied, caps, group, model, wind_cost_in, pv_cost_in)
            LCOE_val     = lcoe_pack['LCOE_$/MWh']
            lcoh_pack    = _levelize_hydrogen(params_applied, caps, LCOE_val, model)
            lcoheat_pack = _levelize_heat(params_applied, caps, LCOE_val, model)
            lcoc_pack    = _levelize_dac(params_applied, caps, LCOE_val, lcoheat_pack['LCOHeat_$/MWh_th'], model)

            # 合成装置/储存贡献
            synth_pack = _synthesis_contribs(params_applied, model, caps, prod_key)

            rec = {
                "capex_profile": profile_name,
                "Annual_cost_$": annual_cost,
                f"LCO_{prod_key}_$/t": lco_prod,
                "LCOE_$/MWh": float(LCOE_val),
                "LCOH_$/kg":  float(lcoh_pack['LCOH_$/kg']),
                "LCOHeat_$/MWh_th": float(lcoheat_pack['LCOHeat_$/MWh_th']),
                "LCOC_$/tCO2": float(lcoc_pack['LCOC_$/tCO2']),
                "opex_frac": _opex_frac(params_applied, 0.03),
                "wind_cost_input": wind_cost_in,
                "pv_cost_input":   pv_cost_in,
            }
            rec.update(caps)
            for pack in (lcoe_pack, lcoh_pack, lcoheat_pack, lcoc_pack):
                for k, v in pack.get("annual_contribs", {}).items():
                    rec[k] = float(v)
            rec.update(synth_pack)

            # —— 把“覆盖后的变量”也写进输出（用于后续计算）
            # 1) 显式写出所有 override 的最终值（名字用 override__ 前缀）
            for k, v in overrides.items():
                if k == "name": 
                    continue
                rec[f"override__{k}"] = v
            # 2) 对真正写入 params 的键（点号路径）也做一次显式展开，便于筛选（与上可能重复，不冲突）
            for k, v in applied_params.items():
                rec[f"override__{k}"] = v

            recs.append(rec)

            if stop_after_first:
                break

        if not recs:
            continue

        # 同一节点：合并所有画像 → 索引 capex_profile
        df_res = pd.DataFrame(recs).set_index("capex_profile").sort_index()
        node_dir = os.path.join(scen_out_root, f"{prod_key}_{cont}_cluster{cl}")
        os.makedirs(node_dir, exist_ok=True)

        out_csv = os.path.join(node_dir, "param_sensitivity_summary.csv")
        df_res.to_csv(out_csv)
        print(f"✓ 输出：{out_csv}")

        if stop_after_first:
            print("· 单节点测试完成，提前结束。")
            break


def run_param_sensitivity(cf_dir: str, out_root: str, profiles: List[Dict[str, Any]]):
    scen_list = _list_scenarios_single_table(cf_dir, YEAR_FILTER)
    if ONLY_ONE_SCENARIO:
        scen_list = [s for s in scen_list if re.search(ONLY_ONE_SCENARIO, s, flags=re.IGNORECASE)]
    print(f"▶ 规划年 {YEAR_FILTER} | 场景数量：{len(scen_list)} | 参数画像：{len(profiles)}")
    for i, scen_raw in enumerate(scen_list, 1):
        print(f"\n==== [{i}/{len(scen_list)}] {scen_raw} ====")
        _run_param_override_for_scenario(
            scen_raw, cf_dir, out_root, profiles,
            only_one_node=ONLY_ONE_NODE,
            stop_after_first=STOP_AFTER_FIRST_NODE
        )


# ===== CLI / Notebook 入口 =====
def _build_cli():
    p = argparse.ArgumentParser(description="参数直覆敏感性分析（不含气候年对比）")
    p.add_argument("--cf-dir", required=True)
    p.add_argument("--out-root", required=True)
    p.add_argument("--year", default=YEAR_FILTER)
    p.add_argument("--only-scenario", default=ONLY_ONE_SCENARIO)
    p.add_argument("--only-node", default=None)
    p.add_argument("--stop-after-first", action="store_true")
    # 多个画像；分号分隔画像，逗号分隔键值；支持点路径与 wind/pv 直接输入覆盖
    # 例：
    #   name=Base
    #   name=HiAE,capex.electrolyzer_cost_AE=850,econ.opex_frac=0.035
    #   name=NH3storUp,capex.ammonia_storage_cost_param=0.72
    #   name=FixPV,wind_cost_input=900,pv_cost_input=250
    p.add_argument("--param-profiles", default="name=Base")
    return p


# ===== Notebook/Script 入口（避免 Notebook 触发 CLI） =====
def _is_notebook():
    try:
        from IPython import get_ipython
        ip = get_ipython()
        return bool(ip) and hasattr(ip, "kernel")
    except Exception:
        return False

if __name__ == "__main__":
    if _is_notebook():
        # 在 Notebook 中不走 argparse，直接提示如何调用
        print("[INFO] Detected Jupyter/Notebook. Skipping CLI. "
              "Call run_param_sensitivity(cf_dir, out_root, profiles) manually.")
    else:
        args = _build_cli().parse_args()
        global YEAR_FILTER, ONLY_ONE_SCENARIO, ONLY_ONE_NODE, STOP_AFTER_FIRST_NODE
        YEAR_FILTER = args.year
        ONLY_ONE_SCENARIO = args.only_scenario
        ONLY_ONE_NODE = None
        STOP_AFTER_FIRST_NODE = bool(args.stop_after_first)

        profiles = _parse_override_profiles(args.param_profiles)
        run_param_sensitivity(args.cf_dir, args.out_root, profiles)

# # Notebook 快速运行示例
cf_dir = os.path.dirname(FORCED_CF_TIMESERIES)
out_root = r"D:\OneDrive\POTSED\8760Supply&Demand\Global_cn"

profiles = _parse_override_profiles(
    "name=Base;"
    "name=HighAE,electrolyzer_cost_AE=350;"
    "name=HighPEM,electrolyzer_cost_PEM=650;"
    "name=HighSyn,ftl_synthesis_cost=22000 ,ammonia_synthesis_cost=9000 ,methanol_synthesis_cost=8000," \
    "ftl_hydrogen_ratio=0.51 ,ftl_co2_ratio=3.9 ,ftl_electricity_ratio=0.3;"
    "name=LowSyn,ftl_synthesis_cost=9400 ,ammonia_synthesis_cost=3500 ,methanol_synthesis_cost=2600," \
    "ftl_hydrogen_ratio=0.48 ,ftl_co2_ratio=3.3 ,ftl_electricity_ratio=0.3;"
    "name=HighBat,battery_energy_cost=200;"
    "name=LowBat,battery_energy_cost=50;"
    "name=HighDAC,dac_cost=7800, dac_elec_per_kg= 0.31, dac_heat_per_kg=1.3, sorbent_cost=0.02;"
    "name=LowDAC,dac_cost=3000, dac_elec_per_kg= 0.22, dac_heat_per_kg=1.0, sorbent_cost=0.009;"
)

run_param_sensitivity(cf_dir, out_root, profiles)


In [ ]:
# -*- coding: utf-8 -*-
"""
参数敏感性：产品聚合（两层横轴标签 + ammonia 三图布局 + 剔除 HighDAC/LowDAC）
图 a：AE/PEM 堆积 + LCOH（右轴）
图 b：风/光（宽堆积）套 电池E/P（窄堆积） + LCOE（右轴）
图 c：仅 methanol/ftl：DAC 装机（统一 t/h） + 年容量因子标注 + LCOC（右轴）
图 d：合成 Cap_syn 柱 + 年容量因子标注；右轴 所有情景 风/光（窄堆叠，置上层）
图 e：成本分解（类型 -> 情景），ammonia 剔除 HighDAC/LowDAC
"""

import os, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from matplotlib.lines import Line2D


# ===================== 基本配置 =====================
OUT_ROOT = r"D:\OneDrive\POTSED\8760Supply&Demand\Global_cn"
YEAR_TAG = "2035"
PRODUCTS = ["ammonia", "methanol", "ftl"]

PARAM_DIR = os.path.join(OUT_ROOT, f"Year_{YEAR_TAG}", "RegionRE", "param_sensitivity")

# 情景分组（capex_profile 的取值；大小写不敏感）
GROUP_A = ["Base", "HighAE",  "HighPEM"]
GROUP_B = ["Base", "HighBat", "LowBat"]
GROUP_C = ["Base", "HighDAC", "LowDAC"]
GROUP_D = ["Base", "HighSyn", "LowSyn"]
ALL_PROFILES = ["Base","HighAE","HighPEM","HighBat","LowBat","HighDAC","LowDAC","HighSyn","LowSyn"]
# 成本分解要剔除的情景（仅对 ammonia 剔除 HighDAC/LowDAC）
EXCLUDE_IN_COST_AMMONIA = {"HighDAC","LowDAC"}

# ===== 手动轴范围（None 表示自动） =====
# 例：AX_LIMITS['b']['left']  = (0, 120000)
#     AX_LIMITS['b']['right'] = (20, 80)
AX_LIMITS = {
    'a': {'left': (None, None), 'right': (None, 4)},
    'b': {'left': (None, None), 'right': (None, 60)},
    'c': {'left': (None, None), 'right': (None, 400)},
    'd': {'left': (None, None), 'right': (1, 3.5)},
    'e': {'left': (None, None)},
}
AUTO_TOP_PAD = 0.10  # 当轴未手动设置时，顶部额外留白比例

# ========= 指定配色（你给的配色表） =========
PANEL_COLORS = {
    '风电/光伏':   ('#4DBBD5', '#E64B35'),
    'AE/PEM':      ('#3C5488', '#F39B7F'),
    '电池/风+光': ('#00A087', '#7E6148'),
    '储能/风+光': ('#8491B4', '#91D1C2'),
    '氢储/电解槽': ('#B09C85', '#8FAADC'),
    '电解槽/合成': ('#2F8F7F', '#D89000'),
}

# —— 面板内具体对象颜色映射（统一集中管理）——
COL_AE  = PANEL_COLORS['AE/PEM'][0]
COL_PEM = PANEL_COLORS['AE/PEM'][1]

COL_WIND = PANEL_COLORS['风电/光伏'][0]
COL_PV   = PANEL_COLORS['风电/光伏'][1]

# 电池 E/P（沿用你给定的映射）
COL_BATT_E = PANEL_COLORS['储能/风+光'][0]
COL_BATT_P = PANEL_COLORS['电池/风+光'][0]

# (d) 右轴风/光同样使用风/光配色（实色、无透明）
COL_WIND_BASE = PANEL_COLORS['风电/光伏'][0]
COL_PV_BASE   = PANEL_COLORS['风电/光伏'][1]

# (c) DAC：复用“氢储/电解槽”里的‘氢储’色，避免与电解槽/合成冲突
COL_DAC = PANEL_COLORS['氢储/电解槽'][0]

# 散点与边线统一风格
COL_DOT = "#000000"
EDGE_KWARGS = dict(edgecolor="black", linewidth=0.6)

# 成本分解颜色：发电/电解槽/储能/合成/DAC/供热
COLORS_BREAK = [
    PANEL_COLORS['风电/光伏'][0],      # 发电
    PANEL_COLORS['电解槽/合成'][0],    # 电解槽
    PANEL_COLORS['储能/风+光'][0],     # 储能
    PANEL_COLORS['电解槽/合成'][1],    # 合成
    PANEL_COLORS['氢储/电解槽'][0],    # DAC（借用氢储色）
    PANEL_COLORS['储能/风+光'][1],     # 供热（借用储能/风+光的辅色）
]

plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 300,
    "font.family": "sans-serif",
    "font.sans-serif": ["Microsoft YaHei","SimHei","Noto Sans CJK SC","DejaVu Sans"],
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.8, "xtick.major.width":0.8, "ytick.major.width":0.8,
    "legend.frameon": False,

    # 全局基准字号（不写也行，下面会分别控制）
    "font.size": 12,

    # 各要素字号（建议显式设置，最可控）
    "axes.titlesize": 12,    # 子图标题 (ax.set_title)
    "axes.labelsize": 12,    # 坐标轴标签 (ax.set_xlabel / ylabel)
    "xtick.labelsize": 12,   # x 轴刻度标签
    "ytick.labelsize": 9,   # y 轴刻度标签
    "legend.fontsize": 10,   # 图例
    "figure.titlesize": 12,  # 整体大标题 (fig.suptitle)
})

# ===================== 工具 =====================
def _key_ci(d: dict, key: str):
    """在 dict d 中做不区分大小写的 key 匹配，返回真实 key；找不到则返回 None。"""
    if key in d: 
        return key
    kl = key.lower()
    for k in d.keys():
        if isinstance(k, str) and k.lower() == kl:
            return k
    return None

def _safe_row(agg: dict, prof: str, cluster, warn_prefix=""):
    """稳健地从 agg 里读取某 profile + cluster 的 dict；缺就返回 {}。"""
    kprof = _key_ci(agg, prof)
    if kprof is None:
        return {}
    row = agg[kprof].get(cluster, {})
    if row == {}:
        row = agg[kprof].get(int(cluster) if str(cluster).isdigit() else cluster, {})
    return row if isinstance(row, dict) else {}

def _list_param_nodes():
    return sorted(glob.glob(os.path.join(PARAM_DIR, "*_*_cluster*")))

def _parse_prod_cluster(bn: str):
    m = re.match(r"(?P<prod>[^_]+)_(?P<cont>.+)_cluster(?P<cl>.+)", bn, re.IGNORECASE)
    if not m: return None, None, None
    prod = m.group("prod").lower()
    cont = m.group("cont")
    cl_raw = m.group("cl")
    try: cl = int(float(cl_raw))
    except: cl = cl_raw
    return prod, cont, cl

def _read_param_summary(node_dir: str) -> pd.DataFrame | None:
    fp = os.path.join(node_dir, "param_sensitivity_summary.csv")
    if not os.path.exists(fp): return None
    df = pd.read_csv(fp)
    # ① 情景列优先 capex_profile
    pref_cols = ["capex_profile","profile","param_profile","name"]
    prof_col = next((c for c in pref_cols if c in df.columns), None)
    if prof_col is None:
        prof_col = next((c for c in df.columns if c.lower() in ("capex_profile","profile","param_profile","name")), None)
    if prof_col is None:
        raise ValueError(f"{fp} 缺情景列（capex_profile/profile/param_profile/name）")
    df[prof_col] = df[prof_col].astype(str)
    df = df.set_index(prof_col)
    for c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

WANT_COLS = [
    "Cap_AE_kW","Cap_PEM_kW","Cap_wind_kW","Cap_pv_kW",
    "Cap_batt_E_kWh","Cap_batt_P_kW",
    "Cap_DAC_kgCO2h",
    "Cap_syn",
    "Cap_H2_store",
    "LCOE_$/MWh","LCOH_$/kg","LCOC_$/tCO2",
    # 年度成本分解
    "Wind_annual_cost_$py","PV_annual_cost_$py",
    "AE_annual_cost_$py","PEM_annual_cost_$py",
    "BattE_annual_cost_$py","BattS_annual_cost_$py","H2S_annual_cost_$py",
    "Synth_annual_cost_$py","Synth_store_annual_cost_$py",
    "DAC_annual_cost_$py","CO2_comp_annual_cost_$py","CO2_store_annual_cost_$py",
    "HP_annual_cost_$py","TS_annual_cost_$py",
    # 产量列（合成年容量因子）
    "MeOH_tpy","NH3_tpy","FTL_tpy","methanol_tpy","ammonia_tpy","ftl_tpy",
    # CO2 实绩（优先用于 DAC 年 CF）
    "CO2_tpy"
]

def _aggregate_profiles(product: str):
    """agg[profile][cluster][col] = mean over nodes"""
    buckets = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
    for nd in _list_param_nodes():
        bn = os.path.basename(nd)
        prod, _cont, cl = _parse_prod_cluster(bn)
        if prod != product: continue
        df = _read_param_summary(nd)
        if df is None: continue
        for prof in ALL_PROFILES:
            hits = [idx for idx in df.index if idx.lower() == prof.lower()]
            if not hits: continue
            row = df.loc[hits[0]]
            for c in WANT_COLS:
                if c in row:
                    val = row[c]
                    # 年度投资成本列除以10
                    if c.endswith("annual_cost_$py"):
                        val = val / 10.0 if pd.notna(val) else val
                    buckets[prof][cl][c].append(val)
    agg = {}
    for prof, by_cl in buckets.items():
        agg[prof] = {}
        for cl, cols in by_cl.items():
            agg[prof][cl] = {k: (np.nanmean(v) if len(v)>0 else np.nan) for k, v in cols.items()}
    return agg

def _pick_clusters(agg):
    all_cls = set()
    for d in agg.values(): all_cls |= set(d.keys())
    prio = [c for c in [0,1,2] if c in all_cls]
    rest = [c for c in sorted(all_cls, key=lambda x: int(x) if str(x).isdigit() else 999) if c not in prio]
    out = prio + rest
    return out[:max(3, len(prio))] if out else []

def _dac_tph_and_cf(row: dict, product: str) -> tuple[float, float]:
    """
    返回 (名义装机 t/h, 年容量因子 CF)
    """
    kgph = row.get("Cap_DAC_kgCO2h", np.nan)
    if kgph is None or not np.isfinite(kgph) or kgph <= 0:
        return np.nan, np.nan
    p = (product or "").lower()
    if p == "methanol":
        demand_tpy = 14000.0
    elif p == "ftl":
        demand_tpy = 36000.0
    else:
        demand_tpy = 0.0
    if demand_tpy <= 0:
        return kgph, np.nan
    cf = demand_tpy / (kgph * 8760.0)
    return kgph, cf

def _syn_cf(row: dict, product: str) -> float:
    """合成年容量因子（CF）恒用固定年产量 10000 t/年"""
    cap_syn = row.get("Cap_syn", np.nan)
    if not (cap_syn and np.isfinite(cap_syn) and cap_syn > 0):
        return np.nan
    fixed_tpy = 10000.0
    return fixed_tpy / (cap_syn * 8760.0)

# ========= 标注与坐标轴工具 =========
def _annotate_bars(ax, bars, fmt="{:.0f}", rotation=0, fontsize=9, inside=False):
    """给柱（或假柱）打数值标签；横向；inside=True 时贴在柱内顶端，False 为柱外上方。"""
    ymin, ymax = ax.get_ylim()
    span = (ymax - ymin) if ymax > ymin else 1.0
    pad = 0.01 * span
    seq = bars if hasattr(bars, "__iter__") else [bars]
    for b in seq:
        h = b.get_height()
        if not np.isfinite(h) or h == 0:
            continue
        x = b.get_x() + b.get_width()/2.0
        y = b.get_y() + h
        if inside:
            y = y - pad
            va = "top"
        else:
            y = y + pad
            va = "bottom"
        ax.text(x, y, fmt.format(h), ha="center", va=va, fontsize=fontsize, rotation=rotation, clip_on=False)

def _annotate_top_cross(ax_data, ax_draw, bars, labels, fs=9, dy_frac=0.01):
    """
    在 ax_draw 上绘制文本，但使用 ax_data 的数据坐标系:
      - ax_data: 提供 bars 和坐标系（这里是 axd）
      - ax_draw: 实际绘制的轴（这里是 axd2，位于上层）
      - bars/labels: 与 _annotate_top 相同
    """
    ymin, ymax = ax_data.get_ylim()
    span = (ymax - ymin) if ymax > ymin else 1.0
    dy = dy_frac * span
    for b, lab in zip(bars, labels):
        h = b.get_height()
        if not np.isfinite(h) or h == 0:
            continue
        x = b.get_x() + b.get_width() / 2.0
        y = b.get_y() + h + dy
        ax_draw.text(
            x, y, lab,
            ha="center", va="bottom", fontsize=fs,
            transform=ax_data.transData,   # 用 ax_data 的数据坐标
            zorder=10, clip_on=False
        )


def _annotate_totals(ax, x_positions, heights, width, fmt="{:.0f}", fontsize=9, inside=False):
    """
    仅用于‘堆叠总值’标签：直接在 (x, 总高度) 处写文本，避免类/闭包绑定问题。
    - inside=True 贴在柱内顶端；False 贴在柱外上方。
    - width 参数保留签名一致（此实现不再依赖它）。
    """
    # 统一数值 & 留白
    try:
        xs = np.asarray(x_positions, dtype=float)
    except Exception:
        xs = np.array([float(x) for x in x_positions])
    hs = np.array(
        [0.0 if (h is None or not np.isfinite(h)) else float(h) for h in heights],
        dtype=float
    )

    ymin, ymax = ax.get_ylim()
    span = (ymax - ymin) if ymax > ymin else 1.0
    pad = 0.01 * span

    for x, h in zip(xs, hs):
        if not np.isfinite(h) or h == 0.0:
            continue
        y = h - pad if inside else h + pad
        va = "top" if inside else "bottom"
        ax.text(
            x, y, fmt.format(h),
            ha="center", va=va,
            fontsize=fontsize, rotation=0, clip_on=False
        )

def _annotate_scatter(ax, xs, ys, fmt="{:.1f}", fontsize=9, dy_frac=0.01):
    """散点标签：白色半透明背景 + 灰色边框；横向。"""
    if not (isinstance(xs, (list, tuple, np.ndarray)) and isinstance(ys, (list, tuple, np.ndarray))):
        return
    ymin, ymax = ax.get_ylim()
    span = (ymax - ymin) if ymax > ymin else 1.0
    dy = dy_frac * span
    bbox = dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.8, linewidth=0.8)
    for x, y in zip(xs, ys):
        if np.isfinite(y):
            ax.text(x, y + dy, fmt.format(y), ha="center", va="bottom", fontsize=fontsize, bbox=bbox)

def _apply_axis_limits(ax, key, side="left"):
    lim = AX_LIMITS.get(key, {}).get(side, (None, None))
    if lim is None:
        return
    ymin, ymax = lim
    if ymin is not None or ymax is not None:
        ax.set_ylim(ymin, ymax)
    else:
        # 自动情况下做 10% 顶部留白
        lo, hi = ax.get_ylim()
        ax.set_ylim(lo, hi*(1.2 + AUTO_TOP_PAD))

# ========= 两层横轴：情景（上层竖排）＋类型（下层居中） =========
def grouped_positions(clusters, profiles, gap_between_clusters=2.0, bar_group_width=0.85):
    nC, nP = len(clusters), len(profiles)
    prof_step = 1.0
    centers_flat = []
    prof_centers_by_cluster = []
    cluster_centers = []

    x0 = 0.0
    for _c in clusters:
        centers = [x0 + i*prof_step for i in range(nP)]
        prof_centers_by_cluster.append(centers)
        centers_flat.extend(centers)
        cluster_centers.append(np.mean(centers))
        x0 = centers[-1] + gap_between_clusters

    width_wide = bar_group_width
    width_narrow = bar_group_width * 0.6
    return np.array(centers_flat), prof_centers_by_cluster, width_wide, width_narrow, np.array(cluster_centers)

def apply_two_tier_xticks(ax, clusters, profiles, centers_flat, cluster_centers, pad=34):
    # 上层：情景（竖排）
    labels = []
    for _c in clusters:
        for p in profiles:
            labels.append(p)
    ax.set_xticks(centers_flat)
    ax.set_xticklabels(labels, rotation=90, ha="center")

    # 下层：类型（只显示一次，居中；pad 控制“向下”距离）
    ax2 = ax.secondary_xaxis('bottom')
    ax2.set_xticks(cluster_centers)
    ax2.set_xticklabels([f"类型{int(c)+1}" for c in clusters])
    ax2.tick_params(length=0, pad=pad)

def _annotate_top(ax, bars, labels, fs=9):
    for b, lab in zip(bars, labels):
        h = b.get_height()
        if not np.isfinite(h) or h == 0:
            continue
        ax.text(b.get_x()+b.get_width()/2, h, lab, ha="center", va="bottom", fontsize=fs)

# ===================== 绘图 =====================
def plot_panels_for_product(product: str):
    agg = _aggregate_profiles(product)
    if not agg:
        print(f"[{product}] 未找到任何 profile 的数据。"); return

    clusters = _pick_clusters(agg)
    if not clusters:
        print(f"[{product}] 未识别到 cluster。"); return

    # ammonia：第一行 3 图；methanol/ftl：4 图
    first_row_cols = 3 if product == "ammonia" else 4
    fig_w = 16 if first_row_cols==3 else 18
    fig_h = 9.5 if first_row_cols==3 else 10.0
    fig = plt.figure(figsize=(fig_w, fig_h))

    if first_row_cols == 3:
        gs = fig.add_gridspec(2, 3, left=0.05, right=0.995, bottom=0.14, top=0.92, wspace=0.30, hspace=0.55)
        axa = fig.add_subplot(gs[0,1])
        axb = fig.add_subplot(gs[0,0])
        axd = fig.add_subplot(gs[0,2])   # 注意：ammonia 把 d 提前到第一行
        axe = fig.add_subplot(gs[1, :])
        axc = None
    else:
        gs = fig.add_gridspec(2, 4, left=0.045, right=0.995, bottom=0.14, top=0.93, wspace=0.28, hspace=0.55)
        axa = fig.add_subplot(gs[0,1])
        axb = fig.add_subplot(gs[0,0])
        axc = fig.add_subplot(gs[0,2])
        axd = fig.add_subplot(gs[0,3])
        axe = fig.add_subplot(gs[1, :])

    # ---------- (a) AE/PEM 堆积 + LCOH 点 ----------
    GA = [p for p in GROUP_A if p in agg]
    if GA:
        centers_flat, centers_by_cl, wide_w, narrow_w, cluster_centers = grouped_positions(clusters, GA)
        ae_vals, pem_vals, lcoh = [], [], []
        for c in clusters:
            for prof in GA:
                row = agg.get(prof, {}).get(c, {})
                ae_vals.append(row.get("Cap_AE_kW", np.nan))
                pem_vals.append(row.get("Cap_PEM_kW", np.nan))
                lcoh.append(row.get("LCOH_$/kg", np.nan))
        b1 = axa.bar(centers_flat, np.nan_to_num(ae_vals, nan=0.0),  width=wide_w, color=COL_AE,  **EDGE_KWARGS, label="AE")
        b2 = axa.bar(centers_flat, np.nan_to_num(pem_vals, nan=0.0), width=wide_w, bottom=np.nan_to_num(ae_vals, nan=0.0), color=COL_PEM, **EDGE_KWARGS, label="PEM")
        # 仅标“总值”
        tot_a = (np.nan_to_num(ae_vals, nan=0.0) + np.nan_to_num(pem_vals, nan=0.0))
        _annotate_totals(axa, centers_flat, tot_a, width=wide_w, fmt="{:.0f}", inside=False)

        # LCOH 散点 + 白底半透明标签
        axa2 = axa.twinx()
        axa2.scatter(centers_flat, lcoh, s=28, color=COL_DOT, zorder=5)
        _annotate_scatter(axa2, centers_flat, lcoh, fmt="{:.2f}", fontsize=8)
        axa2.set_ylabel("LCOH ($/kg)")

        axa.set_ylabel("电解槽装机（kW）"); axa.set_title("(b) 电解槽配比 & LCOH")
        apply_two_tier_xticks(axa, clusters, GA, centers_flat, cluster_centers, pad=60)

        # 获取原 legend 项
        handles_a, labels_a = axa.get_legend_handles_labels()

        # 在右边追加散点代理
        scatter_proxy = Line2D([0],[0], marker='o', linestyle='None', color=COL_DOT, markersize=6, label="LCOH（散点）")

        # 合并时：控制顺序，散点在最后即“右边”
        handles_all = handles_a + [scatter_proxy]

        # 调整 ncol 确保在同一行显示
        axa.legend(handles=handles_all, loc="upper left", ncol=len(handles_all))


        _apply_axis_limits(axa, 'a', 'left')
        _apply_axis_limits(axa2, 'a', 'right')


    # ---------- (b) 风光宽堆积 + 电池窄堆积 + LCOE ----------
    GB = [p for p in GROUP_B if p in agg]
    if GB:
        centers_flat, centers_by_cl, wide_w, narrow_w, cluster_centers = grouped_positions(clusters, GB)
        wind_vals, pv_vals, be_vals, bp_vals, lcoe = [], [], [], [], []
        for c in clusters:
            for prof in GB:
                row = agg.get(prof, {}).get(c, {})
                wind_vals.append(row.get("Cap_wind_kW", np.nan))
                pv_vals.append(row.get("Cap_pv_kW",   np.nan))
                be_vals.append(row.get("Cap_batt_E_kWh", np.nan))
                bp_vals.append(row.get("Cap_batt_P_kW",  np.nan))
                lcoe.append(row.get("LCOE_$/MWh", np.nan))
        bw1 = axb.bar(centers_flat, np.nan_to_num(wind_vals, nan=0.0), width=wide_w, color=COL_WIND, **EDGE_KWARGS, label="风电")
        bw2 = axb.bar(centers_flat, np.nan_to_num(pv_vals,   nan=0.0), width=wide_w, bottom=np.nan_to_num(wind_vals, nan=0.0), color=COL_PV, **EDGE_KWARGS, label="光伏")
        bb1 = axb.bar(centers_flat, np.nan_to_num(be_vals, nan=0.0), width=narrow_w, color=COL_BATT_E, **EDGE_KWARGS, label="电池E")
        bb2 = axb.bar(centers_flat, np.nan_to_num(bp_vals, nan=0.0), width=narrow_w, bottom=np.nan_to_num(be_vals, nan=0.0), color=COL_BATT_P, **EDGE_KWARGS, label="电池P")

        # 仅标堆叠“总值”
        tot_wpv = (np.nan_to_num(wind_vals, nan=0.0) + np.nan_to_num(pv_vals, nan=0.0))
        _annotate_totals(axb, centers_flat, tot_wpv, width=wide_w, fmt="{:.0f}", inside=False)

        tot_batt = (np.nan_to_num(be_vals, nan=0.0) + np.nan_to_num(bp_vals, nan=0.0))
        _annotate_totals(axb, centers_flat, tot_batt, width=narrow_w, fmt="{:.0f}", inside=False)

        # LCOE 散点 + 白底半透明标签
        axb2 = axb.twinx()
        axb2.scatter(centers_flat, lcoe, s=26, color=COL_DOT, zorder=6)
        _annotate_scatter(axb2, centers_flat, lcoe, fmt="{:.1f}", fontsize=8)
        axb2.set_ylabel("LCOE ($/MWh)")

        axb.set_ylabel("装机（kW / kWh）"); axb.set_title("(a) 风/光与电池+ LCOE")
        apply_two_tier_xticks(axb, clusters, GB, centers_flat, cluster_centers, pad=60)

        # === 在原有 legend 基础上追加“LCOE（散点）” ===
        handles_b, labels_b = axb.get_legend_handles_labels()
        handles_b.append(Line2D([0],[0], marker='o', linestyle='None', color=COL_DOT, markersize=6, label="LCOE（散点）"))
        axb.legend(handles=handles_b, loc="upper left", ncol=3)  # 版面窄可改回 ncol=2

        _apply_axis_limits(axb, 'b', 'left')
        _apply_axis_limits(axb2, 'b', 'right')

    # ---------- (c) 仅 methanol/ftl：DAC（t/h） + CF + LCOC ----------
    # ammonia 剔除 HighDAC/LowDAC；其他产品不剔除
    if axc is not None:
        if product == "ammonia":
            GC = [p for p in GROUP_C if (p in agg) and (p not in EXCLUDE_IN_COST_AMMONIA)]
        else:
            GC = [p for p in GROUP_C if p in agg]
        if GC:
            centers_flat, centers_by_cl, wide_w, narrow_w, cluster_centers = grouped_positions(clusters, GC)
            kgph_vals, cf_labels, lcoc = [], [], []
            for c in clusters:
                for prof in GC:
                    row = agg.get(prof, {}).get(c, {})
                    kgph, cf = _dac_tph_and_cf(row, product)
                    kgph_vals.append(kgph)
                    cf_labels.append(f"{cf*100:.0f}%" if np.isfinite(cf) else "NA")
                    lcoc.append(row.get("LCOC_$/tCO2", np.nan))
            bc = axc.bar(centers_flat, np.nan_to_num(kgph_vals, nan=0.0), width=wide_w, color=COL_DAC, **EDGE_KWARGS, label="DAC (kg/h)")
            # 不给 DAC 数值标签；仅保留 CF 顶部标注
            _annotate_top(axc, bc, cf_labels)

            # LCOC 散点 + 白底半透明标签
            axc2 = axc.twinx()
            axc2.scatter(centers_flat, lcoc, s=26, color=COL_DOT, zorder=6)
            _annotate_scatter(axc2, centers_flat, lcoc, fmt="{:.0f}")
            axc2.set_ylabel("LCOC ($/tCO2)")

            axc.set_ylabel("DAC 装机（t/h）"); axc.set_title("(c) DAC 装机 & 年容量因子 + LCOC")
            apply_two_tier_xticks(axc, clusters, GC, centers_flat, cluster_centers, pad=60)

            # === 在原有 legend 基础上追加“LCOC（散点）” ===
            handles_c, labels_c = axc.get_legend_handles_labels()
            handles_c.append(Line2D([0],[0], marker='o', linestyle='None', color=COL_DOT, markersize=6, label="LCOC（散点）"))
            axc.legend(handles=handles_c, loc="upper left", ncol=2)

            _apply_axis_limits(axc,  'c', 'left')
            _apply_axis_limits(axc2, 'c', 'right')


    # ---------- (d) 风光（底层宽柱） + 合成装置（上层窄柱） ----------
    GD = [p for p in GROUP_D if _key_ci(agg, p) is not None]
    if GD:
        # ===== 基础布局 =====
        centers_flat, centers_by_cl, wide_w, narrow_w, cluster_centers = grouped_positions(clusters, GD)
        bar_windpv, bar_syn = wide_w, narrow_w  # 风光宽，合成窄

        # ====== 风光（底层） ======
        wind_vals, pv_vals = [], []
        for c in clusters:
            for prof in GD:
                row = _safe_row(agg, prof, c)
                wv = row.get("Cap_wind_kW", np.nan)
                pv = row.get("Cap_pv_kW",   np.nan)
                wind_vals.append(wv if np.isfinite(wv) else 0.0)
                pv_vals.append(  pv if np.isfinite(pv) else 0.0)

        # 先画风/光（底层）
        bw = axd.bar(
            centers_flat, wind_vals, width=bar_windpv,
            color=COL_WIND_BASE, **EDGE_KWARGS, label="风电"
        )
        bp = axd.bar(
            centers_flat, pv_vals, width=bar_windpv, bottom=wind_vals,
            color=COL_PV_BASE, **EDGE_KWARGS, label="光伏"
        )

        # 风光总值标注（横向标签）
        tot_wpv_d = (np.array(wind_vals) + np.array(pv_vals)).tolist()
        _annotate_totals(axd, centers_flat, tot_wpv_d, width=bar_windpv, fmt="{:.0f}", inside=False)

        axd.set_ylabel("风+光装机（kW）")
        axd.set_title("(d) 风/光与合成装置")
        apply_two_tier_xticks(axd, clusters, GD, centers_flat, cluster_centers, pad=60)
        _apply_axis_limits(axd, 'd', 'left')

        # ====== 合成装置（顶层） ======
        axd2 = axd.twinx()
        axd2.set_zorder(axd.get_zorder() + 2)
        axd2.patch.set_alpha(0.0)

        syn_vals, syn_cf_labels = [], []
        for c in clusters:
            for prof in GD:
                row = _safe_row(agg, prof, c)
                val = row.get("Cap_syn", np.nan)
                syn_vals.append(val if np.isfinite(val) else np.nan)
                syn_cf = _syn_cf(row, product)
                syn_cf_labels.append(f"{syn_cf*100:.0f}%" if np.isfinite(syn_cf) else "NA")

        bsyn = axd2.bar(
            centers_flat,
            np.array([v if np.isfinite(v) else 0.0 for v in syn_vals]),
            width=bar_syn,
            color=PANEL_COLORS['电解槽/合成'][1],
            **EDGE_KWARGS,
            label="合成装置"
        )

        # 不给数值，只给 CF 标签（跨轴绘制，避免被遮）
        _annotate_top_cross(axd2, axd2, bsyn, syn_cf_labels, fs=9, dy_frac=0.01)

        axd2.set_ylabel("合成装置（名义产能/小时）")
        _apply_axis_limits(axd2, 'd', 'right')

        # 让合成柱永远在风光之上
        for b in list(bsyn):
            b.set_zorder(axd.get_zorder() + 3)


    # ---------- (e) 成本分解（ammonia 剔除 HighDAC/LowDAC；其他产品不剔除） ----------
    def pick_breakdown(row: dict):
        gen = (row.get("Wind_annual_cost_$py",0) + row.get("PV_annual_cost_$py",0))
        ely = (row.get("AE_annual_cost_$py",0)  + row.get("PEM_annual_cost_$py",0))
        stor= (row.get("BattE_annual_cost_$py",0)+row.get("BattS_annual_cost_$py",0)+row.get("H2S_annual_cost_$py",0))
        syn = (row.get("Synth_annual_cost_$py",0)+row.get("Synth_store_annual_cost_$py",0))
        dac = (row.get("DAC_annual_cost_$py",0) + row.get("CO2_comp_annual_cost_$py",0) + row.get("CO2_store_annual_cost_$py",0))
        heat= (row.get("HP_annual_cost_$py",0)  + row.get("TS_annual_cost_$py",0))
        return [gen,ely,stor,syn,dac,heat]

    if product == "ammonia":
        scen_order_e = [p for p in ALL_PROFILES if (p in agg) and (p not in EXCLUDE_IN_COST_AMMONIA)]
    else:
        scen_order_e = [p for p in ALL_PROFILES if p in agg]
    centers_flat_e, centers_by_cl_e, width_w_e, _, cluster_centers_e = grouped_positions(clusters, scen_order_e)

    stacks = [[] for _ in range(6)]
    for j, c in enumerate(clusters):
        for p in scen_order_e:
            row = agg.get(p, {}).get(c, {})
            vals = pick_breakdown(row)
            for i in range(6): stacks[i].append(vals[i])

    bot = np.zeros(len(centers_flat_e))
    for name, color, colvals in zip(["发电","电解槽","储能","合成","DAC","供热"], COLORS_BREAK, stacks):
        axe.bar(centers_flat_e, colvals, bottom=bot, color=color, **EDGE_KWARGS, label=name)
        bot += np.array(colvals)

    # 仅贴堆叠总值
    totals_e = bot
    _annotate_totals(axe, centers_flat_e, totals_e, width=width_w_e, fmt="{:.0f}", inside=False)

    if product == "ammonia":
        axe.set_ylabel("年度成本（$/年，均值）"); axe.set_title("(e) 成本分解")
    else:
        axe.set_ylabel("年度成本（$/年，均值）"); axe.set_title("(e) 成本分解")
    apply_two_tier_xticks(axe, clusters, scen_order_e, centers_flat_e, cluster_centers_e, pad=60)
    axe.legend(ncol=6, loc="upper center", bbox_to_anchor=(0.5, 1.18))
    _apply_axis_limits(axe, 'e', 'left')

    # ========== 整体 ==========
    os.makedirs(PARAM_DIR, exist_ok=True)
    out_png = os.path.join(PARAM_DIR, f"{product}_scenario_panels.png")
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"✔ 保存：{out_png}")

def main():
    if not os.path.isdir(PARAM_DIR):
        raise FileNotFoundError(f"目录不存在：{PARAM_DIR}")
    for prod in PRODUCTS:
        plot_panels_for_product(prod)

if __name__ == "__main__":
    main()


In [ ]:
# 运行主程序生成优化图例位置的图表
main()

# 技术经济参数不确定性对绿色化学品生产系统的影响分析

## 分产品详细敏感性分析报告

基于上述生成的参数敏感性对比图表，现对氨、甲醇和SAF三种绿色化学品在关键技术经济参数波动下的系统表现分别进行深入分析。本分析涵盖电解槽技术选择、储能配置策略、DAC系统优化、合成工艺改进等核心技术路径的经济性影响。

### 1. 绿氨生产系统的技术经济参数敏感性分析

图~\ref{fig:green-ammonia-param-sensitivity} 展示了绿氨生产系统在关键技术经济参数波动下的系统表现差异。从电解槽技术路径来看，绿氨生产系统对电解槽成本与效率参数表现出高度敏感性，技术选择直接影响系统的经济竞争力。

**电解槽技术路径优化**：在高AE成本情景下（350 $/kW），类型1区域的电解槽配置呈现明显的技术替代效应。AE装机占比从基准情景的75%下降至45%，PEM装机相应提升，总体电解槽装机容量增加12-15%以补偿效率差异。LCOH从基准的2.8 $/kg上升至3.2 $/kg，增幅约14%。相比之下，高PEM成本情景（650 $/kW）的影响相对温和，LCOH仅上升8%，这主要因为绿氨系统中AE技术的主导地位以及其在大规模应用中的成本优势。类型2区域由于资源品质相对较低，对电解槽效率的敏感性更强，高效PEM技术在该区域显示出更大的经济价值。

**储能系统配置策略**：储能成本波动对绿氨系统的影响呈现分层特征。高电池成本情景下（200 $/kWh），系统倾向于增加氢储容量作为替代，氢储装机增加25-30%，电池储能容量下降20%。这种"储能技术替代"效应使得总储能成本仅增加10-12%，LCOE上升幅度控制在5%以内。低电池成本情景（50 $/kWh）则促进了更为灵活的"电-氢"耦合储能策略，短周期电池储能与长周期氢储的协同优化使系统整体效率提升，LCOE可下降8-10%。在不同资源类型区域，储能配置策略表现出明显差异：类型1区域更依赖氢储的长周期调节，类型2区域则需要更多电池储能来平滑短期波动。

**合成工艺技术进步**：氨合成装置的技术经济参数变化对系统整体表现产生重要影响。高合成成本情景下（9000 $/单位产能），通过优化合成反应器设计与催化剂性能，可实现氨合成效率提升15-20%，使得单位产品的氢耗降低，部分抵消设备投资增加的影响。低合成成本情景（3500 $/单位产能）结合技术改进，使绿氨的平准化成本在类型1区域可降至520-550 $/t，接近与化石氨的成本平价水平。合成装置的年容量因子在不同情景下保持相对稳定（85-90%），表明氨合成工艺的负荷跟踪能力较强。

**系统集成优化效应**：多技术路径的协同优化效应在绿氨系统中表现突出。在低电解槽成本与低储能成本的组合情景下，系统可实现"高效电解+灵活储能"的理想配置，使得年化投资成本下降20-25%，LCOH降至2.2-2.4 $/kg的水平。相反，高成本组合情景下，系统需要通过增加装机冗余和多技术路径并行来维持生产目标，总投资成本可增加30-35%。这种技术经济参数的交互效应表明，绿氨项目的技术路径选择需要基于系统性优化而非单一技术的成本最小化。

**经济竞争力与风险评估**：技术经济参数波动对绿氨市场竞争力的影响呈现显著的区域分化特征。在类型1区域，技术进步情景下绿氨成本可降至450-500 $/t，具备与化石氨竞争的潜力；在类型2区域，即使在最优技术组合下，绿氨成本仍需依赖碳价支撑（50-80 $/tCO2）才能实现经济性。参数敏感性分析表明，电解槽成本每下降10%，绿氨成本可下降5-7%；储能成本每下降10%，LCOE下降3-5%，进而传导至绿氨成本下降2-3%。这为技术研发投资方向和政策支持重点提供了定量化的决策依据。

### 2. 绿色甲醇生产系统的技术经济参数敏感性分析

图~\ref{fig:green-methanol-param-sensitivity} 揭示了绿色甲醇生产系统在技术经济参数波动下表现出独特的响应特征。甲醇合成路径涉及CO₂直接空气捕集（DAC）与甲醇合成工艺的耦合，其对不同技术参数的敏感性呈现出鲜明的技术经济特征。

**DAC技术路径优化效应**：DAC系统的技术经济参数对绿色甲醇成本结构产生决定性影响。高DAC成本情景下（7800 $/单位捕集能力），系统LCOC从基准的120 $/tCO2上升至180 $/tCO2，增幅达50%，直接传导至甲醇成本上升15-20%。DAC单位电耗参数的敏感性更为显著：从低电耗情景（0.22 MWh/tCO2）到高电耗情景（0.31 MWh/tCO2），甲醇生产的总电耗增加8-12%，在类型2区域（相对低资源品质）的成本影响可达12-15%。低DAC成本情景（3000 $/单位捕集能力）结合技术优化，使得LCOC降至80-90 $/tCO2，为绿色甲醇接近化石甲醇成本平价提供了技术路径。DAC年容量因子在不同情景下维持在75-85%的水平，表明其作为基荷设备的运行稳定性。

**电解槽与储能系统协同优化**：绿色甲醇系统中电解槽技术选择与储能配置表现出更为均衡的特征。高AE成本情景下，AE装机占比从基准的65%调整至55%，PEM占比相应提升，但总体装机容量变化较小（±8%），这主要归因于甲醇合成对氢气供应稳定性的要求相对宽松。储能配置方面，高电池成本情景下系统更倾向于利用DAC的负荷调节能力来平衡电力供需，电池储能容量下降15-20%，而通过优化DAC运行策略实现系统柔性。低储能成本情景则促进了"DAC-储能"协同调度策略的发展，使得系统可在可再生能源丰富时段大量捕集CO₂并储存，在电力紧缺时段优先保障甲醇合成，整体效率提升10-15%。

**合成工艺集成技术进步**：甲醇合成装置的技术经济改进对系统整体性能具有显著影响。高合成成本情景下（8000 $/单位产能），通过采用新型催化剂和反应器优化技术，可实现甲醇合成效率提升20-25%，单位产品的氢耗和CO₂耗量均有所下降。低合成成本情景（2600 $/单位产能）结合工艺集成优化，使得绿色甲醇在类型1区域的成本可降至600-650 $/t，显著缩小与化石甲醇的成本差距。合成装置年容量因子保持在80-85%的高水平，反映出甲醇合成工艺良好的负荷跟踪性能和工艺稳定性。

**系统集成与工艺柔性优势**：绿色甲醇系统的独特优势在于DAC-甲醇合成的工艺耦合带来的系统柔性。在技术经济最优组合情景下（低DAC成本+低电解槽成本+低合成成本），系统可实现"柔性捕集+高效转化"的理想运行模式，总投资成本下降25-30%，甲醇成本降至500-550 $/t的水平。CO₂储存缓冲系统使得甲醇生产可以在一定程度上去耦合于实时电力供应，这种"时移"特性使得系统对可再生能源波动的适应性显著优于氨系统。在高成本组合情景下，系统仍可通过工艺参数优化和运行策略调整将成本增幅控制在20%以内。

**技术路径经济性与市场竞争力**：技术经济参数敏感性分析揭示了绿色甲醇实现商业化的关键技术节点。DAC成本每下降10%，甲醇成本可下降3-5%；电解槽成本下降10%对应甲醇成本下降2-4%；合成工艺成本下降10%可带来甲醇成本下降1-3%的改善。综合技术进步情景下，绿色甲醇在类型1区域已具备在中等碳价水平（40-60 $/tCO2）下与化石甲醇竞争的能力。在类型2区域，技术经济参数优化可将甲醇成本从基准的800-850 $/t降至650-700 $/t，为后续商业化推广奠定了基础。技术参数波动对甲醇项目IRR的影响约为±2.0-2.5个百分点，相比SAF项目的高敏感性表现出更强的投资稳定性。

### 3. 绿色SAF（费托燃料）生产系统的技术经济参数敏感性分析

图~\ref{fig:green-saf-param-sensitivity} 展示了费托合成路径制备可持续航空燃料（SAF）在技术经济参数波动下的系统表现。SAF生产系统由于其工艺复杂性与多环节耦合特征，对技术经济参数变化表现出最为显著的敏感性响应。

**DAC系统技术经济优化**：DAC技术参数对SAF系统的影响更为突出，这主要源于SAF生产的高CO₂需求量（约3.3-3.9 tCO₂/tSAF）。高DAC成本情景下（7800 $/单位捕集能力），LCOC上升至200-220 $/tCO2，直接推动SAF成本增加25-30%，在类型2区域可使SAF成本超过2200 $/t的水平。DAC电耗参数的影响同样显著：高电耗情景（0.31 MWh/tCO2）使得SAF生产总电耗增加15-20%，对应成本增幅达18-22%。低DAC成本情景（3000 $/单位捕集能力）结合低电耗技术（0.22 MWh/tCO2），可使LCOC降至70-80 $/tCO2，为SAF成本下降20-25%创造条件。DAC年容量因子在SAF系统中通常维持在80-90%的高水平，反映其作为关键基荷设备的重要地位。

**费托合成工艺技术进步效应**：费托合成装置的技术经济参数对SAF系统整体性能具有决定性影响。高合成成本情景下（22000 $/单位产能），通过采用微通道反应器和先进催化剂技术，可实现反应温度降低40-50°C，热需求下降20-25%，部分抵消投资成本增加的影响。低合成成本情景（9400 $/单位产能）结合工艺优化，使得SAF在类型1区域的成本可降至1400-1500 $/t，显著提升与化石航煤的竞争潜力。费托合成的氢耗比和CO₂耗比在不同技术情景下存在10-15%的变化空间，直接影响上游电解槽和DAC的配置需求。合成装置年容量因子保持在75-85%，略低于氨醇系统，这主要受费托工艺复杂性和维护需求影响。

**电解槽技术与热管理系统集成**：SAF系统中电解槽技术选择与热管理系统的集成优化效应尤为突出。高PEM成本情景下（650 $/kW），由于费托反应对氢气纯度的严格要求，PEM装机占比仍需维持在40-45%以上，使得总体电解槽成本显著上升。热泵系统的技术经济参数对SAF生产具有重要影响：高热泵成本情景下（500 $/kW热功率），系统倾向于采用电加热等替代方案，整体能效下降8-12%；低成本高效热泵（COP>4.0）的应用可使SAF系统的综合能效提升15-20%。储热系统的配置在SAF生产中扮演关键角色，热储容量通常需达到日均热负荷的8-12小时以应对费托反应的连续高温需求。

**系统复杂性与技术路径优化**：SAF系统的高复杂性使得技术经济参数的交互效应更为显著。在技术进步组合情景下（低DAC成本+低费托成本+高效热管理），系统可实现"高效捕集+优化转化+智能热管理"的集成运行模式，总投资成本下降30-35%，SAF成本降至1200-1300 $/t的水平。相反，高成本组合情景下，系统需要通过增加设备冗余和多路径备份来确保生产稳定性，总投资成本可增加40-50%。模块化工艺设计在SAF系统中显示出重要价值：通过灵活调节各模块负荷，系统可在电力充足时满负荷运行，在电力紧缺时降至60-70%负荷，这种弹性运行能力有助于降低技术经济参数波动的不利影响。

**技术路径经济性与产业化前景**：技术经济参数敏感性分析揭示了SAF实现商业化的技术难点与机遇。DAC成本每下降10%，SAF成本可下降4-6%；费托合成成本下降10%对应SAF成本下降6-8%；电解槽和热管理系统成本优化可分别带来2-4%的成本改善。综合技术进步情景下，SAF在类型1区域的成本有望降至1200-1400 $/t，虽然仍高于化石航煤，但在考虑碳减排价值（500-800 $/tCO2eq）后具备一定竞争力。技术参数波动对SAF项目IRR的影响达±3.5-4.5个百分点，是三种产品中最为敏感的，这要求SAF项目在技术路径选择和风险管理方面更加审慎。然而，SAF的高技术壁垒和政策支持力度也为技术领先企业提供了显著的市场机遇，特别是在航空业脱碳压力不断增大的背景下。

### 4. 三种产品的技术经济参数敏感性对比与技术发展建议

通过对绿氨、绿色甲醇和绿色SAF三种产品的技术经济参数敏感性分析，可以得出以下关键结论与技术发展建议：

#### 4.1 技术经济敏感性排序与关键参数识别

**敏感性递增序列**：绿色甲醇 < 绿氨 < 绿色SAF。具体表现为：
- **关键参数成本影响**：甲醇±12-18%，氨±15-25%，SAF±20-35%
- **技术路径依赖性**：甲醇（DAC主导），氨（电解槽主导），SAF（多技术耦合）
- **投资风险敏感性**：甲醇±2.0-2.5%，氨±2.5-3.5%，SAF±3.5-4.5%

**关键技术参数识别**：
- **电解槽技术**：AE成本对氨系统影响最大（成本弹性0.5-0.7），PEM技术对SAF纯度要求关键
- **DAC技术**：电耗参数对甲醇和SAF系统影响显著，单位电耗每下降10%可降低产品成本3-5%
- **储能技术**：电池成本对氨系统影响较大，甲醇系统可通过工艺柔性部分替代储能需求
- **合成工艺**：SAF费托合成成本敏感性最高，技术进步潜力最大

#### 4.2 技术发展路径与优先级评估

**近期技术突破重点（2025-2030）**：
- **电解槽技术**：AE规模化制造成本控制在300-400 $/kW，PEM技术成本降至500-600 $/kW
- **储能技术**：电池储能成本目标100 $/kWh以下，氢储技术重点突破长周期储能效率
- **DAC技术**：单位电耗降至0.2 MWh/tCO2以下，设备成本控制在4000 $/单位捕集能力
- **合成工艺**：氨合成装置成本降至5000 $/单位产能，甲醇合成效率提升20%以上

**中期技术集成目标（2030-2035）**：
- **系统集成优化**：多能耦合系统效率提升至80%以上，工艺柔性调节范围达到40-100%负荷
- **智能控制技术**：基于AI的负荷预测与优化调度，系统响应时间缩短至分钟级
- **模块化技术**：标准化模块设计实现快速部署，建设周期缩短30-50%
- **新材料应用**：催化剂寿命延长至5年以上，膜分离技术在气体纯化中的应用

#### 4.3 区域技术发展适应性策略

**类型1区域（高风-高光）**：
- **技术配置优化**：优先发展大型集成化系统，单体规模100MW以上
- **技术路径选择**：SAF项目可采用最先进技术路径，承担技术验证功能
- **创新平台建设**：建立绿色化学品技术创新示范基地，汇聚产学研资源

**类型2区域（中等资源）**：
- **技术适应性改进**：重点发展适应资源波动的柔性技术，如可调负荷电解槽
- **成本控制导向**：优先选择技术成熟度高、成本可控的技术路径
- **分布式技术应用**：发展小型化、模块化技术解决方案

**类型3区域（低风-高光）**：
- **光伏耦合技术**：开发适应光伏出力特性的专用设备和工艺
- **储能技术优化**：重点发展低成本、长寿命的储能技术方案
- **工艺简化技术**：通过工艺优化减少对储能和调节设备的依赖

#### 4.4 技术创新政策建议

**研发支持政策**：
- **差异化支持强度**：SAF技术享受最高研发支持（单个项目支持额度1-2亿元），甲醇技术注重产业化支持，氨技术重点支持规模化制造
- **关键技术攻关**：设立"绿色化学品关键技术专项"，重点支持DAC、高效电解槽、费托合成等核心技术
- **产学研协同机制**：建立"技术经济参数数据库"，实现技术进步的实时跟踪与评估

**标准化与认证体系**：
- **技术标准制定**：建立绿色化学品技术经济评价标准，统一技术参数定义与测试方法
- **设备认证体系**：推行关键设备技术经济参数认证，建立设备性能公开数据库
- **系统集成标准**：制定多技术耦合系统的设计、建设和运营标准

**市场机制创新**：
- **技术进步激励**：建立基于技术经济参数改进的奖励机制，鼓励持续技术创新
- **风险分担机制**：设立"绿色化学品技术风险基金"，为技术路径选择提供保险支撑
- **技术交易平台**：建立绿色化学品技术许可与转让平台，促进技术扩散

#### 4.5 技术发展时间表与里程碑

**2025年目标**：
- 电解槽技术：AE成本400 $/kW，PEM成本600 $/kW
- 储能技术：电池储能成本120 $/kWh
- DAC技术：单位电耗0.25 MWh/tCO2
- 系统集成：绿氨、甲醇示范项目实现稳定运行

**2030年目标**：
- 电解槽技术：AE成本300 $/kW，PEM成本500 $/kW
- 储能技术：电池储能成本80 $/kWh，氢储成本600 $/kW
- DAC技术：单位电耗0.2 MWh/tCO2，设备成本3000 $/单位捕集能力
- 系统集成：SAF示范项目实现技术验证，绿氨、甲醇进入商业化阶段

**2035年目标**：
- 全技术路径成本竞争力：绿氨500 $/t，甲醇600 $/t，SAF 1200 $/t
- 系统效率目标：综合能效达到75%以上
- 产业规模目标：三种产品总产能达到1000万吨/年

通过系统性的技术经济参数敏感性分析，为绿色化学品产业的技术发展路径选择提供了科学决策支撑，有助于在技术进步与成本控制之间实现最优平衡，推动产业健康可持续发展。